# Notebook 02 — Source Field Quality Profile

## Analytical question

How is each field in the source `raceform.db` table actually stored, and what observable quality issues must be understood before cleaning, parsing, filtering, or target-schema design begins?

## Purpose

This notebook profiles the 37 source fields as stored.

It will produce:

- a source data dictionary;
- a field-level quality register;
- evidence on SQLite storage classes;
- counts of blanks, nulls, placeholders and special codes;
- observed numeric and date-like ranges where appropriate;
- representative values and frequency distributions;
- identification of fields requiring separate racing-domain investigation.

## Source

Database: `data/raw/form_2015-present/form_2015-present/raceform.db`

Source table: `data`

The first physical row is an imported header row and must be excluded from field-quality calculations without modifying the raw database.

## Methodological boundaries

This notebook will not:

- modify the raw database;
- replace or impute missing values;
- standardise text;
- parse composite fields into new fields;
- remove records;
- assign permanent identifiers;
- design the analytical target schema;
- assume undocumented racing-domain meanings;
- treat unusual values as errors without supporting evidence.

Observed evidence and later transformation decisions will remain separate.

## Profiling sequence

Fields will be examined in coherent groups:

1. identifier and race-description fields;
2. runner identity and demographics;
3. result and finishing fields;
4. odds, ratings, weight and time fields;
5. pedigree and ownership fields;
6. free-text comments.

The first group is:

- `date`
- `course`
- `race_id`
- `off`
- `race_name`
- `type`
- `class`
- `pattern`
- `rating_band`
- `age_band`
- `sex_rest`
- `dist`
- `going`
- `ran`

In [1]:
from pathlib import Path
import sqlite3

import pandas as pd

# Resolve the repository root whether Jupyter was started from the project root
# or from inside the notebooks directory.
current_directory = Path.cwd().resolve()

if (current_directory / "data").exists():
    project_root = current_directory
elif (current_directory.parent / "data").exists():
    project_root = current_directory.parent
else:
    raise FileNotFoundError(
        "Could not locate the project root containing the data directory."
    )

source_database = (
    project_root
    / "data"
    / "raw"
    / "form_2015-present"
    / "form_2015-present"
    / "raceform.db"
)

if not source_database.is_file():
    raise FileNotFoundError(f"Source database not found: {source_database}")

# Open the immutable source database in read-only mode.
connection = sqlite3.connect(f"file:{source_database}?mode=ro", uri=True)

print(f"Project root:    {project_root}")
print(f"Source database: {source_database}")
print("Connection mode: read-only")

Project root:    /home/rob/Documents/inside-rails-horse-racing
Source database: /home/rob/Documents/inside-rails-horse-racing/data/raw/form_2015-present/form_2015-present/raceform.db
Connection mode: read-only


In [2]:
# Read the declared SQLite column definitions without loading the full table.
schema = pd.read_sql_query(
    'PRAGMA table_info("data")',
    connection,
)

# Keep the most useful schema evidence for this notebook.
source_data_dictionary = schema[
    ["cid", "name", "type", "notnull", "dflt_value", "pk"]
].copy()

source_data_dictionary = source_data_dictionary.rename(
    columns={
        "cid": "column_position",
        "name": "field_name",
        "type": "declared_sqlite_type",
        "notnull": "declared_not_null",
        "dflt_value": "declared_default",
        "pk": "declared_primary_key",
    }
)

# SQLite uses zero-based internal column positions.
source_data_dictionary["column_position"] += 1

print(f"Declared field count: {len(source_data_dictionary):,}")

display(source_data_dictionary)

Declared field count: 37


,column_position,field_name,declared_sqlite_type,declared_not_null,declared_default,declared_primary_key
0,1,date,NUMERIC,0,None,0
1,2,course,TEXT,0,None,0
2,3,race_id,INTEGER,0,None,0
3,4,off,TEXT,0,None,0
4,5,race_name,TEXT,0,None,0
5,6,type,TEXT,0,None,0
6,7,class,TEXT,0,None,0
7,8,pattern,TEXT,0,None,0
8,9,rating_band,TEXT,0,None,0
9,10,age_band,TEXT,0,None,0


In [3]:
# Inspect the first few physical rows, including SQLite's internal rowid.
identifier_race_fields = [
    "date",
    "course",
    "race_id",
    "off",
    "race_name",
    "type",
    "class",
    "pattern",
    "rating_band",
    "age_band",
    "sex_rest",
    "dist",
    "going",
    "ran",
]

quoted_fields = ", ".join(f'"{field}"' for field in identifier_race_fields)

first_rows = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        {quoted_fields}
    FROM data
    ORDER BY rowid
    LIMIT 5
    """,
    connection,
)

display(first_rows)

,source_rowid,date,course,race_id,off,race_name,type,class,pattern,rating_band,age_band,sex_rest,dist,going,ran
0,1,date,course,race_id,off,race_name,type,class,pattern,rating_band,age_band,sex_rest,dist,going,ran
1,2,2015-01-01,Catterick,615704,12:30,Happy New Year Novices Hurdle,Hurdle,Class 4,,,4yo+,,2m3½f,Good To Soft,10
2,3,2015-01-01,Tramore (IRE),616859,12:35,2015 Waterford & Tramore Racecourse Supporters...,Hurdle,,,80-109,4yo+,,2m,Heavy,8
3,4,2015-01-01,Tramore (IRE),616860,1:05,Padraig Curran - South East Cleaners Maiden Hu...,Hurdle,,,,5yo+,,2m5f,Heavy,8
4,5,2015-01-01,Tramore (IRE),616860,1:05,Padraig Curran - South East Cleaners Maiden Hu...,Hurdle,,,,5yo+,,2m5f,Heavy,8


In [4]:
# Verify that the first physical row reproduces every declared field name.
all_source_fields = source_data_dictionary["field_name"].tolist()

header_row = pd.read_sql_query(
    """
    SELECT *
    FROM data
    WHERE rowid = 1
    """,
    connection,
).iloc[0]

header_check = pd.DataFrame(
    {
        "field_name": all_source_fields,
        "stored_header_value": [
            str(header_row[field]) for field in all_source_fields
        ],
    }
)

header_check["matches_field_name"] = (
    header_check["stored_header_value"]
    == header_check["field_name"]
)

matching_fields = int(header_check["matches_field_name"].sum())

print(
    f"Header-name matches: {matching_fields:,} "
    f"of {len(all_source_fields):,}"
)
print(f"All 37 fields match: {header_check['matches_field_name'].all()}")

# This predicate will be included explicitly in profiling queries.
# It excludes the imported header without altering the raw source.
DATA_ROW_PREDICATE = "rowid <> 1"

display(header_check)

Header-name matches: 37 of 37
All 37 fields match: True


,field_name,stored_header_value,matches_field_name
0,date,date,True
1,course,course,True
2,race_id,race_id,True
3,off,off,True
4,race_name,race_name,True
5,type,type,True
6,class,class,True
7,pattern,pattern,True
8,rating_band,rating_band,True
9,age_band,age_band,True


In [5]:
# Build a compact field-level quality summary for the first field group.
#
# No values are cleaned or converted here. All counts describe the source
# exactly as SQLite stores it, excluding only the confirmed header row.

profile_rows = []

for field in identifier_race_fields:
    quoted_field = f'"{field}"'

    query = f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN {quoted_field} IS NULL THEN 1 ELSE 0 END) AS null_count,
        SUM(
            CASE
                WHEN typeof({quoted_field}) = 'text'
                 AND length(trim({quoted_field})) = 0
                THEN 1 ELSE 0
            END
        ) AS blank_text_count,
        COUNT(DISTINCT {quoted_field}) AS distinct_non_null_values,
        SUM(CASE WHEN typeof({quoted_field}) = 'null' THEN 1 ELSE 0 END)
            AS storage_null,
        SUM(CASE WHEN typeof({quoted_field}) = 'integer' THEN 1 ELSE 0 END)
            AS storage_integer,
        SUM(CASE WHEN typeof({quoted_field}) = 'real' THEN 1 ELSE 0 END)
            AS storage_real,
        SUM(CASE WHEN typeof({quoted_field}) = 'text' THEN 1 ELSE 0 END)
            AS storage_text,
        SUM(CASE WHEN typeof({quoted_field}) = 'blob' THEN 1 ELSE 0 END)
            AS storage_blob
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    """

    result = pd.read_sql_query(query, connection).iloc[0].to_dict()

    declared_type = source_data_dictionary.loc[
        source_data_dictionary["field_name"] == field,
        "declared_sqlite_type",
    ].iloc[0]

    result["field_name"] = field
    result["declared_sqlite_type"] = declared_type

    profile_rows.append(result)

identifier_race_quality = pd.DataFrame(profile_rows)

identifier_race_quality = identifier_race_quality[
    [
        "field_name",
        "declared_sqlite_type",
        "total_rows",
        "null_count",
        "blank_text_count",
        "distinct_non_null_values",
        "storage_null",
        "storage_integer",
        "storage_real",
        "storage_text",
        "storage_blob",
    ]
]

display(identifier_race_quality)

,field_name,declared_sqlite_type,total_rows,null_count,blank_text_count,distinct_non_null_values,storage_null,storage_integer,storage_real,storage_text,storage_blob
0,date,NUMERIC,1851285,0,0,4130,0,0,0,1851285,0
1,course,TEXT,1851285,0,0,528,0,0,0,1851285,0
2,race_id,INTEGER,1851285,0,0,188782,0,1851285,0,0,0
3,off,TEXT,1851285,0,0,1380,0,0,0,1851285,0
4,race_name,TEXT,1851285,0,0,108632,0,0,0,1851285,0
5,type,TEXT,1851285,0,0,4,0,0,0,1851285,0
6,class,TEXT,1851285,0,768544,8,0,0,0,1851285,0
7,pattern,TEXT,1851285,0,1587927,11,0,0,0,1851285,0
8,rating_band,TEXT,1851285,0,1081546,384,0,0,0,1851285,0
9,age_band,TEXT,1851285,0,159,28,0,0,0,1851285,0


In [6]:
# Add interpretable rates without changing any source values.
identifier_race_quality_rates = identifier_race_quality.copy()

identifier_race_quality_rates["null_pct"] = (
    identifier_race_quality_rates["null_count"]
    / identifier_race_quality_rates["total_rows"]
    * 100
)

identifier_race_quality_rates["blank_text_pct"] = (
    identifier_race_quality_rates["blank_text_count"]
    / identifier_race_quality_rates["total_rows"]
    * 100
)

identifier_race_quality_rates["populated_rows"] = (
    identifier_race_quality_rates["total_rows"]
    - identifier_race_quality_rates["null_count"]
    - identifier_race_quality_rates["blank_text_count"]
)

identifier_race_quality_rates["populated_pct"] = (
    identifier_race_quality_rates["populated_rows"]
    / identifier_race_quality_rates["total_rows"]
    * 100
)

rate_columns = [
    "field_name",
    "declared_sqlite_type",
    "total_rows",
    "null_count",
    "null_pct",
    "blank_text_count",
    "blank_text_pct",
    "populated_rows",
    "populated_pct",
    "distinct_non_null_values",
]

display(
    identifier_race_quality_rates[rate_columns].round(
        {
            "null_pct": 4,
            "blank_text_pct": 4,
            "populated_pct": 4,
        }
    )
)

,field_name,declared_sqlite_type,total_rows,null_count,null_pct,blank_text_count,blank_text_pct,populated_rows,populated_pct,distinct_non_null_values
0,date,NUMERIC,1851285,0,0.0,0,0.0000,1851285,100.0000,4130
1,course,TEXT,1851285,0,0.0,0,0.0000,1851285,100.0000,528
2,race_id,INTEGER,1851285,0,0.0,0,0.0000,1851285,100.0000,188782
3,off,TEXT,1851285,0,0.0,0,0.0000,1851285,100.0000,1380
4,race_name,TEXT,1851285,0,0.0,0,0.0000,1851285,100.0000,108632
5,type,TEXT,1851285,0,0.0,0,0.0000,1851285,100.0000,4
6,class,TEXT,1851285,0,0.0,768544,41.5141,1082741,58.4859,8
7,pattern,TEXT,1851285,0,0.0,1587927,85.7743,263358,14.2257,11
8,rating_band,TEXT,1851285,0,0.0,1081546,58.4214,769739,41.5786,384
9,age_band,TEXT,1851285,0,0.0,159,0.0086,1851126,99.9914,28


In [7]:
# Show complete value distributions for fields with relatively few distinct values.
low_cardinality_fields = [
    "type",
    "class",
    "pattern",
    "age_band",
    "sex_rest",
    "going",
]

distribution_frames = []

for field in low_cardinality_fields:
    quoted_field = f'"{field}"'

    distribution = pd.read_sql_query(
        f"""
        SELECT
            {quoted_field} AS stored_value,
            COUNT(*) AS row_count
        FROM data
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY {quoted_field}
        ORDER BY row_count DESC, stored_value
        """,
        connection,
    )

    distribution["field_name"] = field
    distribution["row_pct"] = (
        distribution["row_count"]
        / distribution["row_count"].sum()
        * 100
    )

    distribution_frames.append(distribution)

low_cardinality_distributions = pd.concat(
    distribution_frames,
    ignore_index=True,
)

for field in low_cardinality_fields:
    print(f"\n{field.upper()}")
    display(
        low_cardinality_distributions.loc[
            low_cardinality_distributions["field_name"] == field,
            ["stored_value", "row_count", "row_pct"],
        ].round({"row_pct": 4})
    )


TYPE


,stored_value,row_count,row_pct
0,Flat,1268229,68.5053
1,Hurdle,358441,19.3617
2,Chase,179645,9.7038
3,NH Flat,44970,2.4291



CLASS


,stored_value,row_count,row_pct
4,,768544,41.5141
5,Class 5,312687,16.8903
6,Class 4,307371,16.6031
7,Class 6,191263,10.3314
8,Class 3,124168,6.7071
9,Class 2,81783,4.4176
10,Class 1,63008,3.4035
11,Class 7,2461,0.1329



PATTERN


...

,stored_value,row_count,row_pct
23,4yo+,606576,32.7651
24,3yo+,556237,30.0460
25,3yo,237914,12.8513
26,2yo,179439,9.6927
27,5yo+,145227,7.8447
28,4yo,44999,2.4307
29,4-6yo,24236,1.3091
30,4-7yo,10220,0.5520
31,2yo+,10198,0.5509
32,3-5yo,8450,0.4564



SEX_REST


,stored_value,row_count,row_pct
51,,1612425,87.0976
52,F,123667,6.6801
53,M,52478,2.8347
54,F & M,37354,2.0177
55,C & G,25292,1.3662
56,C & F,69,0.0037



GOING


,stored_value,row_count,row_pct
57,Good,499160,26.9629
58,Standard,343818,18.5719
59,Soft,211462,11.4224
60,Good To Soft,184351,9.9580
61,Good To Firm,177363,9.5805
62,Heavy,116237,6.2787
63,Standard To Slow,62870,3.3960
64,Yielding,48747,2.6331
65,Fast,45728,2.4701
66,Very Soft,43035,2.3246


In [8]:
# Examine the source date field as stored text.
#
# The checks below test observed string structure only. They do not yet
# convert the field into a Python or SQLite date type.

date_profile = pd.read_sql_query(
    f"""
    SELECT
        MIN(date) AS minimum_stored_value,
        MAX(date) AS maximum_stored_value,
        MIN(length(date)) AS minimum_length,
        MAX(length(date)) AS maximum_length,
        SUM(
            CASE
                WHEN date GLOB
                    '[0-9][0-9][0-9][0-9]-[0-9][0-9]-[0-9][0-9]'
                 AND length(date) = 10
                THEN 1 ELSE 0
            END
        ) AS yyyy_mm_dd_shape_count,
        SUM(
            CASE
                WHEN NOT (
                    date GLOB
                        '[0-9][0-9][0-9][0-9]-[0-9][0-9]-[0-9][0-9]'
                    AND length(date) = 10
                )
                THEN 1 ELSE 0
            END
        ) AS other_shape_count,
        COUNT(DISTINCT substr(date, 1, 4)) AS distinct_year_strings
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    """,
    connection,
)

date_year_distribution = pd.read_sql_query(
    f"""
    SELECT
        substr(date, 1, 4) AS stored_year,
        COUNT(*) AS row_count,
        COUNT(DISTINCT date) AS distinct_dates
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY substr(date, 1, 4)
    ORDER BY stored_year
    """,
    connection,
)

display(date_profile)
display(date_year_distribution)

,minimum_stored_value,maximum_stored_value,minimum_length,maximum_length,yyyy_mm_dd_shape_count,other_shape_count,distinct_year_strings
0,2015-01-01,2026-05-27,10,10,1851285,0,12


,stored_year,row_count,distinct_dates
0,2015,157683,365
1,2016,157938,366
2,2017,167494,365
3,2018,172047,363
4,2019,173039,365
5,2020,121683,335
6,2021,175552,365
7,2022,167479,365
8,2023,156075,365
9,2024,169702,365


In [9]:
# Profile the scheduled off-time field exactly as stored.
#
# The source appears to contain both zero-padded and non-zero-padded times,
# so the checks distinguish common H:MM and HH:MM string shapes.

off_profile = pd.read_sql_query(
    f"""
    SELECT
        MIN(length(off)) AS minimum_length,
        MAX(length(off)) AS maximum_length,
        SUM(
            CASE
                WHEN off GLOB '[0-9]:[0-9][0-9]'
                 AND length(off) = 4
                THEN 1 ELSE 0
            END
        ) AS h_mm_shape_count,
        SUM(
            CASE
                WHEN off GLOB '[0-9][0-9]:[0-9][0-9]'
                 AND length(off) = 5
                THEN 1 ELSE 0
            END
        ) AS hh_mm_shape_count,
        SUM(
            CASE
                WHEN NOT (
                    (
                        off GLOB '[0-9]:[0-9][0-9]'
                        AND length(off) = 4
                    )
                    OR
                    (
                        off GLOB '[0-9][0-9]:[0-9][0-9]'
                        AND length(off) = 5
                    )
                )
                THEN 1 ELSE 0
            END
        ) AS other_shape_count,
        COUNT(DISTINCT off) AS distinct_stored_values
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    """,
    connection,
)

off_length_distribution = pd.read_sql_query(
    f"""
    SELECT
        length(off) AS stored_length,
        COUNT(*) AS row_count,
        COUNT(DISTINCT off) AS distinct_values
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY length(off)
    ORDER BY stored_length
    """,
    connection,
)

off_examples = pd.read_sql_query(
    f"""
    SELECT
        off AS stored_value,
        COUNT(*) AS row_count
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY off
    ORDER BY row_count DESC, off
    LIMIT 30
    """,
    connection,
)

display(off_profile)
display(off_length_distribution)
display(off_examples)

,minimum_length,maximum_length,h_mm_shape_count,hh_mm_shape_count,other_shape_count,distinct_stored_values
0,4,5,1600952,250333,0,1380


,stored_length,row_count,distinct_values
0,4,1600952,540
1,5,250333,840


,stored_value,row_count
0,5:00,26917
1,2:15,25663
2,5:30,25611
3,2:50,25016
4,2:00,24321
5,6:00,23861
6,3:00,23769
7,2:40,23607
8,4:00,23478
9,3:10,23354


In [10]:
# Profile integer ranges and potentially special values.
#
# These checks describe the stored numbers only. They do not assume that
# race_id is globally unique or that every ran value is necessarily valid.

integer_field_profile = pd.read_sql_query(
    f"""
    SELECT
        MIN(race_id) AS minimum_race_id,
        MAX(race_id) AS maximum_race_id,
        COUNT(DISTINCT race_id) AS distinct_race_ids,
        SUM(CASE WHEN race_id <= 0 THEN 1 ELSE 0 END)
            AS non_positive_race_id_rows,

        MIN(ran) AS minimum_ran,
        MAX(ran) AS maximum_ran,
        COUNT(DISTINCT ran) AS distinct_ran_values,
        SUM(CASE WHEN ran <= 0 THEN 1 ELSE 0 END)
            AS non_positive_ran_rows
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    """,
    connection,
)

ran_distribution = pd.read_sql_query(
    f"""
    SELECT
        ran AS stored_value,
        COUNT(*) AS row_count,
        ROUND(
            100.0 * COUNT(*) /
            (
                SELECT COUNT(*)
                FROM data
                WHERE {DATA_ROW_PREDICATE}
            ),
            4
        ) AS row_pct
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY ran
    ORDER BY ran
    """,
    connection,
)

display(integer_field_profile)
display(ran_distribution)

,minimum_race_id,maximum_race_id,distinct_race_ids,non_positive_race_id_rows,minimum_ran,maximum_ran,distinct_ran_values,non_positive_ran_rows
0,0,4803590,188782,15,1,40,37,0


,stored_value,row_count,row_pct
0,1,22,0.0012
1,2,688,0.0372
2,3,5949,0.3213
3,4,23856,1.2886
4,5,58098,3.1383
5,6,98636,5.3280
6,7,138894,7.5026
7,8,167367,9.0406
8,9,179289,9.6846
9,10,181640,9.8116


In [11]:
# Inspect the small set of non-positive race_id records.
#
# No records are classified as erroneous here. The purpose is to capture
# their surrounding context for later investigation.

non_positive_race_ids = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        date,
        course,
        race_id,
        off,
        race_name,
        type,
        class,
        pattern,
        rating_band,
        age_band,
        sex_rest,
        dist,
        going,
        ran,
        num,
        pos,
        horse
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND race_id <= 0
    ORDER BY date, course, off, rowid
    """,
    connection,
)

print(f"Rows with race_id <= 0: {len(non_positive_race_ids):,}")
print(
    "Distinct provisional races represented: "
    f"{non_positive_race_ids[['date', 'course', 'off', 'race_name']].drop_duplicates().shape[0]:,}"
)

display(non_positive_race_ids)

Rows with race_id <= 0: 15
Distinct provisional races represented: 1


,source_rowid,date,course,race_id,off,race_name,type,class,pattern,rating_band,age_band,sex_rest,dist,going,ran,num,pos,horse
0,286607,2016-10-24,Ayr,0,4:40,Racing UK Day Pass Just £10 Standard Open Nati...,NH Flat,Class 6,,,4-6yo,,2m,Good To Soft,15,1,1,Loud And Clear (GB)
1,286614,2016-10-24,Ayr,0,4:40,Racing UK Day Pass Just £10 Standard Open Nati...,NH Flat,Class 6,,,4-6yo,,2m,Good To Soft,15,4,2,Donnas Delight (IRE)
2,286624,2016-10-24,Ayr,0,4:40,Racing UK Day Pass Just £10 Standard Open Nati...,NH Flat,Class 6,,,4-6yo,,2m,Good To Soft,15,9,3,Pot Committed (IRE)
3,286625,2016-10-24,Ayr,0,4:40,Racing UK Day Pass Just £10 Standard Open Nati...,NH Flat,Class 6,,,4-6yo,,2m,Good To Soft,15,3,4,Carry On Arcadio (IRE)
4,286626,2016-10-24,Ayr,0,4:40,Racing UK Day Pass Just £10 Standard Open Nati...,NH Flat,Class 6,,,4-6yo,,2m,Good To Soft,15,12,6,Strong Economy (IRE)
5,286635,2016-10-24,Ayr,0,4:40,Racing UK Day Pass Just £10 Standard Open Nati...,NH Flat,Class 6,,,4-6yo,,2m,Good To Soft,15,13,15,The Germany One (IRE)
6,286636,2016-10-24,Ayr,0,4:40,Racing UK Day Pass Just £10 Standard Open Nati...,NH Flat,Class 6,,,4-6yo,,2m,Good To Soft,15,11,14,Solway Storm (IRE)
7,286637,2016-10-24,Ayr,0,4:40,Racing UK Day Pass Just £10 Standard Open Nati...,NH Flat,Class 6,,,4-6yo,,2m,Good To Soft,15,2,13,Captain Sam (GB)
8,286638,2016-10-24,Ayr,0,4:40,Racing UK Day Pass Just £10 Standard Open Nati...,NH Flat,Class 6,,,4-6yo,,2m,Good To Soft,15,8,12,Mullaghboy (IRE)
9,286639,2016-10-24,Ayr,0,4:40,Racing UK Day Pass Just £10 Standard Open Nati...,NH Flat,Class 6,,,4-6yo,,2m,Good To Soft,15,15,11,Seelateralligator (IRE)


In [12]:
# Profile course labels exactly as stored.
#
# This does not yet split venue, jurisdiction, surface, or course variant.
# It only records the observed text structure and common values.

course_profile = pd.read_sql_query(
    f"""
    SELECT
        MIN(length(course)) AS minimum_length,
        MAX(length(course)) AS maximum_length,
        COUNT(DISTINCT course) AS distinct_values,
        SUM(CASE WHEN course LIKE '%(%)%' THEN 1 ELSE 0 END)
            AS rows_with_parentheses,
        COUNT(
            DISTINCT CASE
                WHEN course LIKE '%(%)%' THEN course
            END
        ) AS distinct_values_with_parentheses
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    """,
    connection,
)

course_top_values = pd.read_sql_query(
    f"""
    SELECT
        course AS stored_value,
        COUNT(*) AS row_count,
        ROUND(
            100.0 * COUNT(*) /
            (
                SELECT COUNT(*)
                FROM data
                WHERE {DATA_ROW_PREDICATE}
            ),
            4
        ) AS row_pct
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY course
    ORDER BY row_count DESC, course
    LIMIT 40
    """,
    connection,
)

course_parenthesis_examples = pd.read_sql_query(
    f"""
    SELECT
        course AS stored_value,
        COUNT(*) AS row_count
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND course LIKE '%(%)%'
    GROUP BY course
    ORDER BY row_count DESC, course
    LIMIT 40
    """,
    connection,
)

display(course_profile)
display(course_top_values)
display(course_parenthesis_examples)

,minimum_length,maximum_length,distinct_values,rows_with_parentheses,distinct_values_with_parentheses
0,3,30,528,1101537,339


,stored_value,row_count,row_pct
0,Wolverhampton (AW),64746,3.4974
1,Sha Tin (HK),52010,2.8094
2,Kempton (AW),50268,2.7153
3,Chantilly (FR),48744,2.6330
4,Deauville (FR),48618,2.6262
5,Newcastle (AW),42092,2.2737
6,Lingfield (AW),41632,2.2488
7,Dundalk (AW) (IRE),39075,2.1107
8,Chelmsford (AW),37741,2.0386
9,Southwell (AW),34488,1.8629


,stored_value,row_count
0,Wolverhampton (AW),64746
1,Sha Tin (HK),52010
2,Kempton (AW),50268
3,Chantilly (FR),48744
4,Deauville (FR),48618
5,Newcastle (AW),42092
6,Lingfield (AW),41632
7,Dundalk (AW) (IRE),39075
8,Chelmsford (AW),37741
9,Southwell (AW),34488


In [13]:
# Extract every parenthesised fragment from the distinct stored course labels.
#
# This is exploratory evidence only. A fragment may represent jurisdiction,
# surface, course configuration, or something else.

import re
from collections import Counter

distinct_courses = pd.read_sql_query(
    f"""
    SELECT DISTINCT course
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    ORDER BY course
    """,
    connection,
)["course"]

parenthesised_fragments = Counter()

for course_value in distinct_courses:
    fragments = re.findall(r"\(([^()]*)\)", course_value)

    for fragment in fragments:
        parenthesised_fragments[fragment] += 1

course_fragment_profile = pd.DataFrame(
    [
        {
            "parenthesised_fragment": fragment,
            "distinct_course_labels": label_count,
        }
        for fragment, label_count in parenthesised_fragments.items()
    ]
).sort_values(
    ["distinct_course_labels", "parenthesised_fragment"],
    ascending=[False, True],
    ignore_index=True,
)

print(
    "Distinct parenthesised fragments: "
    f"{len(course_fragment_profile):,}"
)

display(course_fragment_profile)

Distinct parenthesised fragments: 40


,parenthesised_fragment,distinct_course_labels
0,FR,73
1,USA,56
2,AUS,51
3,IRE,27
4,JPN,21
5,GER,17
6,NZ,14
7,SAF,9
8,AW,8
9,ITY,7


In [14]:
# Profile race names exactly as stored.
#
# Repeated race names are expected because the same named race can occur
# across years, meetings, or jurisdictions. No uniqueness assumption is made.

race_name_profile = pd.read_sql_query(
    f"""
    SELECT
        MIN(length(race_name)) AS minimum_length,
        MAX(length(race_name)) AS maximum_length,
        ROUND(AVG(length(race_name)), 2) AS average_length,
        COUNT(DISTINCT race_name) AS distinct_values,

        SUM(
            CASE
                WHEN race_name <> trim(race_name)
                THEN 1 ELSE 0
            END
        ) AS rows_with_outer_whitespace,

        COUNT(
            DISTINCT CASE
                WHEN race_name <> trim(race_name)
                THEN race_name
            END
        ) AS distinct_values_with_outer_whitespace
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    """,
    connection,
)

race_name_top_values = pd.read_sql_query(
    f"""
    SELECT
        race_name AS stored_value,
        COUNT(*) AS row_count,
        COUNT(
            DISTINCT date || '|' || course || '|' || off
        ) AS apparent_race_count
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY race_name
    ORDER BY apparent_race_count DESC, row_count DESC, race_name
    LIMIT 30
    """,
    connection,
)

race_name_longest = pd.read_sql_query(
    f"""
    SELECT DISTINCT
        race_name AS stored_value,
        length(race_name) AS stored_length
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    ORDER BY stored_length DESC, race_name
    LIMIT 20
    """,
    connection,
)

display(race_name_profile)
display(race_name_top_values)
display(race_name_longest)

,minimum_length,maximum_length,average_length,distinct_values,rows_with_outer_whitespace,distinct_values_with_outer_whitespace
0,8,100,47.73,108632,0,0


,stored_value,row_count,apparent_race_count
0,Betway Handicap,6029,679
1,Sky Sports Racing Sky 415 Handicap,2346,275
2,Irish Stallion Farms EBF Maiden,2775,259
3,32Red.com Handicap,2212,254
4,Betway Casino Handicap,2336,244
5,32Red Casino Handicap,1818,197
6,Download The At The Races App Handicap,1707,186
7,32Red Handicap,1438,169
8,Irish Stallion Farms EBF Maiden (Plus 10 Race),1871,168
9,Heed Your Hunch At Betway Handicap,1588,167


,stored_value,stored_length
0,100% Racing TV Profits Back To Racing Apprenti...,100
1,188Bet Casino Apprentice Handicap (Racing Exce...,100
2,188Bet Faller Refunds At Cheltenham Thursday H...,100
3,2026 Annual Badges On Sale Now Handicap Chase ...,100
4,67th Prix Georges Courtois - Trophee Studio Ha...,100
5,69th Prix Georges Courtois - Trophee Studio Ha...,100
6,70th Prix Georges Courtois - Trophee Studio Ha...,100
7,7bets4free.com Fixed Brush Hurdle Series Maide...,100
8,All-Electric Macan EBF Restricted Maiden Stake...,100
9,Anderson Anderson & Brown Handicap Hurdle (Qua...,100


In [15]:
# Investigate the apparent 100-character ceiling in race_name.
#
# Hitting the exact maximum does not prove truncation, but a large number of
# values ending mid-word or without natural punctuation would support that
# interpretation.

race_name_length_distribution = pd.read_sql_query(
    f"""
    SELECT
        length(race_name) AS stored_length,
        COUNT(*) AS row_count,
        COUNT(DISTINCT race_name) AS distinct_values
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY length(race_name)
    ORDER BY stored_length
    """,
    connection,
)

race_name_ceiling_profile = pd.read_sql_query(
    f"""
    SELECT
        COUNT(*) AS rows_at_100_characters,
        COUNT(DISTINCT race_name) AS distinct_names_at_100_characters,
        ROUND(
            100.0 * COUNT(*) /
            (
                SELECT COUNT(*)
                FROM data
                WHERE {DATA_ROW_PREDICATE}
            ),
            4
        ) AS row_pct_at_100_characters
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND length(race_name) = 100
    """,
    connection,
)

race_name_ceiling_endings = pd.read_sql_query(
    f"""
    SELECT DISTINCT
        race_name AS stored_value,
        substr(race_name, -20) AS final_20_characters
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND length(race_name) = 100
    ORDER BY race_name
    LIMIT 50
    """,
    connection,
)

display(race_name_length_distribution.tail(15))
display(race_name_ceiling_profile)
display(race_name_ceiling_endings)

,stored_length,row_count,distinct_values
78,86,5146,458
79,87,5604,487
80,88,5321,462
81,89,4941,444
82,90,4925,459
83,91,5167,475
84,92,4730,489
85,93,4967,517
86,94,6163,623
87,95,7123,723


,rows_at_100_characters,distinct_names_at_100_characters,row_pct_at_100_characters
0,2861,303,0.1545


,stored_value,final_20_characters
0,100% Racing TV Profits Back To Racing Apprenti...,ice Training Series)
1,188Bet Casino Apprentice Handicap (Racing Exce...,er Sprint Qualifier)
2,188Bet Faller Refunds At Cheltenham Thursday H...,Hurdle Series Qual)
3,2026 Annual Badges On Sale Now Handicap Chase ...,ualifier) (GBB Race)
4,67th Prix Georges Courtois - Trophee Studio Ha...,men Amateurs) (Poly)
5,69th Prix Georges Courtois - Trophee Studio Ha...,men Amateurs) (Poly)
6,70th Prix Georges Courtois - Trophee Studio Ha...,men Amateurs) (Poly)
7,7bets4free.com Fixed Brush Hurdle Series Maide...,Hurdle Series Qual)
8,All-Electric Macan EBF Restricted Maiden Stake...,/IRE Incentive Race)
9,Anderson Anderson & Brown Handicap Hurdle (Qua...,Mile Hurdle Series)


In [16]:
# Profile race distance labels exactly as stored.
#
# No conversion to furlongs, metres, yards, or miles is attempted here.
# The aim is to document the source notation and identify unusual codes.

distance_profile = pd.read_sql_query(
    f"""
    SELECT
        MIN(length(dist)) AS minimum_length,
        MAX(length(dist)) AS maximum_length,
        COUNT(DISTINCT dist) AS distinct_values,
        SUM(
            CASE
                WHEN dist <> trim(dist)
                THEN 1 ELSE 0
            END
        ) AS rows_with_outer_whitespace
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    """,
    connection,
)

distance_distribution = pd.read_sql_query(
    f"""
    SELECT
        dist AS stored_value,
        COUNT(*) AS row_count,
        ROUND(
            100.0 * COUNT(*) /
            (
                SELECT COUNT(*)
                FROM data
                WHERE {DATA_ROW_PREDICATE}
            ),
            4
        ) AS row_pct
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY dist
    ORDER BY row_count DESC, dist
    """,
    connection,
)

display(distance_profile)
display(distance_distribution)

,minimum_length,maximum_length,distinct_values,rows_with_outer_whitespace
0,2,5,63,0


,stored_value,row_count,row_pct
0,1m,225603,12.1863
1,6f,213342,11.5240
2,7f,205608,11.1062
3,2m,131238,7.0890
4,5f,108768,5.8753
...,...,...,...
58,4m1½f,144,0.0078
59,3m7½f,137,0.0074
60,4m1f,121,0.0065
61,3m7f,70,0.0038


In [17]:
# Show every distinct stored distance value without notebook row truncation.
with pd.option_context("display.max_rows", None):
    display(distance_distribution)

,stored_value,row_count,row_pct
0,1m,225603,12.1863
1,6f,213342,11.5240
2,7f,205608,11.1062
3,2m,131238,7.0890
4,5f,108768,5.8753
5,1m2f,108760,5.8748
6,1m4f,80320,4.3386
7,2m4f,63594,3.4351
8,2m½f,48639,2.6273
9,1m1½f,43606,2.3554


In [18]:
# Profile rating_band notation without parsing it into numeric limits.
#
# The categories below describe visible string structure only. They are not
# racing-domain classifications and do not imply any cleaning decision.

rating_band_distribution = pd.read_sql_query(
    f"""
    SELECT
        rating_band AS stored_value,
        COUNT(*) AS row_count
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY rating_band
    ORDER BY row_count DESC, rating_band
    """,
    connection,
)

def classify_rating_band_shape(value):
    """Describe the visible structure of one stored rating-band value."""
    if value == "":
        return "blank"

    if re.fullmatch(r"\d+-\d+", value):
        return "numeric range"

    if re.fullmatch(r"\d+", value):
        return "single number"

    if re.fullmatch(r"\d+\+", value):
        return "number with plus"

    if re.fullmatch(r"\d+-\d+\s+\([^)]+\)", value):
        return "numeric range with parenthetical text"

    if re.fullmatch(r"\d+\s+\([^)]+\)", value):
        return "single number with parenthetical text"

    return "other text shape"


rating_band_distribution["observed_shape"] = (
    rating_band_distribution["stored_value"].map(
        classify_rating_band_shape
    )
)

rating_band_shape_profile = (
    rating_band_distribution
    .groupby("observed_shape", as_index=False)
    .agg(
        distinct_values=("stored_value", "count"),
        row_count=("row_count", "sum"),
    )
)

rating_band_shape_profile["row_pct"] = (
    rating_band_shape_profile["row_count"]
    / rating_band_shape_profile["row_count"].sum()
    * 100
)

rating_band_shape_profile = rating_band_shape_profile.sort_values(
    ["row_count", "observed_shape"],
    ascending=[False, True],
    ignore_index=True,
)

display(rating_band_shape_profile.round({"row_pct": 4}))

# Show every value that does not match the simple expected shapes.
rating_band_other_values = rating_band_distribution.loc[
    rating_band_distribution["observed_shape"]
    == "other text shape"
].copy()

print(
    "Distinct values with other text shapes: "
    f"{len(rating_band_other_values):,}"
)

with pd.option_context("display.max_rows", None):
    display(rating_band_other_values)

,observed_shape,distinct_values,row_count,row_pct
0,blank,1,1081546,58.4214
1,numeric range,381,769622,41.5723
2,other text shape,2,117,0.0063


Distinct values with other text shapes: 2


,stored_value,row_count,observed_shape
92,--,110,other text shape
360,(75-100),7,other text shape


In [19]:
# Inspect the context of exceptional rating_band values.
#
# This helps distinguish isolated formatting issues from repeated
# source conventions without changing the values.

exceptional_rating_band_rows = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        date,
        course,
        race_id,
        off,
        race_name,
        type,
        class,
        pattern,
        rating_band,
        age_band,
        sex_rest,
        dist,
        going,
        ran,
        num,
        pos,
        horse
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND rating_band IN ('--', '(75-100)')
    ORDER BY rating_band, date, course, off, rowid
    """,
    connection,
)

exceptional_rating_band_races = (
    exceptional_rating_band_rows[
        [
            "rating_band",
            "date",
            "course",
            "race_id",
            "off",
            "race_name",
            "type",
            "class",
            "pattern",
            "age_band",
            "sex_rest",
            "dist",
            "going",
            "ran",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["rating_band", "date", "course", "off"],
        ignore_index=True,
    )
)

print(
    "Runner rows with exceptional rating_band values: "
    f"{len(exceptional_rating_band_rows):,}"
)
print(
    "Distinct apparent races represented: "
    f"{len(exceptional_rating_band_races):,}"
)

with pd.option_context(
    "display.max_rows",
    None,
    "display.max_colwidth",
    120,
):
    display(exceptional_rating_band_races)

Runner rows with exceptional rating_band values: 117
Distinct apparent races represented: 13


,rating_band,date,course,race_id,off,race_name,type,class,pattern,age_band,sex_rest,dist,going,ran
0,(75-100),2026-02-25,Happy Valley,914010,11:40,Magazine Gap Handicap (B Course) (Turf),Flat,Class 2,,,,1m1f,Good,7
1,--,2016-02-19,Jebel Ali (UAE),644459,11:20,Jebel Ali Stakes () (Dirt),Flat,,Listed,3yo+,,1m2f,Fast,11
2,--,2017-11-03,Ohi (JPN),689802,10:07,JBC Sprint (Local (Dirt),Flat,,Grade 1,3yo+,,6f,Muddy,16
3,--,2017-11-03,Ohi (JPN),689592,11:07,JBC Classic (Local (Dirt),Flat,,Grade 1,3yo+,,1m1f,Muddy,13
4,--,2018-01-26,Jebel Ali (UAE),693130,11:30,Jebel Ali Mile Sponsored By Shadwell Farm (Dirt),Flat,,Group 3,3yo+,,1m,Fast,7
5,--,2018-04-30,NAGOYA (JPN),701640,11:07,Kakitsubata Kinen (Handicap) (4yo+) (Local (Dirt),Flat,,Grade 3,4yo+,,7f,Standard,12
6,--,2018-05-13,Les Landes (JER),702426,4:50,Bloodstock Advisory Services Handicap,Flat,,,3yo+,,1m½f,Good To Firm,8
7,--,2018-05-30,Urawa (JPN),704187,11:07,Sakitama Hai (Local (Dirt),Flat,,Grade 2,4yo+,,7f,Standard,11
8,--,2018-07-22,Les Landes (JER),707986,2:30,Milbrook July Conditions Hurdle,Hurdle,,,3yo+,,2m,Firm,6
9,--,2018-07-22,Les Landes (JER),707991,3:05,Patricia K Pritchard Handicap Sprint,Flat,,,3yo+,,5½f,Firm,5


In [20]:
# Inspect every row where age_band is stored as a blank string.
#
# The aim is to determine whether blankness occurs at race level and whether
# it is associated with particular jurisdictions or race types.

blank_age_band_rows = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        date,
        course,
        race_id,
        off,
        race_name,
        type,
        class,
        pattern,
        rating_band,
        age_band,
        sex_rest,
        dist,
        going,
        ran,
        num,
        pos,
        horse
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND length(trim(age_band)) = 0
    ORDER BY date, course, off, rowid
    """,
    connection,
)

blank_age_band_races = (
    blank_age_band_rows[
        [
            "date",
            "course",
            "race_id",
            "off",
            "race_name",
            "type",
            "class",
            "pattern",
            "rating_band",
            "sex_rest",
            "dist",
            "going",
            "ran",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["date", "course", "off"],
        ignore_index=True,
    )
)

print(f"Runner rows with blank age_band: {len(blank_age_band_rows):,}")
print(
    "Distinct apparent races represented: "
    f"{len(blank_age_band_races):,}"
)

with pd.option_context(
    "display.max_rows",
    None,
    "display.max_colwidth",
    120,
):
    display(blank_age_band_races)

Runner rows with blank age_band: 159
Distinct apparent races represented: 13


,date,course,race_id,off,race_name,type,class,pattern,rating_band,sex_rest,dist,going,ran
0,2015-06-07,Baden-Baden (GER),630735,3:30,Badener Roulette Preis (Turf),Flat,,,,,1m2f,Good,5
1,2021-05-27,Limerick (IRE),785625,8:10,Athea INH Flat Race,NH Flat,,,,,2m,Yielding To Soft,18
2,2021-07-16,Kilbeggan (IRE),789491,8:10,Follow Kilbeggan On Facebook INH Flat Race,NH Flat,,,,,2m4f,Good,11
3,2026-02-14,Sha Tin,913671,04:45,Tvb Lok Sin Tong Charity Corner Hcp (C4) (Handicap) (Turf),Flat,,,,,6f,Good To Firm,12
4,2026-02-14,Sha Tin,913672,05:40,Tvb Yan Oi Tong Charity Show Hcp (C4) (Handicap) (Turf),Flat,,,,,7f,Good To Firm,14
5,2026-02-14,Sha Tin,913673,06:10,Tvb Yan Chai Charity Show Hcp (C3) (Handicap) (Turf),Flat,,,,,6f,Good To Firm,11
6,2026-02-14,Sha Tin,913674,06:40,Tvb Pok Oi Charity Show Hcp (C4) (Handicap) (Turf),Flat,,,,,1m1f,Good To Firm,14
7,2026-02-14,Sha Tin,913675,07:10,Tvb Lo And Behold Hcp (C3) (Handicap) (Dirt),Flat,,,,,1m,Standard,14
8,2026-02-14,Sha Tin,913676,07:40,Tvb The Queen Of News Hcp (C4) (Handicap) (Dirt),Flat,,,,,1m,Standard,14
9,2026-02-14,Sha Tin,913677,08:15,The Tvb Cup (C2) (Handicap) (Turf),Flat,,,,,6f,Good To Firm,11


In [21]:
# Inspect every row where going is stored as a blank string.
#
# This determines whether the missing values affect isolated runners,
# complete races, particular meetings, or particular jurisdictions.

blank_going_rows = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        date,
        course,
        race_id,
        off,
        race_name,
        type,
        class,
        pattern,
        rating_band,
        age_band,
        sex_rest,
        dist,
        going,
        ran,
        num,
        pos,
        horse
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND length(trim(going)) = 0
    ORDER BY date, course, off, rowid
    """,
    connection,
)

blank_going_races = (
    blank_going_rows[
        [
            "date",
            "course",
            "race_id",
            "off",
            "race_name",
            "type",
            "class",
            "pattern",
            "rating_band",
            "age_band",
            "sex_rest",
            "dist",
            "ran",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["date", "course", "off"],
        ignore_index=True,
    )
)

print(f"Runner rows with blank going: {len(blank_going_rows):,}")
print(
    "Distinct apparent races represented: "
    f"{len(blank_going_races):,}"
)

with pd.option_context(
    "display.max_rows",
    None,
    "display.max_colwidth",
    120,
):
    display(blank_going_races)

Runner rows with blank going: 45
Distinct apparent races represented: 4


,date,course,race_id,off,race_name,type,class,pattern,rating_band,age_band,sex_rest,dist,ran
0,2015-02-27,Sakhir (BHR),629750,12:15,SH. ISA Bin Salman Bin Hamad Al Khalifa Cup (Local (Turf),Flat,,Grade 2,,3yo+,,1m,12
1,2015-05-01,Tours (FR),625766,4:30,Prix des Fleurs de Touraine (Conditions) (4yo+) (Turf),Flat,,,,4yo+,,1m,11
2,2015-11-17,Doha (QA),639711,4:10,Barzan Cup (Handicap) (Turf),Flat,,,,3yo+,,1m,11
3,2019-12-19,Wolverhampton (AW),745750,8:30,Betway Casino Handicap,Flat,Class 6,,0-55,4yo+,,1m6f,11


In [22]:
# Validate the numeric components of off-time strings as stored.
#
# Both H:MM and HH:MM formats are accepted. This check only asks whether
# the hour and minute portions fall within ordinary clock ranges.

off_component_profile = pd.read_sql_query(
    f"""
    WITH parsed_off AS (
        SELECT
            off,
            CAST(substr(off, 1, instr(off, ':') - 1) AS INTEGER)
                AS stored_hour,
            CAST(substr(off, instr(off, ':') + 1, 2) AS INTEGER)
                AS stored_minute
        FROM data
        WHERE {DATA_ROW_PREDICATE}
    )
    SELECT
        MIN(stored_hour) AS minimum_hour,
        MAX(stored_hour) AS maximum_hour,
        MIN(stored_minute) AS minimum_minute,
        MAX(stored_minute) AS maximum_minute,

        SUM(
            CASE
                WHEN stored_hour < 0 OR stored_hour > 23
                THEN 1 ELSE 0
            END
        ) AS invalid_hour_rows,

        SUM(
            CASE
                WHEN stored_minute < 0 OR stored_minute > 59
                THEN 1 ELSE 0
            END
        ) AS invalid_minute_rows,

        COUNT(
            DISTINCT CASE
                WHEN stored_hour < 0 OR stored_hour > 23
                  OR stored_minute < 0 OR stored_minute > 59
                THEN off
            END
        ) AS distinct_invalid_values
    FROM parsed_off
    """,
    connection,
)

off_hour_distribution = pd.read_sql_query(
    f"""
    SELECT
        CAST(substr(off, 1, instr(off, ':') - 1) AS INTEGER)
            AS stored_hour,
        COUNT(*) AS row_count,
        COUNT(DISTINCT off) AS distinct_off_values
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY stored_hour
    ORDER BY stored_hour
    """,
    connection,
)

display(off_component_profile)
display(off_hour_distribution)

,minimum_hour,maximum_hour,minimum_minute,maximum_minute,invalid_hour_rows,invalid_minute_rows,distinct_invalid_values
0,0,23,0,59,0,0,0


,stored_hour,row_count,distinct_off_values
0,0,167,16
1,1,171356,69
2,2,276436,73
3,3,274007,76
4,4,247001,77
5,5,217114,80
6,6,151027,80
7,7,140310,79
8,8,96760,77
9,9,35770,74


In [23]:
# Test race-level consistency within the provisional race grouping:
# date + course + off + race_name.
#
# These fields are repeated on every runner row. A field should normally have
# one stored value within a race, but this test records evidence rather than
# imposing that assumption.

repeated_race_fields = [
    "race_id",
    "type",
    "class",
    "pattern",
    "rating_band",
    "age_band",
    "sex_rest",
    "dist",
    "going",
    "ran",
]

race_consistency_rows = []

for field in repeated_race_fields:
    quoted_field = f'"{field}"'

    consistency_result = pd.read_sql_query(
        f"""
        WITH apparent_races AS (
            SELECT
                date,
                course,
                off,
                race_name,
                COUNT(*) AS runner_rows,
                COUNT(DISTINCT {quoted_field}) AS distinct_stored_values
            FROM data
            WHERE {DATA_ROW_PREDICATE}
            GROUP BY
                date,
                course,
                off,
                race_name
        )
        SELECT
            COUNT(*) AS apparent_race_count,
            SUM(
                CASE
                    WHEN distinct_stored_values > 1
                    THEN 1 ELSE 0
                END
            ) AS inconsistent_race_count,
            MAX(distinct_stored_values) AS maximum_values_within_race
        FROM apparent_races
        """,
        connection,
    ).iloc[0]

    race_consistency_rows.append(
        {
            "field_name": field,
            "apparent_race_count": int(
                consistency_result["apparent_race_count"]
            ),
            "inconsistent_race_count": int(
                consistency_result["inconsistent_race_count"]
            ),
            "maximum_values_within_race": int(
                consistency_result["maximum_values_within_race"]
            ),
        }
    )

race_level_consistency = pd.DataFrame(race_consistency_rows)

race_level_consistency["inconsistent_race_pct"] = (
    race_level_consistency["inconsistent_race_count"]
    / race_level_consistency["apparent_race_count"]
    * 100
)

display(
    race_level_consistency.round(
        {"inconsistent_race_pct": 6}
    )
)

,field_name,apparent_race_count,inconsistent_race_count,maximum_values_within_race,inconsistent_race_pct
0,race_id,189043,0,1,0.0
1,type,189043,0,1,0.0
2,class,189043,0,1,0.0
3,pattern,189043,0,1,0.0
4,rating_band,189043,0,1,0.0
5,age_band,189043,0,1,0.0
6,sex_rest,189043,0,1,0.0
7,dist,189043,0,1,0.0
8,going,189043,0,1,0.0
9,ran,189043,0,1,0.0


In [24]:
# Compare the stored ran value with the number of physical rows in each
# provisional race grouping.
#
# A mismatch does not automatically indicate an error. It may reflect
# withdrawals, non-runners, missing source rows, or a different domain meaning
# of ran. This cell records the observed relationship only.

ran_row_count_comparison = pd.read_sql_query(
    f"""
    WITH apparent_races AS (
        SELECT
            date,
            course,
            off,
            race_name,
            MIN(race_id) AS race_id,
            MIN(ran) AS stored_ran,
            COUNT(*) AS physical_runner_rows
        FROM data
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY
            date,
            course,
            off,
            race_name
    )
    SELECT
        COUNT(*) AS apparent_race_count,

        SUM(
            CASE
                WHEN stored_ran = physical_runner_rows
                THEN 1 ELSE 0
            END
        ) AS matching_race_count,

        SUM(
            CASE
                WHEN stored_ran <> physical_runner_rows
                THEN 1 ELSE 0
            END
        ) AS mismatching_race_count,

        MIN(stored_ran - physical_runner_rows)
            AS minimum_stored_minus_rows,

        MAX(stored_ran - physical_runner_rows)
            AS maximum_stored_minus_rows
    FROM apparent_races
    """,
    connection,
)

ran_difference_distribution = pd.read_sql_query(
    f"""
    WITH apparent_races AS (
        SELECT
            date,
            course,
            off,
            race_name,
            MIN(ran) AS stored_ran,
            COUNT(*) AS physical_runner_rows
        FROM data
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY
            date,
            course,
            off,
            race_name
    )
    SELECT
        stored_ran - physical_runner_rows AS stored_ran_minus_rows,
        COUNT(*) AS apparent_race_count
    FROM apparent_races
    GROUP BY stored_ran_minus_rows
    ORDER BY stored_ran_minus_rows
    """,
    connection,
)

display(ran_row_count_comparison)

with pd.option_context("display.max_rows", None):
    display(ran_difference_distribution)

,apparent_race_count,matching_race_count,mismatching_race_count,minimum_stored_minus_rows,maximum_stored_minus_rows
0,189043,189038,5,0,4


,stored_ran_minus_rows,apparent_race_count
0,0,189038
1,1,4
2,4,1


In [25]:
# Inspect the five apparent races where stored ran does not equal the number
# of physical runner rows.
#
# The difference may reflect missing runner records, non-runners excluded
# from the table, or another source convention. No cause is assumed yet.

ran_mismatch_races = pd.read_sql_query(
    f"""
    WITH apparent_races AS (
        SELECT
            date,
            course,
            off,
            race_name,
            MIN(race_id) AS race_id,
            MIN(type) AS type,
            MIN(class) AS class,
            MIN(pattern) AS pattern,
            MIN(rating_band) AS rating_band,
            MIN(age_band) AS age_band,
            MIN(sex_rest) AS sex_rest,
            MIN(dist) AS dist,
            MIN(going) AS going,
            MIN(ran) AS stored_ran,
            COUNT(*) AS physical_runner_rows
        FROM data
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY
            date,
            course,
            off,
            race_name
    )
    SELECT
        *,
        stored_ran - physical_runner_rows AS stored_ran_minus_rows
    FROM apparent_races
    WHERE stored_ran <> physical_runner_rows
    ORDER BY
        stored_ran_minus_rows DESC,
        date,
        course,
        off
    """,
    connection,
)

print(f"Races with ran/row-count mismatches: {len(ran_mismatch_races):,}")

with pd.option_context(
    "display.max_rows",
    None,
    "display.max_colwidth",
    120,
):
    display(ran_mismatch_races)

Races with ran/row-count mismatches: 5


,date,course,off,race_name,race_id,type,class,pattern,rating_band,age_band,sex_rest,dist,going,stored_ran,physical_runner_rows,stored_ran_minus_rows
0,2024-09-26,Funabashi (JPN),11:07,Marine Cup (Local (Fillies) (Dirt),878244,Flat,,Grade 3,,3yo,F,1m1f,Standard,6,2,4
1,2024-06-18,Nantes (FR),2:14,Prix Sarah Gosse (Conditions Hurdle) (Turf),873374,Hurdle,,,,3yo,,2m1½f,Heavy,8,7,1
2,2024-06-26,Ohi (JPN),11:07,Teio Sho (Local (Dirt),871722,Flat,,Grade 1,,4yo+,,1m2f,Standard,5,4,1
3,2024-09-03,Morioka (JPN),11:07,Kozukata Sho (Local (Dirt),876244,Flat,,Grade 3,,3yo,,1m2f,Standard,5,4,1
4,2025-10-09,Ohi (JPN),11:07,Tokyo Hai (Local (Dirt),905803,Flat,,Grade 3,,3yo+,,6f,Standard,16,15,1


In [26]:
# Inspect the stored runner rows for the five races where ran exceeds the
# number of physical source rows.
#
# Gaps in num or pos may help describe which records are absent, but this
# still does not establish why those records are missing.

ran_mismatch_runner_rows = pd.read_sql_query(
    f"""
    WITH mismatch_races AS (
        SELECT
            date,
            course,
            off,
            race_name
        FROM data
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY
            date,
            course,
            off,
            race_name
        HAVING MIN(ran) <> COUNT(*)
    )
    SELECT
        d.rowid AS source_rowid,
        d.date,
        d.course,
        d.race_id,
        d.off,
        d.race_name,
        d.ran AS stored_ran,
        d.num,
        d.pos,
        d.horse,
        d.sp,
        d.comment
    FROM data AS d
    INNER JOIN mismatch_races AS m
        ON d.date = m.date
       AND d.course = m.course
       AND d.off = m.off
       AND d.race_name = m.race_name
    WHERE d.rowid <> 1
    ORDER BY
        d.date,
        d.course,
        d.off,
        d.pos,
        d.num,
        d.rowid
    """,
    connection,
)

with pd.option_context(
    "display.max_rows",
    None,
    "display.max_colwidth",
    120,
):
    display(ran_mismatch_runner_rows)

,source_rowid,date,course,race_id,off,race_name,stored_ran,num,pos,horse,sp,comment
0,1524983,2024-06-18,Nantes (FR),873374,2:14,Prix Sarah Gosse (Conditions Hurdle) (Turf),8,,1,Pyrrhaa (FR),74/10,
1,1524982,2024-06-18,Nantes (FR),873374,2:14,Prix Sarah Gosse (Conditions Hurdle) (Turf),8,,2,Lhubert De Houelle (FR),58/10,
2,1524981,2024-06-18,Nantes (FR),873374,2:14,Prix Sarah Gosse (Conditions Hurdle) (Turf),8,,3,Bakarelo (IRE),7/5F,
3,1524980,2024-06-18,Nantes (FR),873374,2:14,Prix Sarah Gosse (Conditions Hurdle) (Turf),8,,4,Shalez (FR),19/1,
4,1524979,2024-06-18,Nantes (FR),873374,2:14,Prix Sarah Gosse (Conditions Hurdle) (Turf),8,,5,Bonham Strand (FR),4/1,
5,1524978,2024-06-18,Nantes (FR),873374,2:14,Prix Sarah Gosse (Conditions Hurdle) (Turf),8,,6,Lulu Stars (FR),22/1,
6,1524977,2024-06-18,Nantes (FR),873374,2:14,Prix Sarah Gosse (Conditions Hurdle) (Turf),8,,7,Cockling (FR),25/1,
7,1528630,2024-06-26,Ohi (JPN),871722,11:07,Teio Sho (Local (Dirt),5,12,1,Kings Sword (JPN),,
8,1528614,2024-06-26,Ohi (JPN),871722,11:07,Teio Sho (Local (Dirt),5,8,2,Wilson Tesoro (JPN),,
9,1528615,2024-06-26,Ohi (JPN),871722,11:07,Teio Sho (Local (Dirt),5,3,3,Diktaean (JPN),,


## Group 1 findings — Identifier and race-description fields

### Storage and completeness

The 14 fields in this group contain no SQL `NULL` values.

Missing race-description information is instead stored as blank text, principally in:

- `class`;
- `pattern`;
- `rating_band`;
- `sex_rest`.

Smaller blank populations occur in:

- `age_band`;
- `going`.

The fields `race_id` and `ran` are stored consistently as SQLite integers. The field `date` is declared `NUMERIC` but is stored entirely as text.

### Race-level consistency

Within the provisional race grouping formed from `date`, `course`, `off`, and `race_name`, every repeated race-description field has exactly one stored value.

This supports the interpretation that the following are race-level attributes repeated on runner rows:

- `race_id`;
- `type`;
- `class`;
- `pattern`;
- `rating_band`;
- `age_band`;
- `sex_rest`;
- `dist`;
- `going`;
- `ran`.

### Fields with strong structural consistency

- `date` always has the stored shape `YYYY-MM-DD`.
- `off` always has the shape `H:MM` or `HH:MM`, with valid clock components.
- `type` contains four stored categories.
- `dist` uses a compact and regular miles-and-furlongs notation.
- `ran` is always positive and ranges from 1 to 40.

### Identified quality issues

#### `race_id`

One complete 15-runner race has `race_id = 0`.

The value is therefore an identifier defect affecting one race, rather than a general missing-value convention.

#### `race_name`

The field has an apparent 100-character ceiling.

There are 303 distinct race names at exactly 100 characters, and some end mid-word. This is evidence of probable source truncation.

#### `rating_band`

Most populated values are simple numeric ranges.

Two exceptional stored forms occur:

- `--`, affecting 110 rows across 12 races;
- `(75-100)`, affecting seven rows from one race.

These require later domain investigation.

#### `age_band`

There are 159 blank rows across 13 complete races. The blankness is concentrated in particular meetings and jurisdictions rather than scattered among individual runners.

#### `going`

There are 45 blank rows across four complete races. These are race-level source omissions.

#### `ran`

The stored value equals the number of physical runner rows in 189,038 of 189,043 apparent races.

Five races have fewer stored runner rows than the stated `ran` value. In each case, the available finishing positions are contiguous from first place onward, indicating that the final one or more runner records are absent from the source table.

### Fields requiring separate domain investigation

- parsing the composite `course` field;
- interpreting course parenthetical fragments;
- converting `dist` to a standard numeric measure;
- interpreting blank `class`, `pattern`, and `sex_rest` values;
- interpreting `rating_band` placeholders and formatting variants;
- determining the source convention behind the five incomplete races;
- determining whether truncated race names can be recovered from another supplied source.

## Group 2 — Runner identity and demographics

The next fields to profile are:

- `num`;
- `horse`;
- `age`;
- `sex`;
- `jockey`;
- `trainer`.

In [27]:
# Define the runner identity and demographic field group.
runner_identity_fields = [
    "num",
    "horse",
    "age",
    "sex",
    "jockey",
    "trainer",
]

# Build the same source-quality summary used for the first field group.
runner_identity_profile_rows = []

for field in runner_identity_fields:
    quoted_field = f'"{field}"'

    query = f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN {quoted_field} IS NULL THEN 1 ELSE 0 END)
            AS null_count,
        SUM(
            CASE
                WHEN typeof({quoted_field}) = 'text'
                 AND length(trim({quoted_field})) = 0
                THEN 1 ELSE 0
            END
        ) AS blank_text_count,
        COUNT(DISTINCT {quoted_field}) AS distinct_non_null_values,
        SUM(CASE WHEN typeof({quoted_field}) = 'null' THEN 1 ELSE 0 END)
            AS storage_null,
        SUM(CASE WHEN typeof({quoted_field}) = 'integer' THEN 1 ELSE 0 END)
            AS storage_integer,
        SUM(CASE WHEN typeof({quoted_field}) = 'real' THEN 1 ELSE 0 END)
            AS storage_real,
        SUM(CASE WHEN typeof({quoted_field}) = 'text' THEN 1 ELSE 0 END)
            AS storage_text,
        SUM(CASE WHEN typeof({quoted_field}) = 'blob' THEN 1 ELSE 0 END)
            AS storage_blob
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    """

    result = pd.read_sql_query(query, connection).iloc[0].to_dict()

    declared_type = source_data_dictionary.loc[
        source_data_dictionary["field_name"] == field,
        "declared_sqlite_type",
    ].iloc[0]

    result["field_name"] = field
    result["declared_sqlite_type"] = declared_type

    runner_identity_profile_rows.append(result)

runner_identity_quality = pd.DataFrame(
    runner_identity_profile_rows
)

runner_identity_quality = runner_identity_quality[
    [
        "field_name",
        "declared_sqlite_type",
        "total_rows",
        "null_count",
        "blank_text_count",
        "distinct_non_null_values",
        "storage_null",
        "storage_integer",
        "storage_real",
        "storage_text",
        "storage_blob",
    ]
]

display(runner_identity_quality)

,field_name,declared_sqlite_type,total_rows,null_count,blank_text_count,distinct_non_null_values,storage_null,storage_integer,storage_real,storage_text,storage_blob
0,num,INTEGER,1851285,0,7032,42,0,1844253,0,7032,0
1,horse,TEXT,1851285,0,0,208631,0,0,0,1851285,0
2,age,INTEGER,1851285,0,0,19,0,1851285,0,0,0
3,sex,TEXT,1851285,0,0,8,0,0,0,1851285,0
4,jockey,TEXT,1851285,0,2,7918,0,0,0,1851285,0
5,trainer,TEXT,1851285,0,9,10709,0,0,0,1851285,0


In [28]:
# Add comparable completeness rates for runner identity fields.
runner_identity_quality_rates = runner_identity_quality.copy()

runner_identity_quality_rates["null_pct"] = (
    runner_identity_quality_rates["null_count"]
    / runner_identity_quality_rates["total_rows"]
    * 100
)

runner_identity_quality_rates["blank_text_pct"] = (
    runner_identity_quality_rates["blank_text_count"]
    / runner_identity_quality_rates["total_rows"]
    * 100
)

runner_identity_quality_rates["populated_rows"] = (
    runner_identity_quality_rates["total_rows"]
    - runner_identity_quality_rates["null_count"]
    - runner_identity_quality_rates["blank_text_count"]
)

runner_identity_quality_rates["populated_pct"] = (
    runner_identity_quality_rates["populated_rows"]
    / runner_identity_quality_rates["total_rows"]
    * 100
)

rate_columns = [
    "field_name",
    "declared_sqlite_type",
    "total_rows",
    "null_count",
    "null_pct",
    "blank_text_count",
    "blank_text_pct",
    "populated_rows",
    "populated_pct",
    "distinct_non_null_values",
    "storage_integer",
    "storage_text",
]

display(
    runner_identity_quality_rates[rate_columns].round(
        {
            "null_pct": 4,
            "blank_text_pct": 4,
            "populated_pct": 4,
        }
    )
)

,field_name,declared_sqlite_type,total_rows,null_count,null_pct,blank_text_count,blank_text_pct,populated_rows,populated_pct,distinct_non_null_values,storage_integer,storage_text
0,num,INTEGER,1851285,0,0.0,7032,0.3798,1844253,99.6202,42,1844253,7032
1,horse,TEXT,1851285,0,0.0,0,0.0000,1851285,100.0000,208631,0,1851285
2,age,INTEGER,1851285,0,0.0,0,0.0000,1851285,100.0000,19,1851285,0
3,sex,TEXT,1851285,0,0.0,0,0.0000,1851285,100.0000,8,0,1851285
4,jockey,TEXT,1851285,0,0.0,2,0.0001,1851283,99.9999,7918,0,1851285
5,trainer,TEXT,1851285,0,0.0,9,0.0005,1851276,99.9995,10709,0,1851285


In [29]:
# Show complete stored-value distributions for age and sex.
#
# These are descriptive source profiles only. No biological or racing-domain
# assumptions are imposed at this stage.

age_distribution = pd.read_sql_query(
    f"""
    SELECT
        age AS stored_value,
        COUNT(*) AS row_count,
        ROUND(
            100.0 * COUNT(*) /
            (
                SELECT COUNT(*)
                FROM data
                WHERE {DATA_ROW_PREDICATE}
            ),
            4
        ) AS row_pct
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY age
    ORDER BY age
    """,
    connection,
)

sex_distribution = pd.read_sql_query(
    f"""
    SELECT
        sex AS stored_value,
        COUNT(*) AS row_count,
        ROUND(
            100.0 * COUNT(*) /
            (
                SELECT COUNT(*)
                FROM data
                WHERE {DATA_ROW_PREDICATE}
            ),
            4
        ) AS row_pct
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY sex
    ORDER BY row_count DESC, sex
    """,
    connection,
)

display(age_distribution)
display(sex_distribution)

,stored_value,row_count,row_pct
0,1,5,0.0003
1,2,180383,9.7437
2,3,400774,21.6484
3,4,342278,18.4887
4,5,299403,16.1727
5,6,231753,12.5185
6,7,161945,8.7477
7,8,105400,5.6933
8,9,64630,3.4911
9,10,35998,1.9445


,stored_value,row_count,row_pct
0,G,1078420,58.2525
1,F,371961,20.0920
2,M,190797,10.3062
3,C,178499,9.6419
4,H,30728,1.6598
5,R,878,0.0474
6,B,1,0.0001
7,BB,1,0.0001


In [30]:
# Inspect rare age and sex values in their full runner and race context.
#
# These values are not classified as errors yet. The purpose is to determine
# whether they are plausible within particular jurisdictions or appear to be
# isolated source defects.

rare_age_sex_rows = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        date,
        course,
        race_id,
        off,
        race_name,
        type,
        age_band,
        sex_rest,
        dist,
        going,
        ran,
        num,
        pos,
        horse,
        age,
        sex,
        jockey,
        trainer
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND (
            age IN (1, 17, 18, 31)
         OR sex IN ('B', 'BB')
      )
    ORDER BY
        age,
        sex,
        date,
        course,
        off,
        pos,
        rowid
    """,
    connection,
)

print(f"Rows with rare age or sex values: {len(rare_age_sex_rows):,}")

with pd.option_context(
    "display.max_rows",
    None,
    "display.max_colwidth",
    120,
):
    display(rare_age_sex_rows)

Rows with rare age or sex values: 15


,source_rowid,date,course,race_id,off,race_name,type,age_band,sex_rest,dist,going,ran,num,pos,horse,age,sex,jockey,trainer
0,528628,2018-04-30,Windsor,698537,5:40,British EBF Novice Stakes (Plus 10 Race),Flat,2yo,,5f,Good,7,2,6,Anthem Of Peace (AUS),1,C,Oisin Murphy,Brian Meehan
1,534896,2018-05-10,Chester,699325,4:05,British Stallion Studs EBF Maiden Stakes (Plus 10 Race),Flat,2yo,,5f,Good,9,1,9,Anthem Of Peace (AUS),1,C,Oisin Murphy,Brian Meehan
2,547934,2018-06-01,Bath,701386,6:20,EBF Novice Stakes,Flat,2yo,,5f,Good,6,2,6,Anthem Of Peace (AUS),1,C,Charles Bishop,Brian Meehan
3,553245,2018-06-10,Goodwood,702154,3:10,Sutton Winson Backing The NSPCC Selling Stakes,Flat,2yo,,5f,Good To Firm,13,1,12,Anthem Of Peace (AUS),1,C,Tom Marquand,Brian Meehan
4,557431,2018-06-18,Windsor,703307,7:00,Sky Bet Best Odds Guaranteed Selling Stakes,Flat,2yo,,6f,Good To Firm,11,1,7,Anthem Of Peace (AUS),1,C,Tom Marquand,Brian Meehan
5,814999,2019-11-29,Gulfstream Park (USA),746195,8:30,Starter Optional Claiming (Claimer) (2yo Fillies) (Turf),Flat,2yo,F,1m,Firm,11,2,7,La Venezolana (VEN),2,B,Jairo Rendon,Rodolfo Garcia
6,448648,2017-10-15,Cologne (GER),687124,1:35,Kolner Steher Cup Der Pferdeklinik Burg () (3yo+) (Turf),Flat,3yo+,,1m7f,Soft,11,10,5,Par Coeur (GER),3,BB,Lukas Delozier,W Mongil
7,880,2015-01-02,Ffos Las,615744,1:10,Radio Carmarthenshire and Scarlet FM Handicap Chase,Chase,5yo+,,3m,Heavy,6,1,CO,Victory Gunner (IRE),17,G,Micheal Nolan,Richard Lee
8,15003,2015-02-16,Lingfield,618067,2:45,888sport.com Handicap Chase,Chase,5yo+,,3m,Heavy,10,1,PU,Victory Gunner (IRE),17,G,Micheal Nolan,Richard Lee
9,913242,2020-10-20,Tipperary (IRE),769812,4:55,Golden Handicap Chase (Div II),Chase,4yo+,,2m4f,Soft To Heavy,9,11,PU,See Double You (IRE),17,G,Sean Flanagan,Ronan M P McNally


In [31]:
# Compare the rare-value horses with their other records in the source.
#
# This checks whether age and sex values remain consistent across appearances
# or whether the unusual values are isolated within each horse's history.

rare_value_horses = [
    "Anthem Of Peace (AUS)",
    "La Venezolana (VEN)",
    "Par Coeur (GER)",
    "Victory Gunner (IRE)",
    "See Double You (IRE)",
    "Ecstasy (USA)",
]

placeholders = ", ".join("?" for _ in rare_value_horses)

rare_horse_histories = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        date,
        course,
        race_id,
        off,
        race_name,
        age_band,
        sex_rest,
        num,
        pos,
        horse,
        age,
        sex,
        jockey,
        trainer
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND horse IN ({placeholders})
    ORDER BY
        horse,
        date,
        course,
        off,
        rowid
    """,
    connection,
    params=rare_value_horses,
)

rare_horse_summary = (
    rare_horse_histories
    .groupby("horse", as_index=False)
    .agg(
        appearance_count=("source_rowid", "count"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        distinct_ages=("age", lambda values: sorted(set(values))),
        distinct_sex_codes=("sex", lambda values: sorted(set(values))),
    )
)

display(rare_horse_summary)

with pd.option_context(
    "display.max_rows",
    None,
    "display.max_colwidth",
    120,
):
    display(rare_horse_histories)

,horse,appearance_count,first_date,last_date,distinct_ages,distinct_sex_codes
0,Anthem Of Peace (AUS),5,2018-04-30,2018-06-18,[1],[C]
1,Ecstasy (USA),4,2024-07-27,2025-07-26,"[3, 4, 31]","[F, M]"
2,La Venezolana (VEN),1,2019-11-29,2019-11-29,[2],[B]
3,Par Coeur (GER),1,2017-10-15,2017-10-15,[3],[BB]
4,See Double You (IRE),42,2015-11-29,2021-05-04,"[12, 13, 14, 15, 16, 17, 18]",[G]
5,Victory Gunner (IRE),2,2015-01-02,2015-02-16,[17],[G]


,source_rowid,date,course,race_id,off,race_name,age_band,sex_rest,num,pos,horse,age,sex,jockey,trainer
0,528628,2018-04-30,Windsor,698537,5:40,British EBF Novice Stakes (Plus 10 Race),2yo,,2,6,Anthem Of Peace (AUS),1,C,Oisin Murphy,Brian Meehan
1,534896,2018-05-10,Chester,699325,4:05,British Stallion Studs EBF Maiden Stakes (Plus 10 Race),2yo,,1,9,Anthem Of Peace (AUS),1,C,Oisin Murphy,Brian Meehan
2,547934,2018-06-01,Bath,701386,6:20,EBF Novice Stakes,2yo,,2,6,Anthem Of Peace (AUS),1,C,Charles Bishop,Brian Meehan
3,553245,2018-06-10,Goodwood,702154,3:10,Sutton Winson Backing The NSPCC Selling Stakes,2yo,,1,12,Anthem Of Peace (AUS),1,C,Tom Marquand,Brian Meehan
4,557431,2018-06-18,Windsor,703307,7:00,Sky Bet Best Odds Guaranteed Selling Stakes,2yo,,1,7,Anthem Of Peace (AUS),1,C,Tom Marquand,Brian Meehan
5,1543314,2024-07-27,Woodbine (CAN),873561,9:47,Ontario Colleen Stakes (3yo Fillies) (Turf),3yo,F,8,11,Ecstasy (USA),31,M,Fraser Aebly,Sid C Attard
6,1596072,2024-11-09,Woodbine (CAN),881605,9:20,Maple Leaf Stakes (3yo+ Fillies & Mares) (All-Weather Track) (Tapeta),3yo+,F & M,3,7,Ecstasy (USA),3,F,Fraser Aebly,Sid C Attard
7,1702256,2025-07-05,Woodbine (CAN),899417,10:20,Hendrie Stakes (3yo+ Fillies & Mares) (All-Weather Track) (Tapeta),3yo+,F & M,4,7,Ecstasy (USA),4,F,Fraser Aebly,Sid C Attard
8,1711509,2025-07-26,Woodbine (CAN),900447,8:07,Trillium Stakes presented by Don Julio (3yo+ Fillies & Mares) (All-Weather Track) (Tapeta),3yo+,F & M,6,6,Ecstasy (USA),4,F,Fraser Aebly,Sid C Attard
9,814999,2019-11-29,Gulfstream Park (USA),746195,8:30,Starter Optional Claiming (Claimer) (2yo Fillies) (Turf),2yo,F,2,7,La Venezolana (VEN),2,B,Jairo Rendon,Rodolfo Garcia


In [32]:
# Inspect the rare blank jockey and trainer records.
#
# Because there are only 11 affected rows in total, complete contextual
# inspection is more useful than aggregate profiling.

blank_jockey_trainer_rows = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        date,
        course,
        race_id,
        off,
        race_name,
        type,
        age_band,
        dist,
        going,
        ran,
        num,
        pos,
        horse,
        age,
        sex,
        jockey,
        trainer,
        comment
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND (
            length(trim(jockey)) = 0
         OR length(trim(trainer)) = 0
      )
    ORDER BY
        date,
        course,
        off,
        pos,
        rowid
    """,
    connection,
)

blank_jockey_trainer_summary = pd.DataFrame(
    {
        "condition": [
            "blank jockey",
            "blank trainer",
            "both blank",
        ],
        "row_count": [
            int((blank_jockey_trainer_rows["jockey"].str.strip() == "").sum()),
            int((blank_jockey_trainer_rows["trainer"].str.strip() == "").sum()),
            int(
                (
                    (blank_jockey_trainer_rows["jockey"].str.strip() == "")
                    & (blank_jockey_trainer_rows["trainer"].str.strip() == "")
                ).sum()
            ),
        ],
    }
)

display(blank_jockey_trainer_summary)

with pd.option_context(
    "display.max_rows",
    None,
    "display.max_colwidth",
    120,
):
    display(blank_jockey_trainer_rows)

,condition,row_count
0,blank jockey,2
1,blank trainer,9
2,both blank,0


,source_rowid,date,course,race_id,off,race_name,type,age_band,dist,going,ran,num,pos,horse,age,sex,jockey,trainer,comment
0,50160,2015-05-06,Sonoda (JPN),627438,11:07,Hyogo Championship (Local (Dirt),Flat,3yo,1m1½f,Standard,12,,10,Maximum Kaiser (JPN),3,C,Shoichi Kawahara,,
1,189632,2016-04-08,Auteuil (FR),648524,2:15,Prix Guy Hunault (Hurdle) (Conditions) (5yo) (Turf),Hurdle,5yo,2m2f,Very Soft,12,12,PU,Ahzana (FR),5,M,,L Viel,
2,203870,2016-05-07,Maisons-Laffitte (FR),651027,3:30,Prix de lEtoile dArtois (Handicap) (4yo) (Round) (Turf),Flat,4yo,1m2f,Good,16,9,7,Star White (FR),4,F,Tony Piccone,,
3,203991,2016-05-07,Maisons-Laffitte (FR),651027,3:30,Prix de lEtoile dArtois (Handicap) (4yo) (Round) (Turf),Flat,4yo,1m2f,Good,16,5,12,Colombia DEmra (FR),4,F,Richard Juteau,,
4,599152,2018-09-10,Chantilly (FR),711601,3:30,Prix de Foulangues (Handicap) (4yo+) (Turf),Flat,4yo+,1m1f,Good,18,17,15,Numbers Talk (IRE),10,G,Ronan Thomas,,
5,600778,2018-09-14,Saint-Cloud (FR),712047,3:10,Prix de Lamarque (Handicap) (5yo+) (Turf),Flat,5yo+,1m2½f,Good,18,7,7,Valley Kid (FR),5,H,Michelle Swinnens,,
6,602610,2018-09-17,Maisons-Laffitte (FR),712142,3:55,Prix Du Moulin De Maisons-laffitte (Handicap) (3yo+) (Turf),Flat,3yo+,6f,Good To Soft,15,6,9,Unital (FR),7,G,Tony Piccone,,
7,602570,2018-09-17,Maisons-Laffitte (FR),712151,4:25,Prix de la Porte du Parc (Handicap) (3yo+) (Turf),Flat,3yo+,6f,Good To Soft,15,2,4,Talento (IRE),5,G,Mickael Barzalona,,
8,608243,2018-09-28,Saint-Cloud (FR),713069,3:40,Prix du Lieu Feral (Handicap) (3yo) (Turf),Flat,3yo,1m,Good,16,8,5,Totem (FR),3,G,Mlle Chloe Hue,,
9,608142,2018-09-28,Saint-Cloud (FR),713071,4:10,Prix du Lieu Jourdain (Handicap) (3yo) (Turf),Flat,3yo,1m,Good,16,16,4,Jasmine A La Plage (FR),3,F,Antoine Coutier,,


In [33]:
# Profile the runner-number field as stored.
#
# The field is declared INTEGER but contains blank text in some rows. This cell
# separates numeric values from blanks and checks whether populated numbers are
# unique within each apparent race.

num_profile = pd.read_sql_query(
    f"""
    SELECT
        MIN(
            CASE
                WHEN typeof(num) = 'integer'
                THEN num
            END
        ) AS minimum_numeric_value,

        MAX(
            CASE
                WHEN typeof(num) = 'integer'
                THEN num
            END
        ) AS maximum_numeric_value,

        COUNT(
            DISTINCT CASE
                WHEN typeof(num) = 'integer'
                THEN num
            END
        ) AS distinct_numeric_values,

        SUM(
            CASE
                WHEN typeof(num) = 'integer'
                 AND num < 0
                THEN 1 ELSE 0
            END
        ) AS negative_value_rows,

        SUM(
            CASE
                WHEN typeof(num) = 'integer'
                 AND num = 0
                THEN 1 ELSE 0
            END
        ) AS zero_value_rows,

        SUM(
            CASE
                WHEN typeof(num) = 'text'
                 AND length(trim(num)) = 0
                THEN 1 ELSE 0
            END
        ) AS blank_value_rows
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    """,
    connection,
)

num_distribution = pd.read_sql_query(
    f"""
    SELECT
        CASE
            WHEN typeof(num) = 'text'
             AND length(trim(num)) = 0
            THEN '<blank>'
            ELSE CAST(num AS TEXT)
        END AS stored_value,
        typeof(num) AS sqlite_storage_class,
        COUNT(*) AS row_count
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY
        stored_value,
        sqlite_storage_class
    ORDER BY
        CASE
            WHEN stored_value = '<blank>' THEN 1
            ELSE 0
        END,
        CAST(stored_value AS INTEGER)
    """,
    connection,
)

num_uniqueness = pd.read_sql_query(
    f"""
    WITH apparent_races AS (
        SELECT
            date,
            course,
            off,
            race_name,
            COUNT(*) AS runner_rows,
            COUNT(
                CASE
                    WHEN typeof(num) = 'integer'
                    THEN 1
                END
            ) AS populated_num_rows,
            COUNT(
                DISTINCT CASE
                    WHEN typeof(num) = 'integer'
                    THEN num
                END
            ) AS distinct_populated_nums
        FROM data
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY
            date,
            course,
            off,
            race_name
    )
    SELECT
        COUNT(*) AS apparent_race_count,

        SUM(
            CASE
                WHEN populated_num_rows > distinct_populated_nums
                THEN 1 ELSE 0
            END
        ) AS races_with_duplicate_populated_nums,

        SUM(
            CASE
                WHEN populated_num_rows < runner_rows
                THEN 1 ELSE 0
            END
        ) AS races_with_blank_nums,

        MAX(populated_num_rows - distinct_populated_nums)
            AS maximum_duplicate_excess
    FROM apparent_races
    """,
    connection,
)

display(num_profile)

with pd.option_context("display.max_rows", None):
    display(num_distribution)

display(num_uniqueness)

,minimum_numeric_value,maximum_numeric_value,distinct_numeric_values,negative_value_rows,zero_value_rows,blank_value_rows
0,0,40,41,0,1179,7032


,stored_value,sqlite_storage_class,row_count
0,0,integer,1179
1,1,integer,175522
2,2,integer,176109
3,3,integer,175925
4,4,integer,175151
5,5,integer,171354
6,6,integer,163013
7,7,integer,150581
8,8,integer,134371
9,9,integer,116310


,apparent_race_count,races_with_duplicate_populated_nums,races_with_blank_nums,maximum_duplicate_excess
0,189043,539,877,16


In [34]:
# Summarise the main num quality patterns by course and race context.
#
# The aim is to see whether zeroes, blanks, and duplicates are concentrated
# in particular jurisdictions or race types before drawing conclusions.

num_issue_rows = pd.read_sql_query(
    f"""
    WITH duplicate_num_races AS (
        SELECT
            date,
            course,
            off,
            race_name
        FROM data
        WHERE {DATA_ROW_PREDICATE}
          AND typeof(num) = 'integer'
        GROUP BY
            date,
            course,
            off,
            race_name,
            num
        HAVING COUNT(*) > 1
    )
    SELECT
        d.rowid AS source_rowid,
        d.date,
        d.course,
        d.race_id,
        d.off,
        d.race_name,
        d.type,
        d.ran,
        d.num,
        typeof(d.num) AS num_storage_class,
        d.pos,
        d.horse,
        CASE
            WHEN typeof(d.num) = 'text'
             AND length(trim(d.num)) = 0
            THEN 'blank'
            WHEN typeof(d.num) = 'integer'
             AND d.num = 0
            THEN 'zero'
            WHEN EXISTS (
                SELECT 1
                FROM duplicate_num_races AS r
                WHERE r.date = d.date
                  AND r.course = d.course
                  AND r.off = d.off
                  AND r.race_name = d.race_name
            )
            THEN 'race contains duplicate populated num'
            ELSE 'other'
        END AS num_issue_type
    FROM data AS d
    WHERE d.rowid <> 1
      AND (
            (
                typeof(d.num) = 'text'
                AND length(trim(d.num)) = 0
            )
         OR (
                typeof(d.num) = 'integer'
                AND d.num = 0
            )
         OR EXISTS (
                SELECT 1
                FROM duplicate_num_races AS r
                WHERE r.date = d.date
                  AND r.course = d.course
                  AND r.off = d.off
                  AND r.race_name = d.race_name
            )
      )
    """,
    connection,
)

num_issue_summary = (
    num_issue_rows
    .groupby("num_issue_type", as_index=False)
    .agg(
        runner_rows=("source_rowid", "count"),
        apparent_races=(
            "race_id",
            lambda values: values.nunique(),
        ),
        distinct_courses=("course", "nunique"),
    )
    .sort_values(
        "runner_rows",
        ascending=False,
        ignore_index=True,
    )
)

num_issue_top_courses = (
    num_issue_rows
    .groupby(
        ["num_issue_type", "course"],
        as_index=False,
    )
    .agg(
        runner_rows=("source_rowid", "count"),
        apparent_races=("race_id", "nunique"),
    )
    .sort_values(
        ["num_issue_type", "runner_rows", "course"],
        ascending=[True, False, True],
        ignore_index=True,
    )
)

display(num_issue_summary)

for issue_type in num_issue_top_courses["num_issue_type"].unique():
    print(f"\n{issue_type.upper()} — TOP COURSES")
    display(
        num_issue_top_courses.loc[
            num_issue_top_courses["num_issue_type"] == issue_type
        ].head(20)
    )

,num_issue_type,runner_rows,apparent_races,distinct_courses
0,blank,7032,877,142
1,race contains duplicate populated num,4316,362,29
2,zero,1179,186,23



BLANK — TOP COURSES


,num_issue_type,course,runner_rows,apparent_races
0,blank,Les Landes (JER),1196,193
1,blank,Ohi (JPN),416,33
2,blank,Kawasaki (JPN),229,20
3,blank,Gulfstream Park (USA),219,26
4,blank,Funabashi (JPN),193,20
5,blank,Santa Anita (USA),178,25
6,blank,San Siro (ITY),174,32
7,blank,Morioka (JPN),165,15
8,blank,Nakayama (JPN),154,11
9,blank,Mombetsu (JPN),146,12



RACE CONTAINS DUPLICATE POPULATED NUM — TOP COURSES


,num_issue_type,course,runner_rows,apparent_races
142,race contains duplicate populated num,San Isidro (ARG),889,66
143,race contains duplicate populated num,Cidade Jardim (BRZ),668,60
144,race contains duplicate populated num,Gavea (BRZ),588,46
145,race contains duplicate populated num,Palermo (ARG),547,48
146,race contains duplicate populated num,Monterrico (PER),497,40
147,race contains duplicate populated num,Maronas (URU),368,28
148,race contains duplicate populated num,La Plata (ARG),172,12
149,race contains duplicate populated num,Aqueduct (USA),82,11
150,race contains duplicate populated num,Belmont Park (USA),74,8
151,race contains duplicate populated num,Churchill Downs (USA),52,5



ZERO — TOP COURSES


,num_issue_type,course,runner_rows,apparent_races
171,zero,Les Landes (JER),911,139
172,zero,L'Ancresse (GUE),57,15
173,zero,Caulfield (AUS),41,4
174,zero,Chukyo,17,1
175,zero,Mombetsu (JPN),17,2
176,zero,Kyoto (JPN),16,1
177,zero,Lyon-La Soie (FR),16,1
178,zero,Nakayama,16,1
179,zero,Santa Anita (USA),16,3
180,zero,Saratoga (USA),16,3


In [35]:
# Profile exact duplicate populated num groups within apparent races.
#
# This identifies how many runners share each duplicated number and provides
# examples. It does not assume whether these represent coupled entries,
# local numbering conventions, or source defects.

duplicate_num_groups = pd.read_sql_query(
    f"""
    SELECT
        date,
        course,
        off,
        race_name,
        num,
        COUNT(*) AS runners_sharing_num,
        GROUP_CONCAT(horse, ' | ') AS horses
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND typeof(num) = 'integer'
    GROUP BY
        date,
        course,
        off,
        race_name,
        num
    HAVING COUNT(*) > 1
    ORDER BY
        runners_sharing_num DESC,
        date,
        course,
        off,
        num
    """,
    connection,
)

duplicate_num_multiplicity = (
    duplicate_num_groups
    .groupby("runners_sharing_num", as_index=False)
    .agg(
        duplicate_number_groups=("num", "count"),
        distinct_apparent_races=(
            "race_name",
            lambda _: 0,
        ),
    )
)

# Recalculate race counts using the full provisional race identity rather
# than race_id, which is not globally reliable.
duplicate_num_multiplicity["distinct_apparent_races"] = [
    duplicate_num_groups.loc[
        duplicate_num_groups["runners_sharing_num"] == multiplicity,
        ["date", "course", "off", "race_name"],
    ].drop_duplicates().shape[0]
    for multiplicity in duplicate_num_multiplicity[
        "runners_sharing_num"
    ]
]

duplicate_num_multiplicity = duplicate_num_multiplicity.sort_values(
    "runners_sharing_num",
    ignore_index=True,
)

print(
    "Exact duplicated num groups: "
    f"{len(duplicate_num_groups):,}"
)
print(
    "Distinct apparent races with duplicated populated num: "
    f"{duplicate_num_groups[['date', 'course', 'off', 'race_name']].drop_duplicates().shape[0]:,}"
)

display(duplicate_num_multiplicity)

with pd.option_context(
    "display.max_rows",
    50,
    "display.max_colwidth",
    140,
):
    display(duplicate_num_groups.head(50))

Exact duplicated num groups: 700
Distinct apparent races with duplicated populated num: 539


,runners_sharing_num,duplicate_number_groups,distinct_apparent_races
0,2,488,339
1,3,48,47
2,4,26,26
3,5,29,29
4,6,28,28
5,7,24,24
6,8,23,23
7,9,13,13
8,10,10,10
9,11,5,5


,date,course,off,race_name,num,runners_sharing_num,horses
0,2026-03-21,Chukyo,06:20,Chunichi Sports Sho Falcon Stakes (Turf),0,17,Diamond Knot (JPN) | A Shin Deed (JPN) | Fukuchan Sho (JPN) | Tagano Aralia (JPN) | Vogel (JPN) | Tamamo Icarus (JPN) | Triumph Pass (JP...
1,2016-12-07,Lyon-La Soie (FR),5:10,Prix Du Haras Des Chataigniers (Claimer) (All-Weather),0,16,Absolute Summer (FR) | Miss Charlotte (IRE) | Briseide (IRE) | Txalupa (FR) | National Velvet (FR) | Zahiria (FR) | Highgate (FR) | Marr...
2,2018-11-04,Kyoto (JPN),6:01,JBC Sprint (Local (Dirt),0,16,Graceful Leap (JPN) | Matera Sky (USA) | Kitasan Mikazuki (JPN) | Moanin (USA) | Lets Go Donki (JPN) | Kings Guard (JPN) | T O Helios (U...
3,2026-03-21,Nakayama,06:45,Flower Cup (Fillies) (Turf),0,16,Smart Priere (JPN) | Longing Celine (JPN) | Exceed (JPN) | Ametista (JPN) | Cara Persona (JPN) | Early Harvest (JPN) | Godiamo (JPN) | N...
4,2018-08-16,Mombetsu (JPN),11:07,Breeders Gold Cup (Local (Dirt),0,15,Up To You (JPN) | So This Is Love (JPN) | Cross Wind (JPN) | Noble Sapphire (JPN) | Junaino Kimi (JPN) | Time Beyond (JPN) | Rinehart (J...
5,2015-02-28,Caulfield (AUS),5:35,Crown Golden Ale Peter Young Stakes (Turf),0,13,Mourinho (AUS) | Happy Trails (AUS) | Akzar (IRE) | Spillway (GB) | Au Revoir (IRE) | Protectionist (GER) | Jacquinot Bay (AUS) | Real L...
6,2015-02-28,Caulfield (AUS),3:40,Polytrack Angus Armanasco Stakes (Fillies) (Turf),0,11,Sabatini (AUS) | Fontein Ruby (AUS) | Samartested (AUS) | Bottle Of Smoke (AUS) | Thinking Of You (NZ) | Marple Miss (AUS) | Mossbeat (A...
7,2016-05-15,Les Landes (JER),3:05,Bloodstock Advisory Services Handicap,0,11,Pas DAction (GB) | Valmina (GB) | Tax Reform (IRE) | Spanish Bounty (GB) | Engaging Smile (GB) | Purley Queen (IRE) | Chapeau Bleu (IRE)...
8,2016-10-01,Santa Anita (USA),1:06,Eddie D Stakes (Div II) (3yo+) (Turf),0,11,Holy Lute (USA) | Boozer (USA) | Guns Loaded (USA) | Toowindytohaulrox (USA) | Ohio (BRZ) | Bottle Rocket (USA) | So Sweetitiz (USA) | E...
9,2018-04-22,Les Landes (JER),4:50,The Presidents Handicap,0,11,George Baker (IRE) | Major Tom (GB) | Pas DAction (GB) | Princess Kodia (IRE) | Grey Panel (FR) | Mendacious Harpy (IRE) | House Of Frau...


In [36]:
# Separate duplicate positive runner numbers from races where num = 0
# is repeated across multiple runners.
#
# This prevents the zero-code convention from inflating the duplicate-number
# problem and lets us inspect only repeated positive values.

positive_duplicate_num_groups = pd.read_sql_query(
    f"""
    SELECT
        date,
        course,
        off,
        race_name,
        num,
        COUNT(*) AS runners_sharing_num,
        GROUP_CONCAT(horse, ' | ') AS horses
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND typeof(num) = 'integer'
      AND num > 0
    GROUP BY
        date,
        course,
        off,
        race_name,
        num
    HAVING COUNT(*) > 1
    ORDER BY
        runners_sharing_num DESC,
        date,
        course,
        off,
        num
    """,
    connection,
)

positive_duplicate_summary = pd.DataFrame(
    {
        "measure": [
            "duplicate positive num groups",
            "distinct apparent races affected",
            "distinct courses affected",
            "maximum runners sharing one positive num",
        ],
        "value": [
            len(positive_duplicate_num_groups),
            positive_duplicate_num_groups[
                ["date", "course", "off", "race_name"]
            ].drop_duplicates().shape[0],
            positive_duplicate_num_groups["course"].nunique(),
            (
                int(
                    positive_duplicate_num_groups[
                        "runners_sharing_num"
                    ].max()
                )
                if not positive_duplicate_num_groups.empty
                else 0
            ),
        ],
    }
)

positive_duplicate_top_courses = (
    positive_duplicate_num_groups
    .groupby("course", as_index=False)
    .agg(
        duplicate_groups=("num", "count"),
        apparent_races=(
            "race_name",
            lambda _: 0,
        ),
    )
)

positive_duplicate_top_courses["apparent_races"] = [
    positive_duplicate_num_groups.loc[
        positive_duplicate_num_groups["course"] == course,
        ["date", "course", "off", "race_name"],
    ].drop_duplicates().shape[0]
    for course in positive_duplicate_top_courses["course"]
]

positive_duplicate_top_courses = (
    positive_duplicate_top_courses
    .sort_values(
        ["duplicate_groups", "course"],
        ascending=[False, True],
        ignore_index=True,
    )
)

display(positive_duplicate_summary)
display(positive_duplicate_top_courses.head(30))

with pd.option_context(
    "display.max_rows",
    50,
    "display.max_colwidth",
    140,
):
    display(positive_duplicate_num_groups.head(50))

,measure,value
0,duplicate positive num groups,523
1,distinct apparent races affected,362
2,distinct courses affected,29
3,maximum runners sharing one positive num,4


,course,duplicate_groups,apparent_races
0,San Isidro (ARG),109,66
1,Cidade Jardim (BRZ),90,60
2,Gavea (BRZ),77,46
3,Monterrico (PER),71,40
4,Palermo (ARG),54,48
5,Maronas (URU),34,28
6,La Plata (ARG),21,12
7,Aqueduct (USA),12,11
8,Belmont Park (USA),8,8
9,Club Hipico de Santiago (CHI),6,5


,date,course,off,race_name,num,runners_sharing_num,horses
0,2018-12-08,San Isidro (ARG),9:55,Copa de Plata (3yo+ Fillies & Mares) (Turf),11,4,London Show (URU) | Litle Eatly (BRZ) | Mendel Sweet (BRZ) | Linda Le (BRZ)
1,2015-01-11,Monterrico (PER),10:25,Premio Enrique Meiggs (3yo+) (Dirt),10,3,Good Luck Keny (ARG) | Sweet Cazanova (ARG) | Mr. Rodrigo (CHI)
2,2015-03-07,Aqueduct (USA),9:50,Gotham Stakes (3yo) (Dirt),1,3,Dontbetwithbruno (USA) | Uninfluenced (USA) | Blame Jim (USA)
3,2015-07-05,Club Hipico de Santiago (CHI),8:35,Premio Arturo Lyon P (3yo Fillies) (Turf),2,3,Superdotada (CHI) | Kitcat (CHI) | Cimalta (CHI)
4,2015-08-02,Club Hipico de Santiago (CHI),8:35,Premio Polla de Potrancas (3yo Fillies) (Turf),3,3,Cimalta (CHI) | Wapi (CHI) | Kitcat (CHI)
5,2015-09-06,Monterrico (PER),10:50,Premio Polla de Potrillos - Roberto Alvarez Calderon Rey (3yo Colts & Geldings) (Dirt),3,3,Mr Inspiration (ARG) | Abueyo (ARG) | Street Lolo (ARG)
6,2015-10-04,Monterrico (PER),10:55,Ricardo Ortiz de Zevallos (3yo) (Dirt),2,3,Street Lolo (ARG) | Indio Dakota (ARG) | Abueyo (ARG)
7,2015-11-08,Monterrico (PER),10:35,Derby Nacional (3yo) (Dirt),9,3,Mr. Omar (CHI) | La Candy (PER) | Rubirosa (PER)
8,2015-11-08,Monterrico (PER),10:35,Derby Nacional (3yo) (Dirt),10,3,Street Lolo (ARG) | Blue Demon (ARG) | Mr. Leguia (PER)
9,2015-11-14,San Isidro (ARG),8:25,Premio Enrique Acebal (3yo Fillies) (Turf),5,3,Quatro Folhas (ARG) | Valencia (ARG) | Feet First (ARG)


In [37]:
# Profile runner and participant names exactly as stored.
#
# This cell does not standardise punctuation, split names permanently,
# or attempt entity resolution.

name_field_profile = pd.read_sql_query(
    f"""
    SELECT
        MIN(length(horse)) AS horse_min_length,
        MAX(length(horse)) AS horse_max_length,
        ROUND(AVG(length(horse)), 2) AS horse_avg_length,
        SUM(CASE WHEN horse <> trim(horse) THEN 1 ELSE 0 END)
            AS horse_outer_whitespace_rows,

        MIN(length(jockey)) AS jockey_min_length,
        MAX(length(jockey)) AS jockey_max_length,
        ROUND(AVG(length(jockey)), 2) AS jockey_avg_length,
        SUM(CASE WHEN jockey <> trim(jockey) THEN 1 ELSE 0 END)
            AS jockey_outer_whitespace_rows,

        MIN(length(trainer)) AS trainer_min_length,
        MAX(length(trainer)) AS trainer_max_length,
        ROUND(AVG(length(trainer)), 2) AS trainer_avg_length,
        SUM(CASE WHEN trainer <> trim(trainer) THEN 1 ELSE 0 END)
            AS trainer_outer_whitespace_rows
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    """,
    connection,
)

# Load distinct horse names only, then inspect terminal parenthesised suffixes
# in Python because SQLite has no built-in reverse-string function.
distinct_horse_names = pd.read_sql_query(
    f"""
    SELECT DISTINCT horse
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    ORDER BY horse
    """,
    connection,
)["horse"]

horse_suffix_counts = Counter()

for horse_name in distinct_horse_names:
    match = re.search(r"\(([^()]*)\)$", horse_name)

    if match:
        horse_suffix_counts[match.group(1)] += 1
    else:
        horse_suffix_counts["<no terminal parenthesis>"] += 1

horse_suffix_profile = pd.DataFrame(
    [
        {
            "apparent_suffix": suffix,
            "distinct_horse_names": count,
        }
        for suffix, count in horse_suffix_counts.items()
    ]
).sort_values(
    ["distinct_horse_names", "apparent_suffix"],
    ascending=[False, True],
    ignore_index=True,
)

display(name_field_profile)
display(horse_suffix_profile.head(50))

,horse_min_length,horse_max_length,horse_avg_length,horse_outer_whitespace_rows,jockey_min_length,jockey_max_length,jockey_avg_length,jockey_outer_whitespace_rows,trainer_min_length,trainer_max_length,trainer_avg_length,trainer_outer_whitespace_rows
0,7,26,16.85,0,0,33,12.95,0,0,39,12.86,0


,apparent_suffix,distinct_horse_names
0,IRE,70682
1,GB,41810
2,FR,36245
3,USA,21854
4,AUS,14132
5,JPN,5993
6,NZ,3802
7,GER,2967
8,SAF,2575
9,ARG,2480


In [38]:
# Inspect horse-name suffix coverage and uncommon terminal fragments.
#
# This remains a text-structure investigation. It does not assign countries
# or create a permanent horse identifier.

horse_suffix_summary = pd.DataFrame(
    {
        "measure": [
            "distinct horse names",
            "with terminal parenthesised suffix",
            "without terminal parenthesised suffix",
            "distinct terminal suffixes",
        ],
        "value": [
            len(distinct_horse_names),
            int(
                horse_suffix_profile.loc[
                    horse_suffix_profile["apparent_suffix"]
                    != "<no terminal parenthesis>",
                    "distinct_horse_names",
                ].sum()
            ),
            int(
                horse_suffix_profile.loc[
                    horse_suffix_profile["apparent_suffix"]
                    == "<no terminal parenthesis>",
                    "distinct_horse_names",
                ].sum()
            ),
            int(
                (
                    horse_suffix_profile["apparent_suffix"]
                    != "<no terminal parenthesis>"
                ).sum()
            ),
        ],
    }
)

horse_names_without_suffix = pd.DataFrame(
    {
        "horse": [
            horse_name
            for horse_name in distinct_horse_names
            if re.search(r"\(([^()]*)\)$", horse_name) is None
        ]
    }
)

rare_horse_suffixes = horse_suffix_profile.loc[
    horse_suffix_profile["distinct_horse_names"] <= 10
].copy()

display(horse_suffix_summary)

print(
    "Examples without terminal parenthesised suffix: "
    f"{min(50, len(horse_names_without_suffix)):,}"
)
display(horse_names_without_suffix.head(50))

with pd.option_context("display.max_rows", None):
    display(rare_horse_suffixes)

,measure,value
0,distinct horse names,208631
1,with terminal parenthesised suffix,208631
2,without terminal parenthesised suffix,0
3,distinct terminal suffixes,51


Examples without terminal parenthesised suffix: 0


,horse


,apparent_suffix,distinct_horse_names
33,UAE,8
34,HOL,7
35,SLO,7
36,MEX,6
37,RUS,5
38,UKR,4
39,VEN,4
40,IND,3
41,POR,3
42,AZE,2


In [39]:
# Examine horse names after temporarily separating the terminal suffix.
#
# This is an exploratory text profile only. It does not create permanent
# horse identifiers or assume that equal base names refer to the same horse.

horse_name_parts = pd.DataFrame(
    {
        "stored_horse_name": distinct_horse_names,
    }
)

horse_name_parts[["base_name", "apparent_suffix"]] = (
    horse_name_parts["stored_horse_name"]
    .str.extract(r"^(.*) \(([^()]*)\)$")
)

horse_base_name_summary = pd.DataFrame(
    {
        "measure": [
            "distinct stored horse names",
            "distinct base names",
            "base names used with multiple suffixes",
            "maximum suffixes for one base name",
        ],
        "value": [
            len(horse_name_parts),
            horse_name_parts["base_name"].nunique(),
            int(
                (
                    horse_name_parts
                    .groupby("base_name")["apparent_suffix"]
                    .nunique()
                    > 1
                ).sum()
            ),
            int(
                horse_name_parts
                .groupby("base_name")["apparent_suffix"]
                .nunique()
                .max()
            ),
        ],
    }
)

horse_base_suffix_variants = (
    horse_name_parts
    .groupby("base_name", as_index=False)
    .agg(
        distinct_suffixes=("apparent_suffix", "nunique"),
        suffixes=(
            "apparent_suffix",
            lambda values: " | ".join(sorted(set(values))),
        ),
        stored_names=(
            "stored_horse_name",
            lambda values: " | ".join(sorted(set(values))),
        ),
    )
)

horse_base_suffix_variants = (
    horse_base_suffix_variants.loc[
        horse_base_suffix_variants["distinct_suffixes"] > 1
    ]
    .sort_values(
        ["distinct_suffixes", "base_name"],
        ascending=[False, True],
        ignore_index=True,
    )
)

display(horse_base_name_summary)

print(
    "Examples of base names occurring with multiple suffixes: "
    f"{min(50, len(horse_base_suffix_variants)):,}"
)

with pd.option_context(
    "display.max_rows",
    50,
    "display.max_colwidth",
    160,
):
    display(horse_base_suffix_variants.head(50))

,measure,value
0,distinct stored horse names,208631
1,distinct base names,200156
2,base names used with multiple suffixes,7635
3,maximum suffixes for one base name,5


Examples of base names occurring with multiple suffixes: 50


,base_name,distinct_suffixes,suffixes,stored_names
0,Draco,5,AUS | FR | GB | URU | USA,Draco (AUS) | Draco (FR) | Draco (GB) | Draco (URU) | Draco (USA)
1,Earthquake,5,AUS | BRZ | FR | GB | USA,Earthquake (AUS) | Earthquake (BRZ) | Earthquake (FR) | Earthquake (GB) | Earthquake (USA)
2,Gold Standard,5,AUS | FR | IRE | SAF | USA,Gold Standard (AUS) | Gold Standard (FR) | Gold Standard (IRE) | Gold Standard (SAF) | Gold Standard (USA)
3,Hall Of Fame,5,AUS | FR | IRE | SWE | USA,Hall Of Fame (AUS) | Hall Of Fame (FR) | Hall Of Fame (IRE) | Hall Of Fame (SWE) | Hall Of Fame (USA)
4,Liberty Island,5,AUS | GER | IRE | JPN | USA,Liberty Island (AUS) | Liberty Island (GER) | Liberty Island (IRE) | Liberty Island (JPN) | Liberty Island (USA)
5,Maximus,5,AUS | FR | GER | PER | USA,Maximus (AUS) | Maximus (FR) | Maximus (GER) | Maximus (PER) | Maximus (USA)
6,Moana,5,FR | GB | JPN | NZ | USA,Moana (FR) | Moana (GB) | Moana (JPN) | Moana (NZ) | Moana (USA)
7,Overlord,5,ARG | AUS | FR | GB | TUR,Overlord (ARG) | Overlord (AUS) | Overlord (FR) | Overlord (GB) | Overlord (TUR)
8,Raphael,5,AUS | FR | GER | NZ | USA,Raphael (AUS) | Raphael (FR) | Raphael (GER) | Raphael (NZ) | Raphael (USA)
9,Samoa,5,FR | GER | PER | SAF | USA,Samoa (FR) | Samoa (GER) | Samoa (PER) | Samoa (SAF) | Samoa (USA)


### Runner identity and demographics — interim findings

#### `num`

The field is declared as `INTEGER` but uses mixed SQLite storage:

- 1,844,253 rows contain integers;
- 7,032 rows contain blank text;
- numeric values range from 0 to 40.

The value `0` occurs 1,179 times and is concentrated in particular jurisdictions and meetings. It appears to function as a source convention in at least some races rather than as an ordinary runner number.

Populated positive values are not universally unique within a race:

- 523 duplicated positive-number groups;
- 362 apparent races affected;
- 29 courses affected;
- up to four runners share one positive number.

These duplicates are heavily concentrated in South American and North American racing and may represent jurisdiction-specific coupled-entry or betting-number conventions.

`num` must therefore not be used as a universal runner identifier without further domain rules.

#### `horse`

The field is fully populated and has no outer whitespace.

Every one of the 208,631 distinct stored horse names ends with a terminal parenthesised suffix. There are 51 distinct suffix forms.

Removing the suffix would be unsafe:

- 200,156 distinct base names remain;
- 7,635 base names occur with multiple suffixes;
- some base names occur with as many as five suffixes.

The complete stored name should therefore remain intact until the suffix has been separately verified and parsed.

#### `age`

The field is fully populated and stored consistently as integers.

Most values range from 2 to 16. Rare older ages of 17 and 18 were supported by consistent histories for long-running National Hunt horses.

Two separate quality patterns were identified:

- one Australian-bred horse is stored as age 1 in five British 2-year-old races, suggesting a systematic international age-convention issue;
- one Woodbine record stores `Ecstasy (USA)` as age 31, while later records show ages 3 and 4. This is a clear isolated source defect.

#### `sex`

The field is fully populated and contains eight stored codes.

The dominant codes are:

- `G`;
- `F`;
- `M`;
- `C`;
- `H`;
- `R`.

The values `B` and `BB` each occur once and remain unresolved special codes.

One record for `Ecstasy (USA)` stores `M`, while later records consistently store `F`. Combined with the erroneous age 31 on the same row, this is a clear isolated source defect.

#### `jockey`

There are two blank values, both in French races.

No outer whitespace was found.

#### `trainer`

There are nine blank values:

- eight in French races;
- one in a Japanese race.

No outer whitespace was found.

#### Fields requiring separate domain investigation

- jurisdiction-specific meaning of `num = 0`;
- coupled-entry or shared-number conventions;
- horse suffix semantics and historical jurisdiction codes;
- international horse-age conventions;
- the meanings of `B` and `BB` in `sex`;
- entity resolution for horse, jockey, and trainer names.

In [40]:
# Define the result and finishing field group.
result_finishing_fields = [
    "pos",
    "ovr_btn",
    "btn",
]

# Build the same field-level source-quality summary used for earlier groups.
result_finishing_profile_rows = []

for field in result_finishing_fields:
    quoted_field = f'"{field}"'

    query = f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN {quoted_field} IS NULL THEN 1 ELSE 0 END)
            AS null_count,
        SUM(
            CASE
                WHEN typeof({quoted_field}) = 'text'
                 AND length(trim({quoted_field})) = 0
                THEN 1 ELSE 0
            END
        ) AS blank_text_count,
        COUNT(DISTINCT {quoted_field}) AS distinct_non_null_values,
        SUM(CASE WHEN typeof({quoted_field}) = 'null' THEN 1 ELSE 0 END)
            AS storage_null,
        SUM(CASE WHEN typeof({quoted_field}) = 'integer' THEN 1 ELSE 0 END)
            AS storage_integer,
        SUM(CASE WHEN typeof({quoted_field}) = 'real' THEN 1 ELSE 0 END)
            AS storage_real,
        SUM(CASE WHEN typeof({quoted_field}) = 'text' THEN 1 ELSE 0 END)
            AS storage_text,
        SUM(CASE WHEN typeof({quoted_field}) = 'blob' THEN 1 ELSE 0 END)
            AS storage_blob
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    """

    result = pd.read_sql_query(query, connection).iloc[0].to_dict()

    declared_type = source_data_dictionary.loc[
        source_data_dictionary["field_name"] == field,
        "declared_sqlite_type",
    ].iloc[0]

    result["field_name"] = field
    result["declared_sqlite_type"] = declared_type

    result_finishing_profile_rows.append(result)

result_finishing_quality = pd.DataFrame(
    result_finishing_profile_rows
)

result_finishing_quality = result_finishing_quality[
    [
        "field_name",
        "declared_sqlite_type",
        "total_rows",
        "null_count",
        "blank_text_count",
        "distinct_non_null_values",
        "storage_null",
        "storage_integer",
        "storage_real",
        "storage_text",
        "storage_blob",
    ]
]

display(result_finishing_quality)

,field_name,declared_sqlite_type,total_rows,null_count,blank_text_count,distinct_non_null_values,storage_null,storage_integer,storage_real,storage_text,storage_blob
0,pos,INTEGER,1851285,0,0,46,0,1756674,0,94611,0
1,ovr_btn,NUMERIC,1851285,0,0,951,0,589977,1167316,93992,0
2,btn,NUMERIC,1851285,0,0,245,0,661250,1096043,93992,0


In [41]:
# Profile every text-valued result code exactly as stored.
#
# These values are not interpreted yet. The purpose is to separate numeric
# finishing information from special text codes and quantify each code.

text_result_code_frames = []

for field in result_finishing_fields:
    quoted_field = f'"{field}"'

    text_codes = pd.read_sql_query(
        f"""
        SELECT
            {quoted_field} AS stored_value,
            COUNT(*) AS row_count
        FROM data
        WHERE {DATA_ROW_PREDICATE}
          AND typeof({quoted_field}) = 'text'
        GROUP BY {quoted_field}
        ORDER BY row_count DESC, stored_value
        """,
        connection,
    )

    text_codes["field_name"] = field
    text_codes["row_pct"] = (
        text_codes["row_count"]
        / 1_851_285
        * 100
    )

    text_result_code_frames.append(text_codes)

text_result_codes = pd.concat(
    text_result_code_frames,
    ignore_index=True,
)

for field in result_finishing_fields:
    print(f"\n{field.upper()} — TEXT VALUES")

    with pd.option_context("display.max_rows", None):
        display(
            text_result_codes.loc[
                text_result_codes["field_name"] == field,
                ["stored_value", "row_count", "row_pct"],
            ].round({"row_pct": 4})
        )


POS — TEXT VALUES


,stored_value,row_count,row_pct
0,PU,65832,3.5560
1,F,15681,0.8470
2,UR,9527,0.5146
3,BD,1020,0.0551
4,RR,723,0.0391
5,DSQ,619,0.0334
6,RO,463,0.0250
7,SU,371,0.0200
8,REF,291,0.0157
9,CO,77,0.0042



OVR_BTN — TEXT VALUES


,stored_value,row_count,row_pct
11,-,93992,5.0771



BTN — TEXT VALUES


,stored_value,row_count,row_pct
12,-,93992,5.0771


In [42]:
# Compare ovr_btn and btn storage patterns row by row.
#
# This tests whether the two fields jointly switch between numeric and
# special-text states, without yet interpreting their racing meaning.

button_storage_relationship = pd.read_sql_query(
    f"""
    SELECT
        typeof(ovr_btn) AS ovr_btn_storage,
        typeof(btn) AS btn_storage,
        COUNT(*) AS row_count
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY
        typeof(ovr_btn),
        typeof(btn)
    ORDER BY row_count DESC
    """,
    connection,
)

button_dash_relationship = pd.read_sql_query(
    f"""
    SELECT
        SUM(
            CASE
                WHEN ovr_btn = '-' AND btn = '-'
                THEN 1 ELSE 0
            END
        ) AS both_dash_rows,

        SUM(
            CASE
                WHEN ovr_btn = '-' AND btn <> '-'
                THEN 1 ELSE 0
            END
        ) AS ovr_only_dash_rows,

        SUM(
            CASE
                WHEN ovr_btn <> '-' AND btn = '-'
                THEN 1 ELSE 0
            END
        ) AS btn_only_dash_rows
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    """,
    connection,
)

display(button_storage_relationship)
display(button_dash_relationship)

,ovr_btn_storage,btn_storage,row_count
0,real,real,860050
1,integer,integer,353984
2,real,integer,307266
3,integer,real,235993
4,text,text,93992


,both_dash_rows,ovr_only_dash_rows,btn_only_dash_rows
0,93992,0,0


In [43]:
# Test how finishing-status codes relate to the dash placeholders in
# ovr_btn and btn.
#
# This records the observed relationship only. It does not yet assign formal
# meanings to the status codes or distance fields.

position_button_relationship = pd.read_sql_query(
    f"""
    SELECT
        CASE
            WHEN typeof(pos) = 'text'
            THEN pos
            ELSE '<numeric position>'
        END AS position_state,

        CASE
            WHEN ovr_btn = '-' AND btn = '-'
            THEN 'both dash'
            ELSE 'both numeric'
        END AS button_state,

        COUNT(*) AS row_count
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY
        position_state,
        button_state
    ORDER BY
        position_state,
        button_state
    """,
    connection,
)

position_code_button_summary = pd.read_sql_query(
    f"""
    SELECT
        pos AS position_code,
        COUNT(*) AS row_count,

        SUM(
            CASE
                WHEN ovr_btn = '-' AND btn = '-'
                THEN 1 ELSE 0
            END
        ) AS both_dash_rows,

        SUM(
            CASE
                WHEN ovr_btn <> '-' AND btn <> '-'
                THEN 1 ELSE 0
            END
        ) AS numeric_button_rows
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND typeof(pos) = 'text'
    GROUP BY pos
    ORDER BY row_count DESC, pos
    """,
    connection,
)

display(position_button_relationship)
display(position_code_button_summary)

,position_state,button_state,row_count
0,<numeric position>,both numeric,1756674
1,BD,both dash,1020
2,CO,both dash,77
3,DSQ,both numeric,619
4,F,both dash,15681
5,LFT,both dash,7
6,PU,both dash,65832
7,REF,both dash,291
8,RO,both dash,463
9,RR,both dash,723


,position_code,row_count,both_dash_rows,numeric_button_rows
0,PU,65832,65832,0
1,F,15681,15681,0
2,UR,9527,9527,0
3,BD,1020,1020,0
4,RR,723,723,0
5,DSQ,619,0,619
6,RO,463,463,0
7,SU,371,371,0
8,REF,291,291,0
9,CO,77,77,0


In [44]:
# Profile numeric finishing-distance values exactly as stored.
#
# Integer and real storage classes are combined as numeric values. The dash
# placeholder rows are excluded from these range calculations but remain
# documented separately.

button_numeric_profile = pd.read_sql_query(
    f"""
    SELECT
        MIN(
            CASE
                WHEN typeof(ovr_btn) IN ('integer', 'real')
                THEN ovr_btn
            END
        ) AS minimum_ovr_btn,

        MAX(
            CASE
                WHEN typeof(ovr_btn) IN ('integer', 'real')
                THEN ovr_btn
            END
        ) AS maximum_ovr_btn,

        COUNT(
            DISTINCT CASE
                WHEN typeof(ovr_btn) IN ('integer', 'real')
                THEN ovr_btn
            END
        ) AS distinct_numeric_ovr_btn,

        SUM(
            CASE
                WHEN typeof(ovr_btn) IN ('integer', 'real')
                 AND ovr_btn < 0
                THEN 1 ELSE 0
            END
        ) AS negative_ovr_btn_rows,

        SUM(
            CASE
                WHEN typeof(ovr_btn) IN ('integer', 'real')
                 AND ovr_btn = 0
                THEN 1 ELSE 0
            END
        ) AS zero_ovr_btn_rows,

        MIN(
            CASE
                WHEN typeof(btn) IN ('integer', 'real')
                THEN btn
            END
        ) AS minimum_btn,

        MAX(
            CASE
                WHEN typeof(btn) IN ('integer', 'real')
                THEN btn
            END
        ) AS maximum_btn,

        COUNT(
            DISTINCT CASE
                WHEN typeof(btn) IN ('integer', 'real')
                THEN btn
            END
        ) AS distinct_numeric_btn,

        SUM(
            CASE
                WHEN typeof(btn) IN ('integer', 'real')
                 AND btn < 0
                THEN 1 ELSE 0
            END
        ) AS negative_btn_rows,

        SUM(
            CASE
                WHEN typeof(btn) IN ('integer', 'real')
                 AND btn = 0
                THEN 1 ELSE 0
            END
        ) AS zero_btn_rows
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    """,
    connection,
)

button_common_values = pd.read_sql_query(
    f"""
    SELECT
        CAST(ovr_btn AS TEXT) AS stored_ovr_btn,
        CAST(btn AS TEXT) AS stored_btn,
        COUNT(*) AS row_count
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND typeof(ovr_btn) IN ('integer', 'real')
      AND typeof(btn) IN ('integer', 'real')
    GROUP BY
        ovr_btn,
        btn
    ORDER BY
        row_count DESC,
        ovr_btn,
        btn
    LIMIT 40
    """,
    connection,
)

display(button_numeric_profile)
display(button_common_values)

,minimum_ovr_btn,maximum_ovr_btn,distinct_numeric_ovr_btn,negative_ovr_btn_rows,zero_ovr_btn_rows,minimum_btn,maximum_btn,distinct_numeric_btn,negative_btn_rows,zero_btn_rows
0,0,331.5,950,0,189408,0,186,244,0,192162


,stored_ovr_btn,stored_btn,row_count
0,0,0,189408
1,0.5,0.5,19483
2,0.3,0.3,16795
3,0.75,0.75,15758
4,1.25,1.25,14137
5,1,1,11058
6,0.2,0.2,10957
7,1.5,1.5,10611
8,1.75,1.75,9238
9,2,2,7956


In [45]:
# Compare numeric finishing positions with ovr_btn and btn.
#
# This tests structural relationships only:
# - whether winners have zero values;
# - whether non-winners can also have zero values;
# - whether ovr_btn is ever smaller than btn.
#
# No formal racing-domain meaning is assigned yet.

position_distance_relationship = pd.read_sql_query(
    f"""
    SELECT
        SUM(
            CASE
                WHEN typeof(pos) = 'integer'
                 AND pos = 1
                THEN 1 ELSE 0
            END
        ) AS winner_rows,

        SUM(
            CASE
                WHEN typeof(pos) = 'integer'
                 AND pos = 1
                 AND ovr_btn = 0
                THEN 1 ELSE 0
            END
        ) AS winners_with_zero_ovr_btn,

        SUM(
            CASE
                WHEN typeof(pos) = 'integer'
                 AND pos = 1
                 AND btn = 0
                THEN 1 ELSE 0
            END
        ) AS winners_with_zero_btn,

        SUM(
            CASE
                WHEN typeof(pos) = 'integer'
                 AND pos > 1
                 AND ovr_btn = 0
                THEN 1 ELSE 0
            END
        ) AS non_winners_with_zero_ovr_btn,

        SUM(
            CASE
                WHEN typeof(pos) = 'integer'
                 AND pos > 1
                 AND btn = 0
                THEN 1 ELSE 0
            END
        ) AS non_winners_with_zero_btn,

        SUM(
            CASE
                WHEN typeof(ovr_btn) IN ('integer', 'real')
                 AND typeof(btn) IN ('integer', 'real')
                 AND ovr_btn < btn
                THEN 1 ELSE 0
            END
        ) AS rows_where_ovr_btn_less_than_btn,

        SUM(
            CASE
                WHEN typeof(ovr_btn) IN ('integer', 'real')
                 AND typeof(btn) IN ('integer', 'real')
                 AND ovr_btn = btn
                THEN 1 ELSE 0
            END
        ) AS rows_where_values_equal,

        SUM(
            CASE
                WHEN typeof(ovr_btn) IN ('integer', 'real')
                 AND typeof(btn) IN ('integer', 'real')
                 AND ovr_btn > btn
                THEN 1 ELSE 0
            END
        ) AS rows_where_ovr_btn_greater_than_btn
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    """,
    connection,
)

position_distance_examples = pd.read_sql_query(
    f"""
    SELECT
        pos,
        ovr_btn,
        btn,
        COUNT(*) AS row_count
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND typeof(pos) = 'integer'
      AND typeof(ovr_btn) IN ('integer', 'real')
      AND typeof(btn) IN ('integer', 'real')
    GROUP BY
        pos,
        ovr_btn,
        btn
    ORDER BY
        pos,
        row_count DESC,
        ovr_btn,
        btn
    LIMIT 60
    """,
    connection,
)

display(position_distance_relationship)
display(position_distance_examples)

,winner_rows,winners_with_zero_ovr_btn,winners_with_zero_btn,non_winners_with_zero_ovr_btn,non_winners_with_zero_btn,rows_where_ovr_btn_less_than_btn,rows_where_values_equal,rows_where_ovr_btn_greater_than_btn
0,189344,188844,188845,371,3121,421,386778,1370094


,pos,ovr_btn,btn,row_count
0,0,0.00,0.00,8
1,1,0.00,0.00,188844
2,1,0.05,0.05,86
3,1,0.20,0.20,82
4,1,0.10,0.10,74
5,1,0.30,0.30,54
6,1,0.50,0.50,39
7,1,0.75,0.75,28
8,1,1.00,1.00,15
9,1,1.50,1.50,15


In [46]:
# Inspect unusual relationships among pos, ovr_btn, and btn.
#
# The selected rows include:
# - numeric position zero;
# - winners with a non-zero distance;
# - non-winners with zero overall distance;
# - rows where overall distance is smaller than the adjacent margin.
#
# These are evidence candidates only. No correction is applied.

unusual_position_distance_rows = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        date,
        course,
        race_id,
        off,
        race_name,
        type,
        ran,
        num,
        pos,
        ovr_btn,
        btn,
        horse,
        comment,
        CASE
            WHEN typeof(pos) = 'integer'
             AND pos = 0
            THEN 'position zero'

            WHEN typeof(pos) = 'integer'
             AND pos = 1
             AND (
                    ovr_btn <> 0
                 OR btn <> 0
             )
            THEN 'winner with non-zero distance'

            WHEN typeof(pos) = 'integer'
             AND pos > 1
             AND ovr_btn = 0
            THEN 'non-winner with zero overall distance'

            WHEN typeof(ovr_btn) IN ('integer', 'real')
             AND typeof(btn) IN ('integer', 'real')
             AND ovr_btn < btn
            THEN 'overall distance smaller than btn'

            ELSE 'other'
        END AS issue_type
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND (
            (
                typeof(pos) = 'integer'
                AND pos = 0
            )
         OR (
                typeof(pos) = 'integer'
                AND pos = 1
                AND (
                       ovr_btn <> 0
                    OR btn <> 0
                )
            )
         OR (
                typeof(pos) = 'integer'
                AND pos > 1
                AND ovr_btn = 0
            )
         OR (
                typeof(ovr_btn) IN ('integer', 'real')
                AND typeof(btn) IN ('integer', 'real')
                AND ovr_btn < btn
            )
      )
    ORDER BY
        issue_type,
        date,
        course,
        off,
        pos,
        rowid
    """,
    connection,
)

unusual_position_distance_summary = (
    unusual_position_distance_rows
    .groupby("issue_type", as_index=False)
    .agg(
        runner_rows=("source_rowid", "count"),
        apparent_races=(
            "race_name",
            lambda _: 0,
        ),
        distinct_courses=("course", "nunique"),
    )
)

unusual_position_distance_summary["apparent_races"] = [
    unusual_position_distance_rows.loc[
        unusual_position_distance_rows["issue_type"] == issue_type,
        ["date", "course", "off", "race_name"],
    ].drop_duplicates().shape[0]
    for issue_type in unusual_position_distance_summary["issue_type"]
]

display(unusual_position_distance_summary)

for issue_type in unusual_position_distance_summary["issue_type"]:
    print(f"\n{issue_type.upper()} — FIRST 30 ROWS")

    with pd.option_context(
        "display.max_rows",
        30,
        "display.max_colwidth",
        120,
    ):
        display(
            unusual_position_distance_rows.loc[
                unusual_position_distance_rows["issue_type"] == issue_type
            ].head(30)
        )

,issue_type,runner_rows,apparent_races,distinct_courses
0,non-winner with zero overall distance,371,341,124
1,overall distance smaller than btn,421,421,121
2,position zero,8,2,2
3,winner with non-zero distance,500,499,133



NON-WINNER WITH ZERO OVERALL DISTANCE — FIRST 30 ROWS


,source_rowid,date,course,race_id,off,race_name,type,ran,num,pos,ovr_btn,btn,horse,comment,issue_type
0,2688,2015-01-08,Gulfstream Park (USA),617268,7:07,Maiden Special Weight (Maiden) (3yo) (Dirt),Flat,8,8,6,0.0,0.0,Tradesman (USA),Broke well from wide draw and chased leaders - close 3rd and nudged along halfway - shaken up to challenge outside t...,non-winner with zero overall distance
1,4538,2015-01-15,Wolverhampton (AW),616318,6:40,Bet In Play At Coral Handicap (Tapeta),Flat,7,7,2,0.0,0.0,Warfare (GB),Chased leaders - led over 1f out - ridden and edged left inside final furlong - edged right towards finish - joined ...,non-winner with zero overall distance
2,9570,2015-01-31,Tampa Bay Downs (USA),618671,10:50,Lambholm South Endeavour Stakes (Fillies & Mares) (Turf),Flat,11,,2,0.0,0.0,Hard Not To Like (CAN),Finished 1st - disqualified and placed 2nd,non-winner with zero overall distance
3,12161,2015-02-08,Delta downs (USA),619102,3:24,LA Bred Premier Night Matron Stakes (Fillies & Mares) (Dirt),Flat,9,,2,0.0,0.0,Solid Sender (USA),Finsihed 1st - disqualified and placed 2nd,non-winner with zero overall distance
4,13538,2015-02-13,Chantilly (FR),619427,4:25,Prix du Chemin Royal (Handicap) (5yo+) (Polytrack),Flat,17,14,3,0.0,0.0,Gonetrio (USA),Finished 1st - disqualified and placed 3rd,non-winner with zero overall distance
5,16967,2015-02-21,Gulfstream Park (USA),619838,10:30,Besilu Stables Fountain Of Youth Stakes (3yo) (Dirt),Flat,8,,2,0.0,0.0,Upstart (USA),Finished 1st - disqualified and placed 2nd,non-winner with zero overall distance
6,17590,2015-02-24,Catterick,618467,5:00,Racing Again 4th March Handicap Hurdle,Hurdle,7,1,2,0.0,0.0,Harris (IRE),Made all - driven 8th - eged left run-in - all out - finished 1st - disqualified and placed 2nd(op 14/1 tchd 18/1),non-winner with zero overall distance
7,41078,2015-04-18,Keeneland (USA),624644,9:11,Maiden Special Weight (3yo) (Turf),Flat,12,2,10,0.0,0.0,Lookaroundcorners (USA),Finished 1st - disqualified and placed 10th,non-winner with zero overall distance
8,45551,2015-04-28,Nottingham,623004,4:50,£10 Free At 32Red Handicap,Flat,7,5,2,0.0,0.0,Mountain Man (GB),In rear - switched right to outer and headway well over 1f out - soon ridden and edged left - strong run to lead ent...,non-winner with zero overall distance
9,50532,2015-05-07,Worcester,626399,5:05,Post Your Bets @ bookies.com Mares Maiden Hurdle (Div II),Hurdle,10,4,2,0.0,0.0,Nellies Quest (GB),Held up - headway and not fluent 5th - led 2 out - ridden and went right last - all out - finished 1st - placed 2nd(...,non-winner with zero overall distance



OVERALL DISTANCE SMALLER THAN BTN — FIRST 30 ROWS


,source_rowid,date,course,race_id,off,race_name,type,ran,num,pos,ovr_btn,btn,horse,comment,issue_type
371,5651,2015-01-18,Nakayama (JPN),617866,6:35,Kiesei Hai Stakes (Turf),Flat,17,,3,0.25,0.3,Kluger (JPN),,overall distance smaller than btn
372,6683,2015-01-22,Meydan (UAE),617889,6:15,gulfnews.com (Handicap) (Turf),Flat,14,11,3,0.25,0.3,Earth Drummer (IRE),Mid-division - ran on final 2f - just failed,overall distance smaller than btn
373,7027,2015-01-23,Wolverhampton (AW),616393,4:15,Bet In Play At Coral Apprentice Handicap (Tapeta),Flat,10,5,3,0.25,0.3,Thrtypointstothree (IRE),Raced keenly - chased leaders - ridden and outpaced over 2f out - rallied inside final furlong - ran on and closed t...,overall distance smaller than btn
374,9103,2015-01-30,Wolverhampton (AW),616967,5:45,Download The Coral App Handicap (Tapeta),Flat,8,8,3,0.25,0.3,Perfect Cracker (GB),Tracked leaders - raced keenly - ridden to lead 1f out - edged left and headed well inside final furlong - ran on(tc...,overall distance smaller than btn
375,12878,2015-02-11,Lingfield (AW),617696,3:05,32RedPoker.com Handicap,Flat,8,9,8,111.00,147.0,Premier Jacks (GB),Took keen hold - led and soon clear - headed 7f out - dropped out rapidly 5f out - tailed final 4f,overall distance smaller than btn
376,13937,2015-02-14,Caulfield (AUS),619440,4:55,Herald Sun Carlyon Cup (Turf),Flat,11,8,3,0.25,0.3,Backstedt (AUS),,overall distance smaller than btn
377,13954,2015-02-14,Lingfield (AW),617718,1:25,Unibet Offer Daily Jockey/Trainer Specials Median Auction Maiden Stakes,Flat,7,1,3,0.25,0.3,Autumn Tonic (IRE),In touch in midfield - effort on inner over 1f out - every chance final 100yds - ran on - no extra close home(op 6/4...,overall distance smaller than btn
378,14611,2015-02-15,Kyoto (JPN),619464,6:35,Kyoto Kinen (Grade2) (Turf),Flat,11,,3,0.25,0.3,Kizuna (JPN),,overall distance smaller than btn
379,14954,2015-02-16,Laurel Park (USA),619524,8:57,Miracle Wood Stakes (Dirt),Flat,6,2,3,0.25,0.3,Golden Years (USA),,overall distance smaller than btn
380,14975,2015-02-16,Wolverhampton (AW),618007,3:05,Coral Connect Maiden Stakes (Tapeta),Flat,9,5,3,0.25,0.3,Goathland (IRE),Prominent - chased leader 7f out - shaken up over 3f out - led over 2f out - ridden and hung left over 1f out - head...,overall distance smaller than btn



POSITION ZERO — FIRST 30 ROWS


,source_rowid,date,course,race_id,off,race_name,type,ran,num,pos,ovr_btn,btn,horse,comment,issue_type
792,40679,2015-04-18,Nakayama (JPN),624701,7:40,Nakayama Grand Jump (Turf),Flat,15,,0,0.0,0.0,Rikiai Kurofune (JPN),,position zero
793,40680,2015-04-18,Nakayama (JPN),624701,7:40,Nakayama Grand Jump (Turf),Flat,15,,0,0.0,0.0,Fire (JPN),,position zero
794,40681,2015-04-18,Nakayama (JPN),624701,7:40,Nakayama Grand Jump (Turf),Flat,15,,0,0.0,0.0,Shonan Coming (JPN),,position zero
795,40682,2015-04-18,Nakayama (JPN),624701,7:40,Nakayama Grand Jump (Turf),Flat,15,,0,0.0,0.0,Tenjin Kiyomori (JPN),,position zero
796,40683,2015-04-18,Nakayama (JPN),624701,7:40,Nakayama Grand Jump (Turf),Flat,15,,0,0.0,0.0,Country Snow (JPN),,position zero
797,77097,2015-06-28,Klampenborg (DEN),630668,3:00,Dansk Hesteforsikring Scandinavian Open Championship (3yo+) (Turf),Flat,13,1,0,0.0,0.0,Bank Of Burden (USA),,position zero
798,77098,2015-06-28,Klampenborg (DEN),630668,3:00,Dansk Hesteforsikring Scandinavian Open Championship (3yo+) (Turf),Flat,13,10,0,0.0,0.0,Mekong River (IRE),,position zero
799,77099,2015-06-28,Klampenborg (DEN),630668,3:00,Dansk Hesteforsikring Scandinavian Open Championship (3yo+) (Turf),Flat,13,2,0,0.0,0.0,Moe Green (IRE),,position zero



WINNER WITH NON-ZERO DISTANCE — FIRST 30 ROWS


,source_rowid,date,course,race_id,off,race_name,type,ran,num,pos,ovr_btn,btn,horse,comment,issue_type
800,2657,2015-01-08,Gulfstream Park (USA),617268,7:07,Maiden Special Weight (Maiden) (3yo) (Dirt),Flat,8,2,1,0.50,0.50,Royal Son (USA),Finished 2nd - awarded the race,winner with non-zero distance
801,9564,2015-01-31,Tampa Bay Downs (USA),618671,10:50,Lambholm South Endeavour Stakes (Fillies & Mares) (Turf),Flat,11,,1,0.75,0.75,Testa Rossi (FR),Finished 2nd - awarded the race,winner with non-zero distance
802,12159,2015-02-08,Delta downs (USA),619102,3:24,LA Bred Premier Night Matron Stakes (Fillies & Mares) (Dirt),Flat,9,,1,0.05,0.05,Afternoon Tango (USA),Finished 2nd - awarded the race,winner with non-zero distance
803,13537,2015-02-13,Chantilly (FR),619427,4:25,Prix du Chemin Royal (Handicap) (5yo+) (Polytrack),Flat,17,1,1,2.50,2.50,El Nino (FR),Finished 2nd - placed 1st,winner with non-zero distance
804,14782,2015-02-15,Navan (IRE),619171,3:55,Ten Up Novice Chase,Chase,6,2,1,4.25,4.25,Noble Emperor (IRE),Led narrowly until joined before 4th where not fluent - regained advantage from 6th until headed from 8th - regained...,winner with non-zero distance
805,15824,2015-02-19,Meydan (UAE),619483,5:05,District One (Handicap) (Dirt),Flat,15,6,1,1.25,1.25,Toolain (IRE),Mid-division - smooth progress 3f out - ran on well final 2f but no chance with winner - finished 2nd - awarded the ...,winner with non-zero distance
806,16168,2015-02-20,Jebel Ali (UAE),619514,11:35,Jebel Ali Mile Sponsored By Derrinstown Stud (Dirt),Flat,9,10,1,0.10,0.10,Silver Galaxy (GB),Tracked leader - led 2 1/2f out - headed 100yds out - battled back - just failed - finished 2nd - later awarded the ...,winner with non-zero distance
807,16970,2015-02-21,Gulfstream Park (USA),619838,10:30,Besilu Stables Fountain Of Youth Stakes (3yo) (Dirt),Flat,8,,1,2.75,2.75,Itsaknockout (USA),Finished 2nd - awarded the race,winner with non-zero distance
808,17591,2015-02-24,Catterick,618467,5:00,Racing Again 4th March Handicap Hurdle,Hurdle,7,5,1,0.10,0.10,Allez Cool (IRE),Chased leaders - 2nd from 8th - driven approaching 2 out - upsides when carried left run-in - just held - finished 2...,winner with non-zero distance
809,19222,2015-02-28,Meydan (UAE),619987,3:20,IPIC 30th Anniversary (Handicap) (Dirt),Flat,8,5,1,1.50,1.50,Press Room (USA),Tracked leaders - every chance 1 1/2f out - ran on same pace final furlong - finished 2nd - later awarded the race,winner with non-zero distance


In [47]:
# Separate small notation differences from larger contradictions where
# overall distance is smaller than the adjacent margin.
#
# A difference of 0.05 is common in the data and may reflect rounding between
# quarter-length and decimal representations. Larger gaps need inspection.

ovr_btn_less_than_btn_profile = pd.read_sql_query(
    f"""
    SELECT
        ROUND(btn - ovr_btn, 4) AS btn_minus_ovr_btn,
        COUNT(*) AS row_count,
        COUNT(
            DISTINCT date || '|' || course || '|' || off || '|' || race_name
        ) AS apparent_race_count
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND typeof(ovr_btn) IN ('integer', 'real')
      AND typeof(btn) IN ('integer', 'real')
      AND ovr_btn < btn
    GROUP BY ROUND(btn - ovr_btn, 4)
    ORDER BY btn_minus_ovr_btn
    """,
    connection,
)

material_ovr_btn_contradictions = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        date,
        course,
        race_id,
        off,
        race_name,
        type,
        ran,
        num,
        pos,
        ovr_btn,
        btn,
        ROUND(btn - ovr_btn, 4) AS btn_minus_ovr_btn,
        horse,
        comment
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND typeof(ovr_btn) IN ('integer', 'real')
      AND typeof(btn) IN ('integer', 'real')
      AND ovr_btn < btn
      AND (btn - ovr_btn) > 0.05
    ORDER BY
        btn_minus_ovr_btn DESC,
        date,
        course,
        off,
        pos,
        rowid
    """,
    connection,
)

print(
    "Rows where ovr_btn < btn: "
    f"{ovr_btn_less_than_btn_profile['row_count'].sum():,}"
)
print(
    "Rows where btn exceeds ovr_btn by more than 0.05: "
    f"{len(material_ovr_btn_contradictions):,}"
)

with pd.option_context("display.max_rows", None):
    display(ovr_btn_less_than_btn_profile)

with pd.option_context(
    "display.max_rows",
    100,
    "display.max_colwidth",
    120,
):
    display(material_ovr_btn_contradictions.head(100))

Rows where ovr_btn < btn: 421
Rows where btn exceeds ovr_btn by more than 0.05: 61


,btn_minus_ovr_btn,row_count,apparent_race_count
0,0.05,360,360
1,1.00,1,1
2,1.25,2,2
3,1.50,1,1
4,2.50,2,2
5,3.75,1,1
6,4.00,1,1
7,4.50,1,1
8,5.00,2,2
9,5.50,1,1


,source_rowid,date,course,race_id,off,race_name,type,ran,num,pos,ovr_btn,btn,btn_minus_ovr_btn,horse,comment
0,553992,2018-06-12,Thirsk,702720,3:15,100% Racing UK Profits Back To Racing Handicap,Flat,16,2,16,43.75,182,138.25,Lexington Times (IRE),Effectively refused to race (starter reported gelding had been reluctant to race and virtually refused to race)
1,272120,2016-09-26,Newton Abbot,658353,3:00,Bryan And Ilva Westcott Memorial Novices Hurdle,Hurdle,3,2,3,100.75,166,65.25,Captain Bonsai (GB),Tracked leaders - awkward on bend after 6th - soon lost touch - tailed off
2,202772,2016-05-06,Lingfield (AW),648250,1:30,Heart FM Maiden Stakes,Flat,7,4,7,108.25,166,57.75,Northman (IRE),Pushed along after 3f - under strong pressure by halfway and soon dropped away - tailed off when virtually pulled up...
3,203540,2016-05-07,Lingfield,648279,4:35,betfred.com Handicap,Flat,7,1,7,109.50,166,56.50,Slowfoot (GER),Steadied start - held up in last pair - lost touch 3f out - virtually pulled up final 2f - tailed off (trainer said ...
4,695646,2019-04-13,Wolverhampton (AW),725295,6:30,Betway Sprint Handicap,Flat,10,2,10,109.50,166,56.50,The Establishment (GB),Reared start and lost all chance(op 11/1 tchd 10/1)
5,63575,2015-05-31,Fakenham,626000,4:30,Welcome to Norfolk Princess Charlotte Novices Limited Handicap Chase,Chase,4,5,3,102.00,155,53.00,Radsoc De Sivola (FR),Soon detached in last - tailed off from 7th(op 66/1)
6,302856,2016-11-30,Lingfield (AW),662707,12:10,sunbets.co.uk EBF Maiden Stakes,Flat,7,7,7,114.00,155,41.00,Portland Belle (IRE),Very slowly away - flashing tail and refused to race properly - soon tailed off(op 14/1 tchd 16/1 and 10/1)
7,308624,2016-12-14,Lingfield (AW),663694,3:00,Betway Maiden Stakes,Flat,9,2,9,115.75,156,40.25,Thenobleprankster (IRE),Chased leader 3f out - soon niggled along - lost place and behind 6f out - lost touch quickly 4f out - soon eased an...
8,580608,2018-08-04,Chelmsford (AW),706799,1:40,Bet toteexacta At totesport.com Handicap,Flat,9,5,9,126.00,166,40.00,Fantasy Keeper (GB),Chased leading pair on the outer - niggled along when blowing bend over 3f out - virtually pulled up after (jockey s...
9,582419,2018-08-08,Yarmouth,707176,5:20,Eastern Power Systems Of Norwich Handicap,Flat,7,7,6,106.75,145,38.25,Caledonia Earl (GB),Badly hampered start - always tailed off and hacking along (jockey said gelding was badly hampered by unseated horse...


In [48]:
# Profile numeric and special finishing-position values separately.
#
# This records the complete stored range and checks whether numeric positions
# exceed the stated number of runners in the race.

position_numeric_profile = pd.read_sql_query(
    f"""
    SELECT
        MIN(
            CASE
                WHEN typeof(pos) = 'integer'
                THEN pos
            END
        ) AS minimum_numeric_position,

        MAX(
            CASE
                WHEN typeof(pos) = 'integer'
                THEN pos
            END
        ) AS maximum_numeric_position,

        COUNT(
            DISTINCT CASE
                WHEN typeof(pos) = 'integer'
                THEN pos
            END
        ) AS distinct_numeric_positions,

        SUM(
            CASE
                WHEN typeof(pos) = 'integer'
                 AND pos < 0
                THEN 1 ELSE 0
            END
        ) AS negative_position_rows,

        SUM(
            CASE
                WHEN typeof(pos) = 'integer'
                 AND pos = 0
                THEN 1 ELSE 0
            END
        ) AS zero_position_rows,

        SUM(
            CASE
                WHEN typeof(pos) = 'integer'
                 AND pos > ran
                THEN 1 ELSE 0
            END
        ) AS position_greater_than_ran_rows
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    """,
    connection,
)

position_numeric_distribution = pd.read_sql_query(
    f"""
    SELECT
        pos AS stored_position,
        COUNT(*) AS row_count,
        ROUND(
            100.0 * COUNT(*) /
            (
                SELECT COUNT(*)
                FROM data
                WHERE {DATA_ROW_PREDICATE}
            ),
            4
        ) AS row_pct
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND typeof(pos) = 'integer'
    GROUP BY pos
    ORDER BY pos
    """,
    connection,
)

display(position_numeric_profile)

with pd.option_context("display.max_rows", None):
    display(position_numeric_distribution)

,minimum_numeric_position,maximum_numeric_position,distinct_numeric_positions,negative_position_rows,zero_position_rows,position_greater_than_ran_rows
0,0,34,35,0,8,10


,stored_position,row_count,row_pct
0,0,8,0.0004
1,1,189344,10.2277
2,2,189007,10.2095
3,3,188141,10.1627
4,4,184662,9.9748
5,5,175886,9.5008
6,6,161327,8.7143
7,7,142993,7.7240
8,8,122268,6.6045
9,9,101726,5.4949


In [49]:
# Inspect every row where the numeric finishing position exceeds the stated
# number of runners.
#
# This will show whether the mismatch comes from:
# - an incorrect ran value;
# - a malformed finishing position;
# - a race record affected by withdrawals, disqualifications, or incomplete
#   source coverage.

position_exceeds_ran_rows = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        date,
        course,
        race_id,
        off,
        race_name,
        type,
        ran,
        num,
        pos,
        ovr_btn,
        btn,
        horse,
        jockey,
        trainer,
        comment
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND typeof(pos) = 'integer'
      AND pos > ran
    ORDER BY
        date,
        course,
        off,
        pos,
        rowid
    """,
    connection,
)

print(
    "Rows where numeric pos exceeds ran: "
    f"{len(position_exceeds_ran_rows):,}"
)

with pd.option_context(
    "display.max_rows",
    None,
    "display.max_columns",
    None,
    "display.max_colwidth",
    140,
):
    display(position_exceeds_ran_rows)

Rows where numeric pos exceeds ran: 10


,source_rowid,date,course,race_id,off,race_name,type,ran,num,pos,ovr_btn,btn,horse,jockey,trainer,comment
0,270486,2016-09-23,Haydock,657954,5:05,Griffiths And Armour Handicap,Flat,13,14,14,19.75,3.00,Jack Of Diamonds (IRE),Oisin Murphy,Roger Teal,In touch on outer - ridden along over 3f out - soon weakened(op 18/1)
1,330864,2017-02-12,Gavea (BRZ),669090,7:45,Grande Premio Henrique Possollo (3yo Fillies) (Turf),Flat,18,8,19,15.50,0.75,Pista Olimpica (BRZ),W Blandi,R Solanes,
2,334725,2017-02-23,Meydan (UAE),669525,5:25,Curlin Handicap Sponsored by Al Naboodah SUNWIN Buses (Listed Handicap) (Dirt),Flat,9,9,10,58.75,10.50,Beach Bar (IRE),Tadhg OShea,Brendan Powell,Never better than mid-division
3,854999,2020-04-10,Gulfstream Park (USA),755688,9:41,Allowance Optional Claiming Race (Allowance) (3yo Fillies) (Main Track) (Dirt),Flat,8,6,9,42.25,20.00,Takestwotowiggle (USA),Marcos Meneses,Steve Budhoo,
4,875197,2020-08-15,Merano (ITY),765471,5:55,EBF Terme Di Merano () (3yo+) (Fillies & Mares) (Turf),Flat,6,,7,8.25,0.00,Quita (GB),Ivan Rossi,Waldemar Hickst,In rear of midfield - ridden along over 2f out - kept on - never on terms with leaders
5,875198,2020-08-15,Merano (ITY),765471,5:55,EBF Terme Di Merano () (3yo+) (Fillies & Mares) (Turf),Flat,6,,9,8.25,0.00,Volkovka (FR),Dario Vargiu,Simone Brogi,Prominent - pushed along to chase leader over 3f out - ridden 2f out - limited response - eased when held inside final furlong - btn 20 ...
6,1439296,2023-12-09,Oaklawn Park (USA),857908,10:14,Mistletoe Stakes (Black Type) (Fillies & Mares) (Dirt),Flat,4,,8,5.25,4.50,Effortlesslyelgant (USA),Rafael Bejarano,Norm W Casse,Finished fourth - placed eighth - caused interference
7,1445952,2023-12-23,Gulfstream Park (USA),857782,9:36,Mr. Prospector Stakes (3yo+) (Main Track) (Dirt),Flat,8,3,9,28.25,11.50,Scaramouche (USA),John R Velazquez,Guadalupe Preciado,
8,1461415,2024-02-04,Tokyo (JPN),860553,6:45,Tokyo Shimbun Hai (4yo+) (Turf),Flat,16,12,18,18.00,2.50,Kona Coast (JPN),Keita Tosaki,Hisashi Shimizu,
9,1813783,2026-03-13,Meydan,915628,17:45,Binghatti (Handicap) (Dirt),Flat,9,2,10,35.50,20.00,Local Dynasty (IRE),Silvestre De Sousa,Simon & Ed Crisford,Never better than midfield


In [50]:
# Inspect every runner from the races containing pos > ran.
#
# This compares:
# - the repeated ran value;
# - the number of physical source rows;
# - the complete set of finishing positions and status codes.
#
# The aim is to determine whether the mismatch is caused by a single bad row
# or by a race-level convention.

position_exceeds_ran_races = position_exceeds_ran_rows[
    ["date", "course", "off", "race_name"]
].drop_duplicates()

race_condition_parts = []
race_parameters = []

for race in position_exceeds_ran_races.itertuples(index=False):
    race_condition_parts.append(
        "(date = ? AND course = ? AND off = ? AND race_name = ?)"
    )
    race_parameters.extend(
        [race.date, race.course, race.off, race.race_name]
    )

race_condition_sql = " OR ".join(race_condition_parts)

affected_race_rows = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        date,
        course,
        race_id,
        off,
        race_name,
        ran,
        num,
        pos,
        ovr_btn,
        btn,
        horse,
        comment
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND ({race_condition_sql})
    ORDER BY
        date,
        course,
        off,
        CASE
            WHEN typeof(pos) = 'integer' THEN pos
            ELSE 999
        END,
        rowid
    """,
    connection,
    params=race_parameters,
)

affected_race_summary = (
    affected_race_rows
    .groupby(
        ["date", "course", "off", "race_name"],
        as_index=False,
    )
    .agg(
        repeated_ran=("ran", "first"),
        physical_rows=("source_rowid", "count"),
        maximum_numeric_pos=(
            "pos",
            lambda values: max(
                [
                    int(value)
                    for value in values
                    if isinstance(value, int)
                ],
                default=None,
            ),
        ),
        stored_positions=(
            "pos",
            lambda values: " | ".join(
                str(value) for value in values
            ),
        ),
    )
)

display(affected_race_summary)

for race in position_exceeds_ran_races.itertuples(index=False):
    print(
        f"\n{race.date} — {race.course} — "
        f"{race.off} — {race.race_name}"
    )

    race_rows = affected_race_rows.loc[
        (affected_race_rows["date"] == race.date)
        & (affected_race_rows["course"] == race.course)
        & (affected_race_rows["off"] == race.off)
        & (affected_race_rows["race_name"] == race.race_name)
    ]

    with pd.option_context(
        "display.max_rows",
        None,
        "display.max_columns",
        None,
        "display.max_colwidth",
        120,
    ):
        display(race_rows)

,date,course,off,race_name,repeated_ran,physical_rows,maximum_numeric_pos,stored_positions
0,2016-09-23,Haydock,5:05,Griffiths And Armour Handicap,13,13,14,1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 | ...
1,2017-02-12,Gavea (BRZ),7:45,Grande Premio Henrique Possollo (3yo Fillies)...,18,18,19,1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 | ...
2,2017-02-23,Meydan (UAE),5:25,Curlin Handicap Sponsored by Al Naboodah SUNWI...,9,9,10,1 | 2 | 3 | 4 | 5 | 6 | 7 | 9 | 10
3,2020-04-10,Gulfstream Park (USA),9:41,Allowance Optional Claiming Race (Allowance) (...,8,8,9,1 | 2 | 3 | 5 | 6 | 7 | 8 | 9
4,2020-08-15,Merano (ITY),5:55,EBF Terme Di Merano () (3yo+) (Fillies & Mares...,6,6,9,1 | 2 | 3 | 4 | 7 | 9
5,2023-12-09,Oaklawn Park (USA),10:14,Mistletoe Stakes (Black Type) (Fillies & Mares...,4,4,8,1 | 2 | 3 | 8
6,2023-12-23,Gulfstream Park (USA),9:36,Mr. Prospector Stakes (3yo+) (Main Track) (Dirt),8,8,9,1 | 2 | 3 | 4 | 6 | 7 | 8 | 9
7,2024-02-04,Tokyo (JPN),6:45,Tokyo Shimbun Hai (4yo+) (Turf),16,16,18,1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 | ...
8,2026-03-13,Meydan,17:45,Binghatti (Handicap) (Dirt),9,9,10,1 | 2 | 3 | 4 | 5 | 6 | 7 | 9 | 10



2016-09-23 — Haydock — 5:05 — Griffiths And Armour Handicap


,source_rowid,date,course,race_id,off,race_name,ran,num,pos,ovr_btn,btn,horse,comment
0,270465,2016-09-23,Haydock,657954,5:05,Griffiths And Armour Handicap,13,13,1,0.00,0.00,Pumaflor (IRE),Slight lead - headed over 4f out - close up - led again 2f out - ridden well over 1f out - driven inside final furlo...
1,270481,2016-09-23,Haydock,657954,5:05,Griffiths And Armour Handicap,13,8,2,0.20,0.20,Top Beak (IRE),Close up on inner - slight lead over 4f out - ridden and headed 2f out - driven and rallied entering final furlong -...
2,270467,2016-09-23,Haydock,657954,5:05,Griffiths And Armour Handicap,13,11,3,2.50,2.25,Dark Devil (IRE),Tracked leaders - smooth headway on outer over 3f out - close up 2f out - soon ridden and every chance - driven and ...
3,270495,2016-09-23,Haydock,657954,5:05,Griffiths And Armour Handicap,13,2,4,3.00,0.50,Rousayan (IRE),Chased leaders - ridden along over 2f out - driven over 1f out - kept on final furlong(op 7/1)
4,270494,2016-09-23,Haydock,657954,5:05,Griffiths And Armour Handicap,13,10,5,3.50,0.50,Column (GB),Tracked leaders - headway over 3f out - ridden along 2f out - driven over 1f out - kept on same pace
5,270493,2016-09-23,Haydock,657954,5:05,Griffiths And Armour Handicap,13,7,6,4.25,0.75,Wilde Inspiration (IRE),Held up in touch - effort well over 2f out - soon ridden - kept on same pace(op 11/1)
6,270492,2016-09-23,Haydock,657954,5:05,Griffiths And Armour Handicap,13,6,7,5.00,0.75,See The Rock (IRE),Held up in rear - headway over 2f out - ridden well over 1f out - stayed on final furlong - nearest finish
7,270491,2016-09-23,Haydock,657954,5:05,Griffiths And Armour Handicap,13,1,8,5.00,0.05,Fort Bastion (IRE),Held up in rear - headway 2f out - soon ridden and kept on final furlong - nearest finish (jockey said that the geld...
8,270490,2016-09-23,Haydock,657954,5:05,Griffiths And Armour Handicap,13,4,9,5.50,0.50,Commodore (IRE),In touch - headway to chase leaders over 4f out - ridden along over 3f out - one pace final 2f(op 16/1)
9,270489,2016-09-23,Haydock,657954,5:05,Griffiths And Armour Handicap,13,3,10,6.50,1.00,Gratzie (GB),Dwelt and behind - headway on inner 3f out - soon not clear run and never dangerous (jockey said that the mare was s...



2017-02-12 — Gavea (BRZ) — 7:45 — Grande Premio Henrique Possollo  (3yo Fillies) (Turf)


,source_rowid,date,course,race_id,off,race_name,ran,num,pos,ovr_btn,btn,horse,comment
13,330842,2017-02-12,Gavea (BRZ),669090,7:45,Grande Premio Henrique Possollo (3yo Fillies) (Turf),18,13,1,0.00,0.00,No Regrets (BRZ),
14,330871,2017-02-12,Gavea (BRZ),669090,7:45,Grande Premio Henrique Possollo (3yo Fillies) (Turf),18,11,2,2.00,2.00,Vega Sicilia (BRZ),
15,330840,2017-02-12,Gavea (BRZ),669090,7:45,Grande Premio Henrique Possollo (3yo Fillies) (Turf),18,3,3,4.25,2.25,Etapa Vencida (BRZ),
16,330839,2017-02-12,Gavea (BRZ),669090,7:45,Grande Premio Henrique Possollo (3yo Fillies) (Turf),18,5,4,4.50,0.30,Expense Account (BRZ),
17,330838,2017-02-12,Gavea (BRZ),669090,7:45,Grande Premio Henrique Possollo (3yo Fillies) (Turf),18,7,5,5.50,1.00,Envoi De Fleurs (BRZ),
18,330837,2017-02-12,Gavea (BRZ),669090,7:45,Grande Premio Henrique Possollo (3yo Fillies) (Turf),18,1,6,6.00,0.50,Kings Gate (BRZ),
19,330836,2017-02-12,Gavea (BRZ),669090,7:45,Grande Premio Henrique Possollo (3yo Fillies) (Turf),18,9,7,6.75,0.75,Et La Vie Continue (BRZ),
20,330835,2017-02-12,Gavea (BRZ),669090,7:45,Grande Premio Henrique Possollo (3yo Fillies) (Turf),18,15,8,7.50,0.75,Domenica (BRZ),
21,330834,2017-02-12,Gavea (BRZ),669090,7:45,Grande Premio Henrique Possollo (3yo Fillies) (Turf),18,4,9,7.75,0.30,Friendlys (BRZ),
22,330849,2017-02-12,Gavea (BRZ),669090,7:45,Grande Premio Henrique Possollo (3yo Fillies) (Turf),18,9,10,8.25,0.30,Electora (BRZ),


...

,source_rowid,date,course,race_id,off,race_name,ran,num,pos,ovr_btn,btn,horse,comment
58,1445932,2023-12-23,Gulfstream Park (USA),857782,9:36,Mr. Prospector Stakes (3yo+) (Main Track) (Dirt),8,9,1,4.25,4.25,Sibelius (USA),
59,1445951,2023-12-23,Gulfstream Park (USA),857782,9:36,Mr. Prospector Stakes (3yo+) (Main Track) (Dirt),8,7,2,4.00,4.00,Gilmore (USA),
60,1445947,2023-12-23,Gulfstream Park (USA),857782,9:36,Mr. Prospector Stakes (3yo+) (Main Track) (Dirt),8,6,3,9.00,0.75,Dreaming Of Kona (USA),
61,1445948,2023-12-23,Gulfstream Park (USA),857782,9:36,Mr. Prospector Stakes (3yo+) (Main Track) (Dirt),8,8,4,9.00,0.05,Long Range Toddy (USA),
62,1445949,2023-12-23,Gulfstream Park (USA),857782,9:36,Mr. Prospector Stakes (3yo+) (Main Track) (Dirt),8,5,6,9.50,0.50,Hurricane J (USA),
63,1445950,2023-12-23,Gulfstream Park (USA),857782,9:36,Mr. Prospector Stakes (3yo+) (Main Track) (Dirt),8,2,7,10.50,1.00,Howbeit (USA),
64,1445955,2023-12-23,Gulfstream Park (USA),857782,9:36,Mr. Prospector Stakes (3yo+) (Main Track) (Dirt),8,4,8,16.75,6.25,Winfromwithin (USA),
65,1445952,2023-12-23,Gulfstream Park (USA),857782,9:36,Mr. Prospector Stakes (3yo+) (Main Track) (Dirt),8,3,9,28.25,11.50,Scaramouche (USA),



2024-02-04 — Tokyo (JPN) — 6:45 — Tokyo Shimbun Hai  (4yo+) (Turf)


,source_rowid,date,course,race_id,off,race_name,ran,num,pos,ovr_btn,btn,horse,comment
66,1461428,2024-02-04,Tokyo (JPN),860553,6:45,Tokyo Shimbun Hai (4yo+) (Turf),16,1,1,0.00,0.00,Sakura Toujours (JPN),
67,1461427,2024-02-04,Tokyo (JPN),860553,6:45,Tokyo Shimbun Hai (4yo+) (Turf),16,5,2,1.00,1.00,Win Carnelian (JPN),
68,1461598,2024-02-04,Tokyo (JPN),860553,6:45,Tokyo Shimbun Hai (4yo+) (Turf),16,8,3,1.25,0.30,Ho O Biscuits (JPN),
69,1461426,2024-02-04,Tokyo (JPN),860553,6:45,Tokyo Shimbun Hai (4yo+) (Turf),16,2,4,1.25,0.05,Ask Konnamonda (JPN),
70,1461425,2024-02-04,Tokyo (JPN),860553,6:45,Tokyo Shimbun Hai (4yo+) (Turf),16,11,5,2.00,0.75,Matenro Sky (JPN),Prominent early - soon dropped into midfield - pushed along to close over 2f out - challenged for places approaching...
71,1461424,2024-02-04,Tokyo (JPN),860553,6:45,Tokyo Shimbun Hai (4yo+) (Turf),16,6,6,2.50,0.30,Masked Diva (JPN),
72,1461423,2024-02-04,Tokyo (JPN),860553,6:45,Tokyo Shimbun Hai (4yo+) (Turf),16,16,7,3.25,0.75,Avverare (JPN),
73,1461422,2024-02-04,Tokyo (JPN),860553,6:45,Tokyo Shimbun Hai (4yo+) (Turf),16,4,8,3.75,0.50,Rouge Lignage (JPN),
74,1461421,2024-02-04,Tokyo (JPN),860553,6:45,Tokyo Shimbun Hai (4yo+) (Turf),16,9,9,3.75,0.20,Umbrail (JPN),
75,1461420,2024-02-04,Tokyo (JPN),860553,6:45,Tokyo Shimbun Hai (4yo+) (Turf),16,13,10,4.25,0.30,Tudo De Bom (JPN),



2026-03-13 — Meydan — 17:45 — Binghatti (Handicap) (Dirt)


,source_rowid,date,course,race_id,off,race_name,ran,num,pos,ovr_btn,btn,horse,comment
82,1813775,2026-03-13,Meydan,915628,17:45,Binghatti (Handicap) (Dirt),9,3,1,0.00,0.00,Diamond Dealer (USA),Prominent - led after 1f - joined 5f out - against far rail and clear when ridden over 1f out - went further clear i...
83,1813776,2026-03-13,Meydan,915628,17:45,Binghatti (Handicap) (Dirt),9,5,2,2.75,2.75,Nam Phrik (BRZ),Towards rear - hung left but steady headway 2f out - went second towards finish - no match for winner
84,1813777,2026-03-13,Meydan,915628,17:45,Binghatti (Handicap) (Dirt),9,6,3,3.00,0.30,Jolly Roger (IRE),In touch with leaders on outer early - soon midfield - headway on inner over 3f out - went second inside final furlo...
85,1813778,2026-03-13,Meydan,915628,17:45,Binghatti (Handicap) (Dirt),9,4,4,3.75,0.75,Gun Carriage (USA),Soon prominent - joined leader 5f out - lost position over 2f out - weakened over 1f out
86,1813779,2026-03-13,Meydan,915628,17:45,Binghatti (Handicap) (Dirt),9,8,5,4.75,1.00,Nyaar (USA),Reared in stalls before start - led - prominent after 1f - weakened 2f out
87,1813780,2026-03-13,Meydan,915628,17:45,Binghatti (Handicap) (Dirt),9,1,6,6.00,1.25,Shaq Diesel (USA),In touch with leaders - on outer when outpaced 3f out - weakened under 2f out
88,1813781,2026-03-13,Meydan,915628,17:45,Binghatti (Handicap) (Dirt),9,7,7,7.50,1.50,Laasudood (USA),Prominent early - soon in touch with leaders - outpaced 4f out - some headway over 2f out - weakened inside final fu...
89,1813782,2026-03-13,Meydan,915628,17:45,Binghatti (Handicap) (Dirt),9,9,9,15.50,8.00,Quality Humor (USA),Reared in stalls before start - in rear - soon switched right and on outer - weakened over 2f out
90,1813783,2026-03-13,Meydan,915628,17:45,Binghatti (Handicap) (Dirt),9,2,10,35.50,20.00,Local Dynasty (IRE),Never better than midfield


### Result and finishing fields — interim findings

#### `pos`

`pos` uses mixed SQLite storage:

- 1,756,674 integer rows;
- 94,611 text rows;
- no SQL `NULL` values;
- no blank text values.

Numeric positions range from `0` to `34`.

Eight rows contain `pos = 0`. They occur in only two international races and appear to represent unresolved or unavailable placings rather than a normal finishing position.

The text values are finishing-status codes:

- `PU`;
- `F`;
- `UR`;
- `BD`;
- `RR`;
- `DSQ`;
- `RO`;
- `SU`;
- `REF`;
- `CO`;
- `LFT`.

Most text-status rows have `-` in both distance fields. `DSQ` is the exception: all 619 disqualified runners retain numeric `ovr_btn` and `btn` values.

Numeric `pos` is not guaranteed to form a dense sequence from `1` to `ran`.

Ten rows have `pos > ran`, but whole-race inspection showed that:

- `ran` still matches the number of physical runner rows;
- the stored position sequence contains gaps;
- demotions and revised official placings can produce positions greater than the number of stored runners.

Therefore, `pos <= ran` and contiguous-position checks would be invalid universal quality rules.

#### `ovr_btn` and `btn`

Both fields use mixed numeric and text storage.

`ovr_btn` contains:

- integers;
- real numbers;
- 93,992 text values.

`btn` contains:

- integers;
- real numbers;
- the same 93,992 text values.

Every text row contains `-` in both fields. There are no rows where only one of the two fields contains `-`.

Numeric ranges are:

- `ovr_btn`: `0` to `331.5`;
- `btn`: `0` to `186`.

No negative values occur.

The observed structure is consistent with:

- `ovr_btn` representing a cumulative or overall distance;
- `btn` representing a local or adjacent finishing margin.

This interpretation remains provisional until confirmed against source documentation.

Most winners have zero in both distance fields. Exceptions are largely explained by promoted winners whose comments state that they originally finished behind another horse and were later awarded the race.

Likewise, many non-winners with zero overall distance were originally first past the post and later demoted or disqualified.

There are 421 rows where `ovr_btn < btn`:

- 360 differ by exactly `0.05`, commonly `0.25` versus `0.3`, and appear to reflect notation or rounding differences;
- 61 contain larger contradictions, concentrated among tailed-off, eased, reluctant, virtually pulled-up, or otherwise abnormal performances.

These 61 rows should be flagged for investigation, not automatically corrected.

#### Cleaning implications

- Preserve the raw `pos`, `ovr_btn`, and `btn` values.
- Derive separate numeric and status representations for `pos`.
- Treat `pos = 0` as a special unresolved state.
- Do not assume finishing positions are contiguous.
- Do not enforce `pos <= ran`.
- Convert `-` in the distance fields to an explicit missing or not-applicable state only in a derived layer.
- Retain a quality flag for material cases where `ovr_btn < btn`.
- Do not overwrite promoted, demoted, or disqualified official placings using the stored distance values.

In [51]:
# Identify the source fields that have not yet been profiled.
#
# This prevents us from relying on memory or assumptions about the remaining
# schema and gives us the exact next field group to examine.

profiled_fields = {
    # Race and event description
    "date",
    "course",
    "race_id",
    "off",
    "race_name",
    "type",
    "class",
    "pattern",
    "rating_band",
    "age_band",
    "sex_rest",
    "dist",
    "going",
    "ran",

    # Runner identity and demographics
    "num",
    "horse",
    "age",
    "sex",
    "jockey",
    "trainer",

    # Result and finishing information
    "pos",
    "ovr_btn",
    "btn",
}

remaining_source_fields = (
    source_data_dictionary.loc[
        ~source_data_dictionary["field_name"].isin(profiled_fields)
    ]
    .reset_index(drop=True)
)

print(
    "Remaining unprofiled source fields: "
    f"{len(remaining_source_fields)}"
)

display(remaining_source_fields)

Remaining unprofiled source fields: 14


,column_position,field_name,declared_sqlite_type,declared_not_null,declared_default,declared_primary_key
0,17,draw,INTEGER,0,None,0
1,23,wgt,TEXT,0,None,0
2,24,hg,TEXT,0,None,0
3,25,time,TEXT,0,None,0
4,26,sp,TEXT,0,None,0
5,29,prize,INTEGER,0,None,0
6,30,or,INTEGER,0,None,0
7,31,rpr,INTEGER,0,None,0
8,32,ts,INTEGER,0,None,0
9,33,sire,TEXT,0,None,0


In [52]:
# Profile the next source-quality group:
# draw, weight, headgear, race time, and starting price.
#
# This first pass records missingness, distinct counts, and SQLite storage
# classes without yet interpreting the field formats.

setup_market_fields = [
    "draw",
    "wgt",
    "hg",
    "time",
    "sp",
]

setup_market_profile_rows = []

for field in setup_market_fields:
    quoted_field = f'"{field}"'

    result = pd.read_sql_query(
        f"""
        SELECT
            COUNT(*) AS total_rows,

            SUM(
                CASE
                    WHEN {quoted_field} IS NULL
                    THEN 1 ELSE 0
                END
            ) AS null_count,

            SUM(
                CASE
                    WHEN typeof({quoted_field}) = 'text'
                     AND length(trim({quoted_field})) = 0
                    THEN 1 ELSE 0
                END
            ) AS blank_text_count,

            COUNT(DISTINCT {quoted_field})
                AS distinct_non_null_values,

            SUM(
                CASE
                    WHEN typeof({quoted_field}) = 'null'
                    THEN 1 ELSE 0
                END
            ) AS storage_null,

            SUM(
                CASE
                    WHEN typeof({quoted_field}) = 'integer'
                    THEN 1 ELSE 0
                END
            ) AS storage_integer,

            SUM(
                CASE
                    WHEN typeof({quoted_field}) = 'real'
                    THEN 1 ELSE 0
                END
            ) AS storage_real,

            SUM(
                CASE
                    WHEN typeof({quoted_field}) = 'text'
                    THEN 1 ELSE 0
                END
            ) AS storage_text,

            SUM(
                CASE
                    WHEN typeof({quoted_field}) = 'blob'
                    THEN 1 ELSE 0
                END
            ) AS storage_blob
        FROM data
        WHERE {DATA_ROW_PREDICATE}
        """,
        connection,
    ).iloc[0].to_dict()

    result["field_name"] = field

    result["declared_sqlite_type"] = (
        source_data_dictionary.loc[
            source_data_dictionary["field_name"] == field,
            "declared_sqlite_type",
        ].iloc[0]
    )

    setup_market_profile_rows.append(result)

setup_market_quality = pd.DataFrame(
    setup_market_profile_rows
)[
    [
        "field_name",
        "declared_sqlite_type",
        "total_rows",
        "null_count",
        "blank_text_count",
        "distinct_non_null_values",
        "storage_null",
        "storage_integer",
        "storage_real",
        "storage_text",
        "storage_blob",
    ]
]

display(setup_market_quality)

,field_name,declared_sqlite_type,total_rows,null_count,blank_text_count,distinct_non_null_values,storage_null,storage_integer,storage_real,storage_text,storage_blob
0,draw,INTEGER,1851285,0,592778,38,0,1258507,0,592778,0
1,wgt,TEXT,1851285,0,0,79,0,0,0,1851285,0
2,hg,TEXT,1851285,0,1122490,61,0,0,0,1851285,0
3,time,TEXT,1851285,0,0,46537,0,0,0,1851285,0
4,sp,TEXT,1851285,0,9097,843,0,0,0,1851285,0


In [53]:
# Inspect the stored-value distributions for draw, weight, headgear, time,
# and starting price.
#
# For low-cardinality fields, show all values.
# For time and starting price, show the most common values first because
# their distinct-value counts are much larger.

setup_market_value_limits = {
    "draw": None,
    "wgt": None,
    "hg": None,
    "time": 50,
    "sp": 100,
}

for field in setup_market_fields:
    quoted_field = f'"{field}"'
    row_limit = setup_market_value_limits[field]

    limit_sql = (
        ""
        if row_limit is None
        else f"LIMIT {row_limit}"
    )

    value_distribution = pd.read_sql_query(
        f"""
        SELECT
            CASE
                WHEN typeof({quoted_field}) = 'text'
                 AND length({quoted_field}) = 0
                THEN '<blank>'
                ELSE CAST({quoted_field} AS TEXT)
            END AS stored_value,

            typeof({quoted_field}) AS storage_class,

            COUNT(*) AS row_count,

            ROUND(
                100.0 * COUNT(*) /
                (
                    SELECT COUNT(*)
                    FROM data
                    WHERE {DATA_ROW_PREDICATE}
                ),
                4
            ) AS row_pct
        FROM data
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY
            {quoted_field},
            typeof({quoted_field})
        ORDER BY
            row_count DESC,
            stored_value
        {limit_sql}
        """,
        connection,
    )

    print(f"\n{field.upper()} — STORED VALUES")

    with pd.option_context(
        "display.max_rows",
        None,
        "display.max_colwidth",
        120,
    ):
        display(value_distribution)


DRAW — STORED VALUES


,stored_value,storage_class,row_count,row_pct
0,<blank>,text,592778,32.0198
1,3,integer,118714,6.4125
2,2,integer,118683,6.4108
3,1,integer,118603,6.4065
4,4,integer,118128,6.3809
5,5,integer,116584,6.2975
6,6,integer,112277,6.0648
7,7,integer,104480,5.6436
8,8,integer,94065,5.0811
9,9,integer,81576,4.4065



WGT — STORED VALUES


,stored_value,storage_class,row_count,row_pct
0,9-0,text,127748,6.9005
1,9-2,text,117269,6.3345
2,9-5,text,81374,4.3955
3,8-11,text,74459,4.0220
4,9-7,text,71569,3.8659
5,9-4,text,70493,3.8078
6,9-3,text,62768,3.3905
7,8-13,text,61389,3.3160
8,9-6,text,58025,3.1343
9,9-1,text,57427,3.1020



HG — STORED VALUES


,stored_value,storage_class,row_count,row_pct
0,<blank>,text,1122490,60.6330
1,t,text,178727,9.6542
2,p,text,173713,9.3834
3,b,text,128859,6.9605
4,h,text,62258,3.3630
5,tp,text,52073,2.8128
6,v,text,40585,2.1923
7,tb,text,40299,2.1768
8,ht,text,22190,1.1986
9,tv,text,9729,0.5255



TIME — STORED VALUES


,stored_value,storage_class,row_count,row_pct
0,-,text,94408,5.0996
1,1:13.75,text,374,0.0202
2,1:13.95,text,351,0.0190
3,1:13.30,text,350,0.0189
4,1:10.90,text,348,0.0188
5,1:14.00,text,347,0.0187
6,1:12.85,text,345,0.0186
7,1:13.90,text,342,0.0185
8,1:40.52,text,340,0.0184
9,1:13.60,text,338,0.0183



SP — STORED VALUES


,stored_value,storage_class,row_count,row_pct
0,12/1,text,72682,3.9260
1,16/1,text,72258,3.9031
2,14/1,text,66930,3.6153
3,25/1,text,64100,3.4625
4,10/1,text,63656,3.4385
5,33/1,text,63615,3.4363
6,20/1,text,61820,3.3393
7,8/1,text,56458,3.0497
8,9/1,text,49756,2.6876
9,50/1,text,48761,2.6339


In [54]:
# Validate the structural formats of draw, weight, time, and starting price.
#
# This identifies rows that do not match the dominant observed patterns:
# - draw: blank or positive integer;
# - wgt: stone-pounds text such as 9-0;
# - time: dash or M:SS.xx / MM:SS.xx;
# - sp: blank, Evens, or fractional odds with optional suffixes.
#
# The patterns are descriptive checks only. They do not alter source values.

format_validation_summary = pd.read_sql_query(
    f"""
    SELECT
        SUM(
            CASE
                WHEN typeof(draw) = 'text'
                 AND length(trim(draw)) = 0
                THEN 0
                WHEN typeof(draw) = 'integer'
                 AND draw >= 1
                THEN 0
                ELSE 1
            END
        ) AS invalid_draw_rows,

        SUM(
            CASE
                WHEN wgt GLOB '[0-9]-[0-9]'
                  OR wgt GLOB '[0-9]-[0-9][0-9]'
                  OR wgt GLOB '[0-9][0-9]-[0-9]'
                  OR wgt GLOB '[0-9][0-9]-[0-9][0-9]'
                THEN 0
                ELSE 1
            END
        ) AS invalid_weight_format_rows,

        SUM(
            CASE
                WHEN time = '-'
                THEN 0
                WHEN time GLOB '[0-9]:[0-9][0-9].[0-9][0-9]'
                  OR time GLOB '[0-9][0-9]:[0-9][0-9].[0-9][0-9]'
                THEN 0
                ELSE 1
            END
        ) AS invalid_time_format_rows,

        SUM(
            CASE
                WHEN length(trim(sp)) = 0
                THEN 0
                WHEN sp IN (
                    'Evens',
                    'EvensF',
                    'EvensJ',
                    'EvensJF'
                )
                THEN 0
                WHEN sp GLOB '[0-9]*/[0-9]*'
                  OR sp GLOB '[0-9]*/[0-9]*F'
                  OR sp GLOB '[0-9]*/[0-9]*J'
                  OR sp GLOB '[0-9]*/[0-9]*JF'
                THEN 0
                ELSE 1
            END
        ) AS invalid_sp_format_rows
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    """,
    connection,
)

weight_components = pd.read_sql_query(
    f"""
    SELECT
        MIN(CAST(substr(wgt, 1, instr(wgt, '-') - 1) AS INTEGER))
            AS minimum_stone,

        MAX(CAST(substr(wgt, 1, instr(wgt, '-') - 1) AS INTEGER))
            AS maximum_stone,

        MIN(CAST(substr(wgt, instr(wgt, '-') + 1) AS INTEGER))
            AS minimum_pounds_component,

        MAX(CAST(substr(wgt, instr(wgt, '-') + 1) AS INTEGER))
            AS maximum_pounds_component,

        SUM(
            CASE
                WHEN CAST(
                    substr(wgt, instr(wgt, '-') + 1)
                    AS INTEGER
                ) >= 14
                THEN 1 ELSE 0
            END
        ) AS pounds_component_14_or_more
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    """,
    connection,
)

display(format_validation_summary)
display(weight_components)

,invalid_draw_rows,invalid_weight_format_rows,invalid_time_format_rows,invalid_sp_format_rows
0,0,0,289743,182


,minimum_stone,maximum_stone,minimum_pounds_component,maximum_pounds_component,pounds_component_14_or_more
0,6,12,0,13,0


In [55]:
# Inspect values rejected by the provisional time and starting-price patterns.
#
# The earlier validation deliberately used narrow patterns. This cell shows
# the actual rejected values so that the accepted formats can be based on
# evidence rather than assumptions.

invalid_time_values = pd.read_sql_query(
    f"""
    SELECT
        time AS stored_time,
        COUNT(*) AS row_count
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND time <> '-'
      AND NOT (
            time GLOB '[0-9]:[0-9][0-9].[0-9][0-9]'
         OR time GLOB '[0-9][0-9]:[0-9][0-9].[0-9][0-9]'
      )
    GROUP BY time
    ORDER BY
        row_count DESC,
        stored_time
    """,
    connection,
)

invalid_sp_values = pd.read_sql_query(
    f"""
    SELECT
        CASE
            WHEN length(trim(sp)) = 0
            THEN '<blank>'
            ELSE sp
        END AS stored_sp,
        COUNT(*) AS row_count
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND length(trim(sp)) > 0
      AND sp NOT IN (
            'Evens',
            'EvensF',
            'EvensJ',
            'EvensJF'
      )
      AND NOT (
            sp GLOB '[0-9]*/[0-9]*'
         OR sp GLOB '[0-9]*/[0-9]*F'
         OR sp GLOB '[0-9]*/[0-9]*J'
         OR sp GLOB '[0-9]*/[0-9]*JF'
      )
    GROUP BY sp
    ORDER BY
        row_count DESC,
        stored_sp
    """,
    connection,
)

print(
    "Distinct time values rejected by provisional pattern: "
    f"{len(invalid_time_values):,}"
)
print(
    "Rows represented by rejected time values: "
    f"{invalid_time_values['row_count'].sum():,}"
)

with pd.option_context(
    "display.max_rows",
    150,
    "display.max_colwidth",
    120,
):
    display(invalid_time_values.head(150))

print(
    "\nDistinct SP values rejected by provisional pattern: "
    f"{len(invalid_sp_values):,}"
)
print(
    "Rows represented by rejected SP values: "
    f"{invalid_sp_values['row_count'].sum():,}"
)

with pd.option_context(
    "display.max_rows",
    None,
    "display.max_colwidth",
    120,
):
    display(invalid_sp_values)

Distinct time values rejected by provisional pattern: 7,127
Rows represented by rejected time values: 289,743


,stored_time,row_count
0,4:8.00,254
1,4:9.70,252
2,4:3.80,250
3,4:5.00,249
4,4:2.20,247
5,4:8.30,247
6,4:4.60,246
7,4:7.90,246
8,4:3.10,245
9,4:3.30,245



Distinct SP values rejected by provisional pattern: 3
Rows represented by rejected SP values: 182


,stored_sp,row_count
0,Evs,123
1,EvsJ,58
2,F,1


In [56]:
# Classify every distinct race-time string using Python regular expressions.
#
# SQLite GLOB was too rigid for this field because valid-looking times may use
# either one or two digits for the seconds component, for example:
#   1:9.98
#   1:13.75
#   4:8.00
#
# Also inspect the single starting-price value stored as "F".

import re

distinct_time_values = pd.read_sql_query(
    f"""
    SELECT
        time AS stored_time,
        COUNT(*) AS row_count
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY time
    ORDER BY row_count DESC, stored_time
    """,
    connection,
)

def classify_time_format(value):
    if value == "-":
        return "dash placeholder"

    if re.fullmatch(r"\d+:\d{1,2}\.\d{2}", value):
        return "minutes:seconds.hundredths"

    return "unclassified"

distinct_time_values["format_class"] = (
    distinct_time_values["stored_time"]
    .map(classify_time_format)
)

time_format_summary = (
    distinct_time_values
    .groupby("format_class", as_index=False)
    .agg(
        distinct_values=("stored_time", "count"),
        runner_rows=("row_count", "sum"),
    )
    .sort_values("runner_rows", ascending=False)
    .reset_index(drop=True)
)

unclassified_time_values = distinct_time_values.loc[
    distinct_time_values["format_class"] == "unclassified"
].copy()

single_f_sp_rows = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        date,
        course,
        race_id,
        off,
        race_name,
        num,
        pos,
        horse,
        sp,
        comment
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND sp = 'F'
    """,
    connection,
)

display(time_format_summary)

print(
    "Unclassified distinct time values: "
    f"{len(unclassified_time_values):,}"
)
print(
    "Rows represented by unclassified time values: "
    f"{unclassified_time_values['row_count'].sum():,}"
)

with pd.option_context(
    "display.max_rows",
    None,
    "display.max_colwidth",
    120,
):
    display(unclassified_time_values)
    display(single_f_sp_rows)

,format_class,distinct_values,runner_rows
0,minutes:seconds.hundredths,46536,1756877
1,dash placeholder,1,94408


Unclassified distinct time values: 0
Rows represented by unclassified time values: 0


,stored_time,row_count,format_class


,source_rowid,date,course,race_id,off,race_name,num,pos,horse,sp,comment
0,1708860,2025-07-20,Del Mar (USA),900160,1:03,Wickerr Stakes () (3yo+) (Turf),9,1,Almendares (GB),F,


In [57]:
# Test whether race-level time is repeated consistently across all runner rows.
#
# The source is runner-grain, but time appears to describe the whole race.
# This checks:
# - whether every apparent race has only one stored time value;
# - whether dash placeholders occur for the complete race or only some runners;
# - how many races have a usable time.

race_time_consistency = pd.read_sql_query(
    f"""
    WITH race_time_summary AS (
        SELECT
            date,
            course,
            off,
            race_name,

            COUNT(*) AS physical_rows,
            COUNT(DISTINCT time) AS distinct_time_values,

            SUM(
                CASE
                    WHEN time = '-'
                    THEN 1 ELSE 0
                END
            ) AS dash_rows,

            SUM(
                CASE
                    WHEN time <> '-'
                    THEN 1 ELSE 0
                END
            ) AS populated_time_rows
        FROM data
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY
            date,
            course,
            off,
            race_name
    )
    SELECT
        COUNT(*) AS apparent_races,

        SUM(
            CASE
                WHEN distinct_time_values = 1
                THEN 1 ELSE 0
            END
        ) AS races_with_one_time_value,

        SUM(
            CASE
                WHEN distinct_time_values > 1
                THEN 1 ELSE 0
            END
        ) AS races_with_multiple_time_values,

        SUM(
            CASE
                WHEN dash_rows = physical_rows
                THEN 1 ELSE 0
            END
        ) AS races_all_dash,

        SUM(
            CASE
                WHEN populated_time_rows = physical_rows
                THEN 1 ELSE 0
            END
        ) AS races_all_populated,

        SUM(
            CASE
                WHEN dash_rows > 0
                 AND populated_time_rows > 0
                THEN 1 ELSE 0
            END
        ) AS races_mixing_dash_and_time
    FROM race_time_summary
    """,
    connection,
)

mixed_race_time_examples = pd.read_sql_query(
    f"""
    SELECT
        date,
        course,
        off,
        race_name,
        COUNT(*) AS physical_rows,
        COUNT(DISTINCT time) AS distinct_time_values,
        GROUP_CONCAT(DISTINCT time) AS stored_time_values
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY
        date,
        course,
        off,
        race_name
    HAVING COUNT(DISTINCT time) > 1
    ORDER BY
        date,
        course,
        off,
        race_name
    LIMIT 50
    """,
    connection,
)

display(race_time_consistency)

print(
    "Examples of races with multiple stored time values: "
    f"{len(mixed_race_time_examples):,}"
)
display(mixed_race_time_examples)

,apparent_races,races_with_one_time_value,races_with_multiple_time_values,races_all_dash,races_all_populated,races_mixing_dash_and_time
0,189043,77,188966,66,145607,43370


Examples of races with multiple stored time values: 50


,date,course,off,race_name,physical_rows,distinct_time_values,stored_time_values
0,2015-01-01,Aqueduct (USA),6:20,Affectionately Stakes () (4yo+ Fillies & Mares...,8,8,"1:44.68,1:44.69,1:45.28,1:45.38,1:45.83,1:46.0..."
1,2015-01-01,Ascot (AUS),6:50,La Trice Classic (Fillies & Mares),8,8,"1:53.01,1:52.60,1:52.51,1:51.81,1:51.91,1:52.1..."
2,2015-01-01,Ascot (AUS),8:50,Golden River Development Perth Cup (Handicap),16,16,"2:30.24,2:30.44,2:30.74,2:31.19,2:31.20,2:31.3..."
3,2015-01-01,Catterick,12:30,Happy New Year Novices Hurdle,10,10,"4:50.90,4:51.20,4:51.50,4:51.60,4:52.60,4:55.0..."
4,2015-01-01,Catterick,1:05,Buy Your 2015 Annual Badge Today Handicap Hurdle,11,11,"-,4:58.35,4:55.55,4:52.05,4:46.70,4:47.70,4:48..."
5,2015-01-01,Catterick,1:40,Dine And View At Catterick Races Novices Chase,5,4,"6:39.90,6:41.10,6:43.70,-"
6,2015-01-01,Catterick,2:15,Racing Again On 8th January Novices Hurdle,3,3,"6:39.40,6:39.85,6:44.25"
7,2015-01-01,Catterick,2:50,Watch On 3 Devices racinguk.com/anywhere Handi...,6,6,"4:51.60,4:51.90,4:52.90,4:55.30,5:0.50,-"
8,2015-01-01,Catterick,3:25,yorkshire-outdoors.co.uk Handicap Hurdle,11,8,"6:33.60,6:34.80,6:39.20,6:44.20,6:44.95,6:45.1..."
9,2015-01-01,Cheltenham,12:10,Neptune Investment Management Novices Hurdle,8,8,"5:7.20,5:8.90,5:9.25,5:9.40,5:9.70,5:9.85,5:25..."


In [58]:
# Test the runner-level interpretation of time.
#
# Convert each parseable time string to total seconds, then compare it with:
# - finishing position;
# - overall distance behind;
# - completion status.
#
# This does not yet assume that the fastest stored time is always the official
# winner because promoted and demoted runners may retain their original times.

runner_time_sample = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        date,
        course,
        off,
        race_name,
        pos,
        ovr_btn,
        btn,
        time,
        horse
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    """,
    connection,
)

def time_to_seconds(value):
    if value == "-":
        return pd.NA

    minutes_text, seconds_text = value.split(":")
    return (
        int(minutes_text) * 60
        + float(seconds_text)
    )

runner_time_sample["time_seconds"] = (
    runner_time_sample["time"]
    .map(time_to_seconds)
    .astype("Float64")
)

runner_time_sample["numeric_position"] = pd.to_numeric(
    runner_time_sample["pos"],
    errors="coerce",
)

runner_time_relationship_summary = pd.DataFrame(
    {
        "measure": [
            "rows with parseable time",
            "rows with dash time",
            "numeric finishers with parseable time",
            "text-status runners with parseable time",
            "correlation: numeric position vs time seconds",
            "correlation: numeric ovr_btn vs time seconds",
        ],
        "value": [
            runner_time_sample["time_seconds"].notna().sum(),
            runner_time_sample["time_seconds"].isna().sum(),
            (
                runner_time_sample["numeric_position"].notna()
                & runner_time_sample["time_seconds"].notna()
            ).sum(),
            (
                runner_time_sample["numeric_position"].isna()
                & runner_time_sample["time_seconds"].notna()
            ).sum(),
            runner_time_sample[
                ["numeric_position", "time_seconds"]
            ].corr().iloc[0, 1],
            runner_time_sample.assign(
                numeric_ovr_btn=pd.to_numeric(
                    runner_time_sample["ovr_btn"],
                    errors="coerce",
                )
            )[["numeric_ovr_btn", "time_seconds"]]
            .corr()
            .iloc[0, 1],
        ],
    }
)

race_time_order_checks = (
    runner_time_sample
    .dropna(subset=["time_seconds", "numeric_position"])
    .groupby(
        ["date", "course", "off", "race_name"],
        as_index=False,
    )
    .agg(
        timed_runners=("source_rowid", "count"),
        minimum_position=("numeric_position", "min"),
        maximum_position=("numeric_position", "max"),
        fastest_time=("time_seconds", "min"),
        slowest_time=("time_seconds", "max"),
    )
)

print(
    "Apparent races containing at least two timed numeric finishers: "
    f"{(race_time_order_checks['timed_runners'] >= 2).sum():,}"
)

display(runner_time_relationship_summary)
display(race_time_order_checks.head(30))

Apparent races containing at least two timed numeric finishers: 188,897


,measure,value
0,rows with parseable time,1.756877e+06
1,rows with dash time,9.440800e+04
2,numeric finishers with parseable time,1.756258e+06
3,text-status runners with parseable time,6.190000e+02
4,correlation: numeric position vs time seconds,-9.200331e-02
5,correlation: numeric ovr_btn vs time seconds,4.044796e-01


,date,course,off,race_name,timed_runners,minimum_position,maximum_position,fastest_time,slowest_time
0,2015-01-01,Aqueduct (USA),6:20,Affectionately Stakes () (4yo+ Fillies & Mares...,8,1.0,8.0,104.68,109.38
1,2015-01-01,Ascot (AUS),6:50,La Trice Classic (Fillies & Mares),8,1.0,8.0,111.81,113.01
2,2015-01-01,Ascot (AUS),8:50,Golden River Development Perth Cup (Handicap),16,1.0,16.0,149.94,153.19
3,2015-01-01,Catterick,12:30,Happy New Year Novices Hurdle,9,1.0,9.0,290.9,296.25
4,2015-01-01,Catterick,1:05,Buy Your 2015 Annual Badge Today Handicap Hurdle,10,1.0,10.0,286.7,298.35
5,2015-01-01,Catterick,1:40,Dine And View At Catterick Races Novices Chase,3,1.0,3.0,399.9,403.7
6,2015-01-01,Catterick,2:15,Racing Again On 8th January Novices Hurdle,3,1.0,3.0,399.4,404.25
7,2015-01-01,Catterick,2:50,Watch On 3 Devices racinguk.com/anywhere Handi...,5,1.0,5.0,291.6,300.5
8,2015-01-01,Catterick,3:25,yorkshire-outdoors.co.uk Handicap Hurdle,7,1.0,7.0,393.6,409.75
9,2015-01-01,Cheltenham,12:10,Neptune Investment Management Novices Hurdle,8,1.0,8.0,307.2,325.65


In [59]:
# Test whether runner times are ordered sensibly within each race.
#
# Comparing raw time with position across the entire dataset is misleading
# because race distances vary greatly. Instead, compare each runner with the
# fastest recorded time in the same apparent race.
#
# For ordinary numeric finishers, elapsed time behind the race minimum should
# usually be non-negative and should generally increase with finishing position.

timed_numeric_finishers = (
    runner_time_sample
    .dropna(
        subset=[
            "time_seconds",
            "numeric_position",
        ]
    )
    .copy()
)

timed_numeric_finishers["race_minimum_time_seconds"] = (
    timed_numeric_finishers
    .groupby(
        [
            "date",
            "course",
            "off",
            "race_name",
        ]
    )["time_seconds"]
    .transform("min")
)

timed_numeric_finishers["seconds_behind_fastest"] = (
    timed_numeric_finishers["time_seconds"]
    - timed_numeric_finishers["race_minimum_time_seconds"]
)

within_race_time_summary = pd.DataFrame(
    {
        "measure": [
            "timed numeric finisher rows",
            "rows faster than race minimum",
            "position 1 rows that are fastest timed runner",
            "position 1 rows not fastest timed runner",
            "non-winners tied for fastest time",
            "correlation: position vs seconds behind fastest",
            "correlation: ovr_btn vs seconds behind fastest",
        ],
        "value": [
            len(timed_numeric_finishers),

            (
                timed_numeric_finishers["seconds_behind_fastest"]
                < 0
            ).sum(),

            (
                (timed_numeric_finishers["numeric_position"] == 1)
                & (
                    timed_numeric_finishers["seconds_behind_fastest"]
                    == 0
                )
            ).sum(),

            (
                (timed_numeric_finishers["numeric_position"] == 1)
                & (
                    timed_numeric_finishers["seconds_behind_fastest"]
                    > 0
                )
            ).sum(),

            (
                (timed_numeric_finishers["numeric_position"] > 1)
                & (
                    timed_numeric_finishers["seconds_behind_fastest"]
                    == 0
                )
            ).sum(),

            timed_numeric_finishers[
                [
                    "numeric_position",
                    "seconds_behind_fastest",
                ]
            ].corr().iloc[0, 1],

            timed_numeric_finishers.assign(
                numeric_ovr_btn=pd.to_numeric(
                    timed_numeric_finishers["ovr_btn"],
                    errors="coerce",
                )
            )[
                [
                    "numeric_ovr_btn",
                    "seconds_behind_fastest",
                ]
            ].corr().iloc[0, 1],
        ],
    }
)

winner_not_fastest_examples = (
    timed_numeric_finishers.loc[
        (timed_numeric_finishers["numeric_position"] == 1)
        & (
            timed_numeric_finishers["seconds_behind_fastest"]
            > 0
        ),
        [
            "source_rowid",
            "date",
            "course",
            "off",
            "race_name",
            "horse",
            "pos",
            "ovr_btn",
            "btn",
            "time",
            "time_seconds",
            "race_minimum_time_seconds",
            "seconds_behind_fastest",
        ],
    ]
    .sort_values(
        [
            "seconds_behind_fastest",
            "date",
            "course",
            "off",
        ],
        ascending=[
            False,
            True,
            True,
            True,
        ],
    )
)

display(within_race_time_summary)

print(
    "Examples where official position 1 is not the fastest timed runner: "
    f"{len(winner_not_fastest_examples):,}"
)

with pd.option_context(
    "display.max_rows",
    50,
    "display.max_colwidth",
    120,
):
    display(winner_not_fastest_examples.head(50))

,measure,value
0,timed numeric finisher rows,1.756258e+06
1,rows faster than race minimum,0.000000e+00
2,position 1 rows that are fastest timed runner,1.889510e+05
3,position 1 rows not fastest timed runner,3.270000e+02
4,non-winners tied for fastest time,3.710000e+02
5,correlation: position vs seconds behind fastest,4.602975e-01
6,correlation: ovr_btn vs seconds behind fastest,9.988109e-01


Examples where official position 1 is not the fastest timed runner: 327


,source_rowid,date,course,off,race_name,horse,pos,ovr_btn,btn,time,time_seconds,race_minimum_time_seconds,seconds_behind_fastest
727219,727221,2019-06-08,Gavea (BRZ),8:00,Grande Premio Major Suckow (2yo+) (Turf),Happy Bryan (BRZ),1,4.5,4.5,0:57.51,57.51,56.61,0.9
1752289,1752291,2025-10-19,Leopardstown,16:35,Bahrain Turf Club October Handicap (Premier Handicap),Blake (GB),1,3.25,3.25,2:47.82,167.82,167.17,0.65
1743583,1743585,2025-10-03,Keeneland (USA),10:16,Darley Alcibiades Stakes (2yo Fillies) (Main Track) (Dirt),Tommy Jo (USA),1,2.75,2.75,1:45.24,105.24,104.69,0.55
16968,16970,2015-02-21,Gulfstream Park (USA),10:30,Besilu Stables Fountain Of Youth Stakes (3yo) (Dirt),Itsaknockout (USA),1,2.75,2.75,1:46.83,106.83,106.28,0.55
970842,970844,2021-02-27,Gulfstream Park (USA),5:26,Herecomesthebride Stakes (3yo Fillies) (Turf),Con Lima (USA),1,2.75,2.75,1:41.85,101.85,101.3,0.55
13535,13537,2015-02-13,Chantilly (FR),4:25,Prix du Chemin Royal (Handicap) (5yo+) (Polytrack),El Nino (FR),1,2.5,2.5,1:56.66,116.66,116.16,0.5
871742,871744,2020-08-08,Saratoga (USA),9:22,Troy Stakes (4yo+) (Mellon Course) (Turf),American Sailor (USA),1,2.25,2.25,1:1.72,61.72,61.27,0.45
116817,116819,2015-09-24,Maisons-Laffitte (FR),3:00,Prix de Carrieres-sous-Poissy (Claimer) (4yo+) (Round) (Turf),Key To Fun (GER),1,2,2,2:11.12,131.12,130.72,0.4
787181,787183,2019-10-05,Woodbine (CAN),9:59,Mazarine Stakes (2yo Fillies) (All-Weather Track) (Tapeta),Curlins Voyage (CAN),1,2,2,1:43.70,103.7,103.3,0.4
855117,855119,2020-04-11,Gulfstream Park (USA),8:52,Maiden Special Weight (Maiden) (3yo+) (Main Track) (Dirt),Colonel Liam (USA),1,2,2,1:38.31,98.31,97.91,0.4


In [60]:
# Quantify the relationship between elapsed-time difference and ovr_btn.
#
# The near-perfect correlation suggests that runner times may be derived from,
# or closely linked to, the stored distance-behind values. However, the
# conversion from lengths to seconds may vary by race code and race distance.
#
# This cell calculates seconds per recorded length for ordinary positive
# distance values, then profiles the ratio by race type.

time_distance_relationship = (
    timed_numeric_finishers
    .assign(
        numeric_ovr_btn=pd.to_numeric(
            timed_numeric_finishers["ovr_btn"],
            errors="coerce",
        )
    )
    .loc[
        lambda frame:
            (frame["numeric_ovr_btn"] > 0)
            & (frame["seconds_behind_fastest"] > 0)
    ]
    .copy()
)

time_distance_relationship["seconds_per_length"] = (
    time_distance_relationship["seconds_behind_fastest"]
    / time_distance_relationship["numeric_ovr_btn"]
)

# Add race type from the source using the source row identifier.
row_type_lookup = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        type
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    """,
    connection,
)

time_distance_relationship = (
    time_distance_relationship
    .merge(
        row_type_lookup,
        on="source_rowid",
        how="left",
        validate="one_to_one",
    )
)

time_distance_overall = pd.DataFrame(
    {
        "measure": [
            "eligible runner rows",
            "minimum seconds per length",
            "median seconds per length",
            "mean seconds per length",
            "maximum seconds per length",
        ],
        "value": [
            len(time_distance_relationship),
            time_distance_relationship["seconds_per_length"].min(),
            time_distance_relationship["seconds_per_length"].median(),
            time_distance_relationship["seconds_per_length"].mean(),
            time_distance_relationship["seconds_per_length"].max(),
        ],
    }
)

time_distance_by_type = (
    time_distance_relationship
    .groupby("type", as_index=False)
    .agg(
        runner_rows=("source_rowid", "count"),
        median_seconds_per_length=(
            "seconds_per_length",
            "median",
        ),
        mean_seconds_per_length=(
            "seconds_per_length",
            "mean",
        ),
        minimum_seconds_per_length=(
            "seconds_per_length",
            "min",
        ),
        maximum_seconds_per_length=(
            "seconds_per_length",
            "max",
        ),
    )
    .sort_values("runner_rows", ascending=False)
    .reset_index(drop=True)
)

display(time_distance_overall)
display(time_distance_by_type)

,measure,value
0,eligible runner rows,1.566928e+06
1,minimum seconds per length,5.714286e-03
2,median seconds per length,2.000000e-01
3,mean seconds per length,1.999497e-01
4,maximum seconds per length,3.000000e-01


,type,runner_rows,median_seconds_per_length,mean_seconds_per_length,minimum_seconds_per_length,maximum_seconds_per_length
0,Flat,1136957,0.2,0.19931,0.01,0.232
1,Hurdle,273704,0.2,0.20172,0.005714,0.3
2,Chase,116927,0.2,0.201539,0.036364,0.3
3,NH Flat,39340,0.2,0.201404,0.036364,0.3


In [61]:
# Test whether runner time is mathematically reconstructed from:
#
#     fastest race time + (ovr_btn * 0.2 seconds)
#
# Small differences may arise from rounding to hundredths of a second.
# This cell measures the reconstruction error without modifying the source.

time_distance_relationship["expected_time_seconds"] = (
    time_distance_relationship["race_minimum_time_seconds"]
    + (time_distance_relationship["numeric_ovr_btn"] * 0.2)
)

time_distance_relationship["time_reconstruction_error"] = (
    time_distance_relationship["time_seconds"]
    - time_distance_relationship["expected_time_seconds"]
)

time_reconstruction_summary = pd.DataFrame(
    {
        "measure": [
            "eligible runner rows",
            "exact reconstruction",
            "absolute error <= 0.01 seconds",
            "absolute error <= 0.02 seconds",
            "absolute error <= 0.05 seconds",
            "absolute error > 0.05 seconds",
            "median absolute error",
            "mean absolute error",
            "maximum absolute error",
        ],
        "value": [
            len(time_distance_relationship),

            (
                time_distance_relationship[
                    "time_reconstruction_error"
                ].abs()
                < 1e-9
            ).sum(),

            (
                time_distance_relationship[
                    "time_reconstruction_error"
                ].abs()
                <= 0.01
            ).sum(),

            (
                time_distance_relationship[
                    "time_reconstruction_error"
                ].abs()
                <= 0.02
            ).sum(),

            (
                time_distance_relationship[
                    "time_reconstruction_error"
                ].abs()
                <= 0.05
            ).sum(),

            (
                time_distance_relationship[
                    "time_reconstruction_error"
                ].abs()
                > 0.05
            ).sum(),

            time_distance_relationship[
                "time_reconstruction_error"
            ].abs().median(),

            time_distance_relationship[
                "time_reconstruction_error"
            ].abs().mean(),

            time_distance_relationship[
                "time_reconstruction_error"
            ].abs().max(),
        ],
    }
)

material_time_reconstruction_errors = (
    time_distance_relationship.loc[
        time_distance_relationship[
            "time_reconstruction_error"
        ].abs() > 0.05,
        [
            "source_rowid",
            "date",
            "course",
            "off",
            "race_name",
            "type",
            "horse",
            "pos",
            "numeric_ovr_btn",
            "time",
            "time_seconds",
            "race_minimum_time_seconds",
            "expected_time_seconds",
            "time_reconstruction_error",
        ],
    ]
    .sort_values(
        "time_reconstruction_error",
        key=lambda series: series.abs(),
        ascending=False,
    )
)

display(time_reconstruction_summary)

print(
    "Rows with absolute reconstruction error above 0.05 seconds: "
    f"{len(material_time_reconstruction_errors):,}"
)

with pd.option_context(
    "display.max_rows",
    100,
    "display.max_colwidth",
    120,
):
    display(material_time_reconstruction_errors.head(100))

,measure,value
0,eligible runner rows,1.566928e+06
1,exact reconstruction,1.366085e+06
2,absolute error <= 0.01 seconds,1.378966e+06
3,absolute error <= 0.02 seconds,1.418335e+06
4,absolute error <= 0.05 seconds,1.509409e+06
5,absolute error > 0.05 seconds,5.751900e+04
6,median absolute error,0.000000e+00
7,mean absolute error,2.212620e-02
8,maximum absolute error,1.561000e+01


Rows with absolute reconstruction error above 0.05 seconds: 57,519


,source_rowid,date,course,off,race_name,type,horse,pos,numeric_ovr_btn,time,time_seconds,race_minimum_time_seconds,expected_time_seconds,time_reconstruction_error
1516227,1790503,2026-01-12,Punchestown,15:48,Irish Stallion Farms EBF Mares INH Flat Race,NH Flat,Smart Answer (IRE),13,312.25,5:35.16,335.16,257.1,319.55,15.61
1518859,1793722,2026-01-21,Catterick,15:50,Racing Again 30th January Junior National Hunt Flat Race (GBB Race),NH Flat,Cribbins (CAN),9,268.75,4:59.19,299.19,232.0,285.75,13.44
1541904,1821608,2026-04-01,Compiegne,10:51,Prix Sicie (Claiming Hurdle) (Turf),Hurdle,Pegase Bailleuil (FR),7,251.25,5:14.87,314.87,252.06,302.31,12.56
1549217,1830435,2026-04-18,Bellewstown,17:35,Irish Stallion Farms EBF Auction (Pro/Am) INH Flat Race (IRE Incentive Race),NH Flat,Deep Manhattan (IRE),15,228.50,5:08.32,308.32,251.2,296.9,11.42
1516226,1790502,2026-01-12,Punchestown,15:48,Irish Stallion Farms EBF Mares INH Flat Race,NH Flat,Aint Over Yet (IRE),12,223.25,5:12.91,312.91,257.1,301.75,11.16
1494087,1763977,2025-11-09,Naas,16:10,Irish Stallion Farms EBF Mares (Pro/Am) INH Flat Race,NH Flat,Mighty Moments (GB),11,217.50,5:00.57,300.57,246.2,289.7,10.87
1496683,1767144,2025-11-18,Limerick,15:35,Limerick (Ladies Pro/Am) INH Flat Race,NH Flat,Abelene (IRE),7,213.75,5:24.84,324.84,271.4,314.15,10.69
1549216,1830434,2026-04-18,Bellewstown,17:35,Irish Stallion Farms EBF Auction (Pro/Am) INH Flat Race (IRE Incentive Race),NH Flat,Billy Bob (IRE),14,207.50,5:03.07,303.07,251.2,292.7,10.37
1510460,1783618,2025-12-26,Limerick,12:58,Signsplus Maiden Hurdle,Hurdle,Lucky Crystal (IRE),17,200.50,5:06.12,306.12,256.0,296.1,10.02
1518621,1793414,2026-01-20,Down Royal,12:50,Molson Coors Beverage Company Maiden Hurdle,Hurdle,Six Mile Suzy (IRE),18,193.50,5:15.77,315.77,267.4,306.1,9.67


In [62]:
# Profile the performance-measure fields:
# prize, official rating, Racing Post Rating, and Topspeed.
#
# This first pass records missingness, distinct counts, and SQLite storage
# classes without yet interpreting the values.

performance_fields = [
    "prize",
    "or",
    "rpr",
    "ts",
]

performance_profile_rows = []

for field in performance_fields:
    quoted_field = f'"{field}"'

    result = pd.read_sql_query(
        f"""
        SELECT
            COUNT(*) AS total_rows,

            SUM(
                CASE
                    WHEN {quoted_field} IS NULL
                    THEN 1 ELSE 0
                END
            ) AS null_count,

            SUM(
                CASE
                    WHEN typeof({quoted_field}) = 'text'
                     AND length(trim({quoted_field})) = 0
                    THEN 1 ELSE 0
                END
            ) AS blank_text_count,

            COUNT(DISTINCT {quoted_field})
                AS distinct_non_null_values,

            SUM(
                CASE
                    WHEN typeof({quoted_field}) = 'null'
                    THEN 1 ELSE 0
                END
            ) AS storage_null,

            SUM(
                CASE
                    WHEN typeof({quoted_field}) = 'integer'
                    THEN 1 ELSE 0
                END
            ) AS storage_integer,

            SUM(
                CASE
                    WHEN typeof({quoted_field}) = 'real'
                    THEN 1 ELSE 0
                END
            ) AS storage_real,

            SUM(
                CASE
                    WHEN typeof({quoted_field}) = 'text'
                    THEN 1 ELSE 0
                END
            ) AS storage_text,

            SUM(
                CASE
                    WHEN typeof({quoted_field}) = 'blob'
                    THEN 1 ELSE 0
                END
            ) AS storage_blob
        FROM data
        WHERE {DATA_ROW_PREDICATE}
        """,
        connection,
    ).iloc[0].to_dict()

    result["field_name"] = field

    result["declared_sqlite_type"] = (
        source_data_dictionary.loc[
            source_data_dictionary["field_name"] == field,
            "declared_sqlite_type",
        ].iloc[0]
    )

    performance_profile_rows.append(result)

performance_quality = pd.DataFrame(
    performance_profile_rows
)[
    [
        "field_name",
        "declared_sqlite_type",
        "total_rows",
        "null_count",
        "blank_text_count",
        "distinct_non_null_values",
        "storage_null",
        "storage_integer",
        "storage_real",
        "storage_text",
        "storage_blob",
    ]
]

display(performance_quality)

,field_name,declared_sqlite_type,total_rows,null_count,blank_text_count,distinct_non_null_values,storage_null,storage_integer,storage_real,storage_text,storage_blob
0,prize,INTEGER,1851285,0,839715,47215,0,225078,618026,1008181,0
1,or,INTEGER,1851285,0,0,182,0,1116633,0,734652,0
2,rpr,INTEGER,1851285,0,0,186,0,1644176,0,207109,0
3,ts,INTEGER,1851285,0,0,179,0,1227384,0,623901,0


In [63]:
# Inspect text-coded values and numeric ranges for the performance fields.
#
# This separates:
# - blank prize values;
# - any non-blank text values;
# - numeric minima and maxima.
#
# No interpretation or cleaning is applied yet.

performance_text_distributions = {}

for field in performance_fields:
    quoted_field = f'"{field}"'

    text_values = pd.read_sql_query(
        f"""
        SELECT
            CASE
                WHEN length(trim({quoted_field})) = 0
                THEN '<blank>'
                ELSE {quoted_field}
            END AS stored_value,

            COUNT(*) AS row_count
        FROM data
        WHERE {DATA_ROW_PREDICATE}
          AND typeof({quoted_field}) = 'text'
        GROUP BY {quoted_field}
        ORDER BY
            row_count DESC,
            stored_value
        """,
        connection,
    )

    performance_text_distributions[field] = text_values

performance_numeric_ranges = pd.read_sql_query(
    f"""
    SELECT
        MIN(
            CASE
                WHEN typeof(prize) IN ('integer', 'real')
                THEN prize
            END
        ) AS minimum_prize,

        MAX(
            CASE
                WHEN typeof(prize) IN ('integer', 'real')
                THEN prize
            END
        ) AS maximum_prize,

        SUM(
            CASE
                WHEN typeof(prize) IN ('integer', 'real')
                 AND prize < 0
                THEN 1 ELSE 0
            END
        ) AS negative_prize_rows,

        MIN(
            CASE
                WHEN typeof("or") = 'integer'
                THEN "or"
            END
        ) AS minimum_or,

        MAX(
            CASE
                WHEN typeof("or") = 'integer'
                THEN "or"
            END
        ) AS maximum_or,

        MIN(
            CASE
                WHEN typeof(rpr) = 'integer'
                THEN rpr
            END
        ) AS minimum_rpr,

        MAX(
            CASE
                WHEN typeof(rpr) = 'integer'
                THEN rpr
            END
        ) AS maximum_rpr,

        MIN(
            CASE
                WHEN typeof(ts) = 'integer'
                THEN ts
            END
        ) AS minimum_ts,

        MAX(
            CASE
                WHEN typeof(ts) = 'integer'
                THEN ts
            END
        ) AS maximum_ts
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    """,
    connection,
)

for field in performance_fields:
    print(f"\n{field.upper()} — TEXT VALUES")

    with pd.option_context(
        "display.max_rows",
        None,
        "display.max_colwidth",
        120,
    ):
        display(performance_text_distributions[field])

print("\nNUMERIC RANGES")
display(performance_numeric_ranges)


PRIZE — TEXT VALUES


,stored_value,row_count
0,<blank>,839715
1,€400,4485
2,€200,4067
3,€900,3860
4,€1900,3464
5,€100,3120
6,€5900,3061
7,€440,2288
8,€420,1828
9,€220,1815



OR — TEXT VALUES


,stored_value,row_count
0,–,734652



RPR — TEXT VALUES


,stored_value,row_count
0,–,207109



TS — TEXT VALUES


,stored_value,row_count
0,–,623901



NUMERIC RANGES


,minimum_prize,maximum_prize,negative_prize_rows,minimum_or,maximum_or,minimum_rpr,maximum_rpr,minimum_ts,maximum_ts
0,0.12,8000000,0,1,181,1,775,1,178


In [64]:
# Summarise prize storage conventions and inspect extreme rating values.
#
# The aim is to identify:
# - how euro-denominated prizes differ from numeric prize storage;
# - whether other currency symbols occur;
# - the rows behind unusually large rating values.
#
# This is a targeted quality check, not a full prize-money analysis.

prize_storage_summary = pd.read_sql_query(
    f"""
    SELECT
        CASE
            WHEN typeof(prize) = 'text'
             AND length(trim(prize)) = 0
            THEN 'blank text'

            WHEN typeof(prize) = 'text'
             AND prize LIKE '€%'
            THEN 'euro text'

            WHEN typeof(prize) = 'text'
            THEN 'other text'

            WHEN typeof(prize) = 'integer'
            THEN 'integer numeric'

            WHEN typeof(prize) = 'real'
            THEN 'real numeric'

            ELSE typeof(prize)
        END AS prize_storage_group,

        COUNT(*) AS row_count
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY prize_storage_group
    ORDER BY row_count DESC
    """,
    connection,
)

other_prize_text_values = pd.read_sql_query(
    f"""
    SELECT
        prize AS stored_prize,
        COUNT(*) AS row_count
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND typeof(prize) = 'text'
      AND length(trim(prize)) > 0
      AND prize NOT LIKE '€%'
    GROUP BY prize
    ORDER BY row_count DESC, stored_prize
    """,
    connection,
)

extreme_rating_rows = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        date,
        course,
        race_id,
        off,
        race_name,
        horse,
        pos,
        "or",
        rpr,
        ts,
        comment
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND (
            (typeof("or") = 'integer' AND "or" > 180)
         OR (typeof(rpr) = 'integer' AND rpr > 200)
         OR (typeof(ts) = 'integer' AND ts > 180)
      )
    ORDER BY
        rpr DESC,
        "or" DESC,
        ts DESC,
        date,
        course,
        off
    """,
    connection,
)

display(prize_storage_summary)

print(
    "Distinct non-euro, non-blank text prize values: "
    f"{len(other_prize_text_values):,}"
)
display(other_prize_text_values)

print(
    "Rows with extreme rating values: "
    f"{len(extreme_rating_rows):,}"
)

with pd.option_context(
    "display.max_rows",
    None,
    "display.max_columns",
    None,
    "display.max_colwidth",
    120,
):
    display(extreme_rating_rows)

,prize_storage_group,row_count
0,blank text,839715
1,real numeric,618026
2,integer numeric,225078
3,euro text,168466


Distinct non-euro, non-blank text prize values: 0


,stored_prize,row_count


Rows with extreme rating values: 3


,source_rowid,date,course,race_id,off,race_name,horse,pos,or,rpr,ts,comment
0,1619851,2025-01-03,Deauville (FR),885653,4:27,Prix de la Seulette (Handicap) (4yo) (All-Weather Track) (Polytrack),Si Capo Si (FR),1,–,775,50,
1,1814076,2026-03-14,Auteuil,915705,16:35,Prix Hubert de Navailles (Conditions Hurdle) (Turf),Theleme (FR),1,181,136,–,
2,1830913,2026-04-19,Auteuil,918364,15:28,Prix Leon Rambaud (Hurdle) (Turf),Theleme (FR),3,181,136,–,


## Performance and prize fields: findings

### `prize`

- The field contains no SQL `NULL` values.
- Missing values are stored as blank text.
- Populated values use two storage conventions:
  - numeric SQLite values for non-euro prize amounts;
  - text beginning with `€` for euro-denominated prize amounts.
- Euro values may contain:
  - decimal amounts;
  - thousands separators.
- No other non-blank text formats were found.
- Currency must therefore be preserved or derived separately rather than
  treating `prize` as one currency-neutral numeric field.

### `or`

- Official ratings are stored as integers when available.
- Missing values are represented by the en dash `–`.
- Numeric values range from 1 to 181.
- The maximum value of 181 appears twice for the same high-class hurdle horse
  and is not treated as an obvious source error.

### `rpr`

- Racing Post Ratings are stored as integers when available.
- Missing values are represented by the en dash `–`.
- Almost all values are within a plausible range.
- One row contains `rpr = 775`, which is an obvious isolated source anomaly and
  should be retained in the raw layer but flagged during transformation.

### `ts`

- Topspeed ratings are stored as integers when available.
- Missing values are represented by the en dash `–`.
- Numeric values range from 1 to 178.
- No obvious extreme anomalies were identified.

### Transformation implications

The raw values should remain unchanged. A later transformation layer should:

1. convert `–` to missing for `or`, `rpr`, and `ts`;
2. cast valid ratings to nullable integers;
3. flag implausible rating outliers such as `rpr = 775`;
4. parse `prize` into separate numeric amount and currency fields;
5. preserve the original `prize` value for auditability.

In [65]:
# Profile the remaining pedigree, ownership, and narrative fields.
#
# These are expected to be text-heavy fields. This first pass checks:
# - SQL NULLs;
# - blank strings;
# - distinct counts;
# - minimum and maximum text lengths.

remaining_text_fields = [
    "sire",
    "dam",
    "damsire",
    "owner",
    "comment",
]

remaining_text_profile_rows = []

for field in remaining_text_fields:
    quoted_field = f'"{field}"'

    result = pd.read_sql_query(
        f"""
        SELECT
            COUNT(*) AS total_rows,

            SUM(
                CASE
                    WHEN {quoted_field} IS NULL
                    THEN 1 ELSE 0
                END
            ) AS null_count,

            SUM(
                CASE
                    WHEN typeof({quoted_field}) = 'text'
                     AND length(trim({quoted_field})) = 0
                    THEN 1 ELSE 0
                END
            ) AS blank_text_count,

            COUNT(DISTINCT {quoted_field})
                AS distinct_non_null_values,

            MIN(
                CASE
                    WHEN typeof({quoted_field}) = 'text'
                     AND length(trim({quoted_field})) > 0
                    THEN length({quoted_field})
                END
            ) AS minimum_populated_length,

            MAX(
                CASE
                    WHEN typeof({quoted_field}) = 'text'
                     AND length(trim({quoted_field})) > 0
                    THEN length({quoted_field})
                END
            ) AS maximum_populated_length,

            SUM(
                CASE
                    WHEN typeof({quoted_field}) = 'text'
                    THEN 1 ELSE 0
                END
            ) AS storage_text,

            SUM(
                CASE
                    WHEN typeof({quoted_field}) <> 'text'
                    THEN 1 ELSE 0
                END
            ) AS non_text_storage_rows
        FROM data
        WHERE {DATA_ROW_PREDICATE}
        """,
        connection,
    ).iloc[0].to_dict()

    result["field_name"] = field
    remaining_text_profile_rows.append(result)

remaining_text_quality = pd.DataFrame(
    remaining_text_profile_rows
)[
    [
        "field_name",
        "total_rows",
        "null_count",
        "blank_text_count",
        "distinct_non_null_values",
        "minimum_populated_length",
        "maximum_populated_length",
        "storage_text",
        "non_text_storage_rows",
    ]
]

display(remaining_text_quality)

,field_name,total_rows,null_count,blank_text_count,distinct_non_null_values,minimum_populated_length,maximum_populated_length,storage_text,non_text_storage_rows
0,sire,1851285,0,0,5445,8,26,1851285,0
1,dam,1851285,0,5,104973,5,26,1851285,0
2,damsire,1851285,0,21,6079,3,20,1851285,0
3,owner,1851285,0,35,98235,2,60,1851285,0
4,comment,1851285,0,340394,1426746,1,2206,1851285,0


In [66]:
# Inspect blank pedigree/ownership rows and unusually long comments.
#
# The purpose is only to determine whether the blanks and long comments look
# like ordinary source behaviour or obvious corruption.

blank_reference_rows = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        date,
        course,
        race_id,
        off,
        race_name,
        horse,
        sire,
        dam,
        damsire,
        owner,
        comment
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND (
            length(trim(dam)) = 0
         OR length(trim(damsire)) = 0
         OR length(trim(owner)) = 0
      )
    ORDER BY
        date,
        course,
        off,
        horse
    """,
    connection,
)

long_comment_rows = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        date,
        course,
        race_id,
        off,
        race_name,
        horse,
        pos,
        length(comment) AS comment_length,
        comment
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND length(comment) >= 500
    ORDER BY
        comment_length DESC,
        date,
        course,
        off
    LIMIT 50
    """,
    connection,
)

print(
    "Rows with blank dam, damsire, or owner: "
    f"{len(blank_reference_rows):,}"
)

with pd.option_context(
    "display.max_rows",
    None,
    "display.max_columns",
    None,
    "display.max_colwidth",
    120,
):
    display(blank_reference_rows)

print(
    "Rows with comments at least 500 characters long: "
    f"{len(long_comment_rows):,}"
)

with pd.option_context(
    "display.max_rows",
    50,
    "display.max_columns",
    None,
    "display.max_colwidth",
    220,
):
    display(long_comment_rows)

Rows with blank dam, damsire, or owner: 54


,source_rowid,date,course,race_id,off,race_name,horse,sire,dam,damsire,owner,comment
0,1443,2015-01-03,Santa Anita (USA),617158,11:30,Santa Ynez Stakes (Fillies) (Dirt),Rattataptap (USA),Tapit (USA),Sindy With An S (USA),Broken Vow,,
1,33009,2015-04-04,Caulfield (AUS),623280,6:10,Le Pine Funerals Easter Cup Handicap) (2yo+) (Turf),Post DFrance (NZ),Postponed (USA),Creil (NZ),Frenchpark,,
2,40683,2015-04-18,Nakayama (JPN),624701,7:40,Nakayama Grand Jump (Turf),Country Snow (JPN),Timber Country (USA),Snow Style (USA),,Yoshio Suzuki,
3,48891,2015-05-03,San Siro (ITY),631223,12:07,Premio Razza Ticino (Turf),Balami Fan (ITY),Johnny Red Kerr (USA),Babwin Di San Jore (ITY),In The Wings,,
4,71791,2015-06-18,Longchamp (FR),630083,11:15,Prix de Bougival (Maiden) (Unraced 3yo Colts & Geldings) (Turf),Rappeur Des Mottes (FR),Racinger (FR),Camille Des Mottes (FR),Abdonski,,
5,127268,2015-10-15,Mombetsu (JPN),638862,11:07,Edelweiss Sho (Local (Fillies) (Dirt),Derma Okaru (JPN),Million Disk (JPN),Admire Lap (JPN),,Hiroyuki Asanuma,
6,191333,2016-04-13,Funabashi (JPN),649077,11:07,Marine Cup (Local (Dirt),Kura Carmen (JPN),Star King Man (USA),Kura Masa Shuttle (JPN),,Toshihiro Kurami,
7,203991,2016-05-07,Maisons-Laffitte (FR),651027,3:30,Prix de lEtoile dArtois (Handicap) (4yo) (Round) (Turf),Colombia DEmra (FR),Naaqoos (GB),Talka (FR),Vettori,,
8,203870,2016-05-07,Maisons-Laffitte (FR),651027,3:30,Prix de lEtoile dArtois (Handicap) (4yo) (Round) (Turf),Star White (FR),Evasive (GB),Spirit Of Monaco (FR),Turtle Island,,
9,234163,2016-07-06,Kawasaki (JPN),655460,11:07,Sparking Lady Cup (Local (Dirt),Kura Carmen (JPN),Star King Man (USA),Kura Masa Shuttle (JPN),,Toshihiro Kurami,


Rows with comments at least 500 characters long: 50


,source_rowid,date,course,race_id,off,race_name,horse,pos,comment_length,comment
0,1622460,2025-01-13,Wolverhampton (AW),884463,5:30,Boost Your Acca At BetMGM Apprentice Handicap,Further Measure (USA),4,2206,Raced in last - still plenty to do when switched right over 1f out - ran on but never near to challenge (inquiry held into running and riding of gelding which raced in rear throughout before staying on strongly downh...
1,1745480,2025-10-06,Wolverhampton (AW),903500,4:20,Free Tips Daily On attheraces.com Amateur Jockeys Handicap,Oman (IRE),9,2083,In rear and raced wide early - still plenty to do and waiting for room over 1f out - nudged along and some headway inside final furlong (enq held into the running and riding of the gelding - which jumped well from st...
2,873883,2020-08-13,Tramore (IRE),764173,6:15,Flynn Hotels Handicap Hurdle,Enzani (IRE),PU,2034,Mid-division - dropped towards rear 6th - detached and pulled up before 3 out (jockey said gelding lost its action and was pulled up; an enquiry was held into the running and riding of Enzani; evidence was heard from...
3,1337070,2023-05-10,Gowran Park (IRE),840379,5:25,Racing Again May 23rd Handicap,Ellaat (GB),4,2023,Dwelt start - towards rear - headway under 3f out - tenderly handled but kept on well inside final furlong - went fourth towards finish - not reach leaders - eyecatcher (inquiry into running and riding of gelding; jo...
4,1753197,2025-10-20,Wolverhampton (AW),904488,16:55,Download The Raceday Ready App Apprentice Handicap,Bandello (GB),4,1990,In rear - still plenty to do 2f out - headway from over 1f out - kept on inside final furlong - nearest finish (inquiry into running and riding of gelding which jumped awkwardly from talls before racing towards the r...
5,1478171,2024-03-19,Wolverhampton (AW),861346,7:30,Win £2 000 000 With BetMGMs Golden Goals Handicap (Rider Restricted Race),Kodi Noir (IRE),10,1986,Slowly away - in rear - no telling impression (inquiry held into running and riding; rider and Mr Clutterbuck were interviewed and shown recordings of race; vet had nothing to report; rider said that his instructions...
6,1653708,2025-03-31,Wolverhampton (AW),889579,4:55,Extra Winnings With BetUK Acca Club Amateur Jockeys Handicap,Nevernay (IRE),9,1948,In rear - awkward start and lost many lengths - soon detached - some headway home straight - no impression (inquiry held into running and riding of gelding which jumped awkwardly from stalls and lost several lengths ...
7,1140782,2022-02-10,Doncaster,802285,3:20,Virgin Bet Novices Hurdle (Novices Championship Novices Hurdle Series Qualifier),Flaming Ambition (IRE),4,1937,Took keen hold - held up in rear - nudged along before 3 out - going okay but still plenty to do before 2 out - pushed along approaching last - stayed on run-in - eyecatcher (an enquiry was held into the running and ...
8,1657172,2025-04-07,Wolverhampton (AW),890629,4:05,Boost Your Acca At BetMGM Handicap,Charlatan (IRE),9,1896,Held up in rear - not clear run over 1f out - no impression (jockey was questioned regarding his riding of gelding over final 2f after reporting at scales that he had been denied a clear run in home straight; jockey ...
9,1434551,2023-11-25,Huntingdon,853038,12:28,Follow Us On Twitter @betrhino Maiden Hurdle (GBB Race),Zain Nights (GB),3,1895,Midfield - not fluent 3 out - soon shaken up - not fluent last - kept on well run-in - went third inside final 110yds - eyecatcher (inquiry held into running and riding of gelding - which raced mid-division and was t...


## Pedigree, ownership, and comment fields: findings

### `sire`, `dam`, and `damsire`

- All three fields are stored consistently as text.
- No SQL `NULL` values are used.
- `sire` is fully populated.
- `dam` has only 5 blank rows.
- `damsire` has only 21 blank rows.
- The small number of blanks appears to reflect incomplete source coverage,
  especially in some overseas records, rather than malformed data.

### `owner`

- The field is stored consistently as text.
- No SQL `NULL` values are used.
- Only 35 rows are blank.
- Blank ownership values are rare and appear to be ordinary missing source data.

### `comment`

- The field is stored consistently as text.
- Missing comments are represented by blank strings.
- 340,394 rows have blank comments.
- Populated comments range from 1 to 2,206 characters.
- Very long comments are legitimate extended steward-enquiry and
  running-and-riding notes rather than obvious corruption.
- The complete raw comment should therefore be preserved without truncation.

### Transformation implications

A later transformation layer should:

1. convert blank `dam`, `damsire`, `owner`, and `comment` values to missing;
2. retain pedigree and ownership fields as text;
3. preserve the full unaltered comment text;
4. avoid imposing a restrictive comment-length limit;
5. retain raw values for auditability.

## Notebook 02 conclusion

This notebook profiled the source fields at row level and established the main
quality rules needed for later transformation work.

### Main conclusions

- The imported header row must be excluded with `rowid <> 1`.
- The source uses blank strings and sentinel values rather than SQL `NULL`.
- Several fields contain mixed storage types despite their declared SQLite type.
- Raw values should remain unchanged in the source layer.
- Cleaning and type conversion should happen only in a separate transformation
  layer.

### Important field-specific rules

- `date`, `course`, `off`, and `race_name` can be used as a provisional race
  grouping key, but this is not yet a permanent race identifier.
- `pos` contains both numeric placings and non-finisher status codes.
- Numeric finishing positions may contain gaps or exceed `ran`.
- `btn`, `ovr_btn`, and `time` preserve aspects of the physical finish even when
  official placings are later revised.
- `draw` is blank where no draw applies.
- `wgt` is stored as stone-pound text and can later be converted to total pounds.
- `hg` contains compact headgear codes and combinations.
- `sp` is mostly fractional-odds text with a small number of malformed values.
- `prize` combines numeric values with euro-prefixed text and therefore requires
  separate amount and currency fields.
- `or`, `rpr`, and `ts` use `–` as their missing-value marker.
- The isolated `rpr = 775` value should be retained raw and flagged as anomalous.
- Pedigree and ownership blanks are rare.
- Long comments are legitimate steward and running notes and must not be
  truncated.

### Transformation principles

The later cleaned layer should:

1. preserve every original raw field;
2. add parsed and standardised columns alongside the raw values;
3. convert known blanks and sentinels to explicit missing values;
4. use nullable numeric types;
5. add anomaly flags instead of silently correcting questionable values;
6. keep transformation logic reproducible and documented;
7. avoid enforcing assumptions that the source evidence does not support.

The next notebook can use these findings to design the cleaned staging schema and
formal transformation rules.

In [67]:
# Close the read-only SQLite connection now that profiling is complete.

connection.close()

print("Notebook 02 complete: source field quality profile finished.")

Notebook 02 complete: source field quality profile finished.
